In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Imports and Environment ===
# Imports, environment report and plot style; no parameters.

import os
import sys
import json
import time
import math
import shutil
import hashlib
import textwrap
import subprocess
import warnings
from pathlib import Path
from datetime import datetime, timezone
from contextlib import contextmanager
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, Polygon as MplPolygon
from PIL import Image, PngImagePlugin

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from scipy import ndimage

# --- Optional dependencies ---
try:
    import tifffile
except ImportError:
    tifffile = None

try:
    import torch
except ImportError:
    torch = None

try:
    import spectral.io.envi as envi          # ENVI .hdr/.raw cubes
except ImportError:
    envi = None

Image.MAX_IMAGE_PIXELS = None                # line-scan frames can be large
warnings.filterwarnings("ignore", category=UserWarning, module="PIL")

# --- Plot style ---
# rcParams must be set before the first plt.subplots() call in the session.
_FW = 16 / 2.54          # LaTeX \textwidth in inches
_FONT_SIZE = 10

_PREFERRED_FONTS = ["DejaVu Sans", "Helvetica", "Arial", "Nimbus Sans",
                    "TeX Gyre Heros", "Liberation Sans"]
_available = {f.name for f in fm.fontManager.ttflist}
_font = next((f for f in _PREFERRED_FONTS if f in _available), "DejaVu Sans")

plt.rcParams["font.family"] = _font
plt.rcParams["font.size"] = _FONT_SIZE
plt.rcParams["axes.titlesize"] = _FONT_SIZE + 1
plt.rcParams["axes.labelsize"] = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["xtick.labelsize"] = _FONT_SIZE - 1
plt.rcParams["ytick.labelsize"] = _FONT_SIZE - 1
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["legend.frameon"] = False


def _panel_label(ax, letter, title):
    """Bold panel letter in front of the subplot title."""
    ax.set_title(f"$\\bf{{({letter})}}$ {title}")


def _titled_legend(fig, title, handles=None, ncol=3, top_rect=0.86):
    """Place a suptitle and an optional figure legend without overlap.

    The suptitle sits highest, the legend just below it; `top_rect` reserves the space
    above the axes so tight_layout does not pull them into either.
    """
    if handles:
        fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.995),
                   ncol=ncol, frameon=False)
        fig.suptitle(title, y=1.09)
    else:
        fig.suptitle(title, y=1.02)
    plt.tight_layout(rect=[0, 0, 1, top_rect])

# --- Environment report ---
print("=" * 78)
print("ENVIRONMENT")
print("=" * 78)
print(f"python        : {sys.executable}")
print(f"python ver    : {sys.version.split()[0]}")
print(f"cwd           : {Path.cwd()}")
print(f"plot font     : {_font}")
print(f"numpy         : {np.__version__}")
print(f"pandas        : {pd.__version__}")
print(f"tifffile      : {getattr(tifffile, '__version__', 'NOT INSTALLED')}")
print(f"spectral/envi : {'available' if envi is not None else 'NOT INSTALLED (ENVI cubes unreadable)'}")

if torch is None:
    print("torch         : NOT INSTALLED  ->  SAM3 stages will not run")
else:
    print(f"torch         : {torch.__version__}")
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        print(f"cuda          : available, {n} device(s)")
        for i in range(n):
            props = torch.cuda.get_device_properties(i)
            print(f"   [{i}] {props.name}  {props.total_memory / 1024**3:.0f} GB")
    else:
        print("cuda          : NOT AVAILABLE  ->  SAM3 on CPU is ~260 s/image (infeasible at scale)")

# Subprocess workers use sys.executable, so the kernel must be the sam3-seg env.
if "sam3-seg" not in sys.executable:
    print()
    print("!! WARNING: interpreter does not look like the 'sam3-seg' env.")
    print("!! Expected something like /home/<user>/.conda/envs/sam3-seg/bin/python")
    print("!! Subprocess workers use sys.executable, so a wrong kernel breaks them silently.")
print("=" * 78)

In [ ]:
# === Cell 3: Config ===
# Paths, compute settings and data invariants. Tunables are set via widgets and frozen
# by lock_stage; the directory layout is discovered in the Archive Discovery cell.

# --- Run identity ---
RUN_TAG = "hsi_preprocessing_v2"
RUN_ID = f"{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"

# --- Input ---
INPUT_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/"
                  "Raw_labeled_dataset_RGB_and_HSI_images")
# alternatives:
#   .../Spectral_Waste/Preprocessed_Labeled_RGB_and_HSI_images     (256x256, downsampled)
#   .../Spectral_Waste/Preprocessed_Unlabeled_RGB_and_HSI_images   (256x256, 6803 frames)

# The alignment transform comes from the dataset authors' source code. A path here only
# enables the side-by-side reference display in the Archive Discovery cell.
REFERENCE_PREPROCESSED_ROOT = None

# --- Output ---
# Same layout as the WaRP and PackWISE notebooks:
# <dataset root>/KI_Sort_Preprocessing/{data,results}/<RUN_ID>.
OUT_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/KI_Sort_Preprocessing")
DATA_OUT_ROOT = OUT_ROOT / "data" / RUN_ID
RESULTS_OUT_ROOT = OUT_ROOT / "results" / RUN_ID

DIR_STAGE1 = DATA_OUT_ROOT / "stage1_instances"      # whole-object masks + instances.csv
DIR_CUTOUTS = DATA_OUT_ROOT / "stage2_cutouts"       # cutout PNGs + cutouts.csv
DIR_PSEUDOGT = DATA_OUT_ROOT / "stage3_pseudo_gt"    # body/cap/label npz
DIR_YOLO_DS = DATA_OUT_ROOT / "yolo_dataset"         # images/ labels/ data.yaml
DIR_SPECTRA = DATA_OUT_ROOT / "stage5_spectra"       # per-region spectra

DIR_WORKERS = RESULTS_OUT_ROOT / "workers"           # generated worker .py + per-GPU logs
DIR_FIGURES = RESULTS_OUT_ROOT / "figures"
DIR_TABLES = RESULTS_OUT_ROOT / "tables"
DIR_LOGS = RESULTS_OUT_ROOT / "logs"
MANIFEST_PATH = RESULTS_OUT_ROOT / "run_manifest.json"

# --- Models ---
SAM3_WEIGHTS = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/08_Generalizability/sam3_weights/sam3.pt")
# Deployment-tier YOLO model; used only for the visual QA preview, no metrics.
SEG_MODEL_DEPLOY = Path(
    f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260603_Yolo_Deployment_Training/"
    "runs/20260603_Yolo_Deployment_Training_DeploymentSegTraining/weights/best.pt"
)

# --- Compute ---
GPU_POLICY = "auto"        # "auto" | "single" | "multi" | "cpu"
GPU_IDS = None             # None -> all visible devices; else e.g. [0, 1]
PROGRESS_EVERY = 25        # emit a progress/ETA line every N items
RATE_WINDOW = 100          # rolling window (items) for the ETA rate estimate
CHECKPOINT_EVERY = 50      # flush checkpoint CSV every N items
RESUME = True              # skip items already present in the checkpoint
WORKER_POLL_S = 5.0        # how often the notebook polls worker checkpoints

# --- Data invariants ---
N_BANDS = 224
WL_MIN_NM, WL_MAX_NM = 900.0, 1700.0   # linear spacing is an assumption; no calibration ships
WAVELENGTHS_NM = np.linspace(WL_MIN_NM, WL_MAX_NM, N_BANDS)

RGB_NATIVE = (1184, 1200)              # Teledyne DALSA Linea  (W, H)
HSI_NATIVE = (640, 600)                # Specim FX17           (W, H)
# Actual grid sizes are discovered per file; nothing downstream assumes 256.

# --- YOLO geometry ---
# Deployment model runs at imgsz=640, rect=True -> inference shape 384x640 (h x w).
# Cutout canvases use this aspect and Stage 4 exports at this pixel size.
YOLO_INFER_H, YOLO_INFER_W = 384, 640
CANVAS_ASPECT_W, CANVAS_ASPECT_H = YOLO_INFER_W, YOLO_INFER_H

# --- Alignment constants ---
# From spectralwaste/alignment/image_alignment.py :: multimodal_manual_align
#   rgb_align = rgb[:1190, :]; hyper_align = hyper[:, :605]; both resized to a square.
# Both crops are origin-anchored, so the composition is a pure per-axis scale.
RGB_CROP_ROWS = 1190                   # rgb[:1190, :]  -> bottom 10 rows have no HSI
HSI_CROP_COLS = 605                    # hyper[:, :605] -> right 35 cols have no RGB
ALIGN_SX = HSI_CROP_COLS / RGB_NATIVE[0]    # 0.5109797
ALIGN_SY = HSI_NATIVE[1] / RGB_CROP_ROWS    # 0.5042017
# The two scales differ by 1.344%. The alignment cell measures the residual shift
# (DX constant, DY per capture session).

# Default mask erosion before HSI sampling, sized to the ~3.3 px parallax residual.
# Widget default in the spectra extraction cell, not a locked value.
ERODE_PX_DEFAULT = 4

CLASS_MAP = {"body": 0, "cap": 1, "label": 2}   # asserted against YOLO model.names on load
CLASS_NAMES = {v: k for k, v in CLASS_MAP.items()}

# --- Output format ---
CUTOUT_FORMAT = "png"      # lossless; cutouts are pseudo-GT inputs
EMBED_PROVENANCE = True    # PNG tEXt chunks / npz provenance array

# --- Style ---
C_BODY, C_CAP, C_LABEL = "#FF0000", "#00FF00", "#0000FF"
REGION_COLORS = {"body": C_BODY, "cap": C_CAP, "label": C_LABEL}
REGION_ORDER = ("body", "cap", "label")
BAND_COLOR = "#f0f0f0"
# Project palette
C_EDT_ALIGN, C_GRID_ALIGN = "#2a9d8f", "#e76f51"

# --- Assemble and hash config ---
_CONFIG_EXCLUDE = {"CONFIG", "CONFIG_HASH"}
CONFIG = {}
for _k, _v in sorted(list(globals().items())):
    if not _k.isupper() or _k.startswith("_") or _k in _CONFIG_EXCLUDE:
        continue
    if isinstance(_v, Path):
        CONFIG[_k] = str(_v)
    elif isinstance(_v, np.ndarray):
        CONFIG[_k] = f"<ndarray shape={_v.shape}>"
    elif isinstance(_v, (str, int, float, bool, type(None), list, tuple, dict)):
        CONFIG[_k] = _v
    else:
        CONFIG[_k] = repr(_v)

CONFIG_HASH = hashlib.sha1(
    json.dumps(CONFIG, sort_keys=True, default=str).encode()
).hexdigest()[:12]

for _d in (DIR_STAGE1, DIR_CUTOUTS, DIR_PSEUDOGT, DIR_YOLO_DS, DIR_SPECTRA,
           DIR_WORKERS, DIR_FIGURES, DIR_TABLES, DIR_LOGS):
    _d.mkdir(parents=True, exist_ok=True)

print(f"RUN_ID      : {RUN_ID}")
print(f"CONFIG_HASH : {CONFIG_HASH}")
print(f"input       : {INPUT_ROOT}")
print(f"              exists: {INPUT_ROOT.is_dir()}")
print(f"reference   : {'disabled - transform taken from authors source' if REFERENCE_PREPROCESSED_ROOT is None else REFERENCE_PREPROCESSED_ROOT}")
print(f"alignment   : scale x {ALIGN_SX:.7f}  scale y {ALIGN_SY:.7f}  "
      f"anisotropy {(ALIGN_SX / ALIGN_SY - 1) * 100:+.3f}%")
print(f"              rgb[:{RGB_CROP_ROWS}, :]  hyper[:, :{HSI_CROP_COLS}]  "
      f"erode default {ERODE_PX_DEFAULT} px")
print(f"canvas      : aspect {CANVAS_ASPECT_W}:{CANVAS_ASPECT_H}  "
      f"export {YOLO_INFER_W}x{YOLO_INFER_H}")
print(f"data out    : {DATA_OUT_ROOT}")
print(f"results out : {RESULTS_OUT_ROOT}")
print(f"sam3 weights: {SAM3_WEIGHTS.exists()}")
print(f"yolo weights: {SEG_MODEL_DEPLOY.exists()}")

In [ ]:
# === Cell 4: Run Infrastructure ===
# Manifest, parameter locking, progress/ETA, compute plan, checkpoints, provenance and
# widget helpers used by all later stages; no tunables.

# --- Small utilities ---

def slugify(text):
    """Filesystem-safe lowercase token (non-alphanumerics become single underscores)."""
    keep = [c.lower() if c.isalnum() else "_" for c in str(text)]
    out = "".join(keep)
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")


def fmt_dur(seconds):
    if seconds is None or not np.isfinite(seconds):
        return "--"
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m:02d}m"
    if m:
        return f"{m}m {s:02d}s"
    return f"{s}s"


def fmt_bytes(n):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} PB"


def hash_params(params):
    return hashlib.sha1(json.dumps(params, sort_keys=True, default=str).encode()).hexdigest()[:12]


def utcnow():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


# --- Manifest ---

def read_manifest():
    """Read the run manifest; an unreadable (e.g. truncated) file is backed up and reset."""
    if not MANIFEST_PATH.exists():
        return {}
    try:
        return json.loads(MANIFEST_PATH.read_text())
    except (json.JSONDecodeError, ValueError):
        broken = MANIFEST_PATH.with_suffix(".json.broken")
        shutil.copy2(MANIFEST_PATH, broken)
        print(f"!! manifest unreadable, copied to {broken.name} and starting fresh")
        return {}


def write_manifest(man):
    """Atomic: write to a temp file in the same dir, then replace."""
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = MANIFEST_PATH.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(man, indent=2, sort_keys=True, default=str))
    tmp.replace(MANIFEST_PATH)


def log_params(stage, **kv):
    """Merge a stage record into the run manifest."""
    man = read_manifest()
    man.setdefault("run_id", RUN_ID)
    man.setdefault("run_tag", RUN_TAG)
    man.setdefault("config_hash", CONFIG_HASH)
    man.setdefault("config", CONFIG)
    man.setdefault("created_utc", utcnow())
    man["updated_utc"] = utcnow()
    stages = man.setdefault("stages", {})
    rec = stages.setdefault(stage, {})
    for k, v in kv.items():
        if k == "params" and isinstance(v, dict):
            rec.setdefault("params", {}).update(v)
        else:
            rec[k] = v
    write_manifest(man)
    return man


# --- Parameter locking ---
# Widget values are frozen and logged to the manifest so batch runs are reproducible.

STAGE_PARAMS = {}   # stage -> {"params": {...}, "param_hash": ..., "locked_utc": ...}


def lock_stage(stage, params, n_preview=None, notes=None):
    """Freeze widget values, log them, and authorise the stage's batch run."""
    entry = {
        "params": dict(params),
        "param_hash": hash_params(params),
        "locked_utc": utcnow(),
        "n_preview": n_preview,
    }
    if notes:
        entry["notes"] = notes
    STAGE_PARAMS[stage] = entry
    log_params(stage, **entry)
    print(f"[{stage}] LOCKED  hash={entry['param_hash']}")
    for k, v in sorted(params.items()):
        print(f"    {k:24s} = {v}")
    return entry


def require_locked(stage):
    """Batch guard. Raises rather than silently running on stale defaults."""
    if stage not in STAGE_PARAMS:
        raise RuntimeError(
            f"'{stage}' has no locked parameters.\n"
            f"Run its preview cell and press 'Lock params' before starting the batch."
        )
    return dict(STAGE_PARAMS[stage]["params"])


def load_params_from_run(run_id, stages=None, results_root=None):
    """Repopulate STAGE_PARAMS from a previous run's manifest.

    Returns the loaded dict; widgets are re-synced by each cell's `sync_widgets_from_params`.
    """
    root = Path(results_root) if results_root else RESULTS_OUT_ROOT.parent
    path = root / run_id / "run_manifest.json"
    if not path.exists():
        raise FileNotFoundError(f"No manifest at {path}")
    man = json.loads(path.read_text())
    loaded = {}
    for stage, rec in man.get("stages", {}).items():
        if stages and stage not in stages:
            continue
        if "params" in rec:
            STAGE_PARAMS[stage] = {
                "params": rec["params"],
                "param_hash": rec.get("param_hash"),
                "locked_utc": rec.get("locked_utc"),
                "n_preview": rec.get("n_preview"),
                "loaded_from": run_id,
            }
            loaded[stage] = rec["params"]
    print(f"Loaded {len(loaded)} stage param set(s) from {run_id}: {sorted(loaded)}")
    return loaded


# --- Progress and ETA ---

class ProgressReporter:
    """Rolling-window throughput and ETA.

    A rolling window keeps model load and GPU warm-up out of the steady-state rate.
    """

    def __init__(self, stage, n_total, every=None, window=None, log_path=None, quiet=False):
        self.stage = stage
        self.n_total = max(int(n_total), 0)
        self.every = every or PROGRESS_EVERY
        self.window = window or RATE_WINDOW
        self.quiet = quiet
        self.n_done = 0
        self.t0 = time.time()
        self._samples = deque(maxlen=self.window)     # (timestamp, n_done)
        self._samples.append((self.t0, 0))
        self._throughput = []
        self.log_path = Path(log_path) if log_path else DIR_LOGS / f"{stage}_progress.log"
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self._fh = open(self.log_path, "a", buffering=1)
        self._emit(f"--- {stage} start {utcnow()}  n_total={self.n_total} ---")

    def _rate(self):
        if len(self._samples) < 2:
            return 0.0
        t_old, n_old = self._samples[0]
        t_new, n_new = self._samples[-1]
        dt = t_new - t_old
        return (n_new - n_old) / dt if dt > 0 else 0.0

    def _emit(self, line):
        self._fh.write(line + "\n")
        if not self.quiet:
            print(line)

    def update(self, n=1, force=False):
        self.n_done += n
        self._samples.append((time.time(), self.n_done))
        if force or self.n_done % self.every == 0 or self.n_done == self.n_total:
            self.report()

    def set_done(self, n_done, force=False):
        """For aggregating across subprocess workers (counts arrive absolute)."""
        self.n_done = int(n_done)
        self._samples.append((time.time(), self.n_done))
        if force or self.n_done >= self.n_total:
            self.report()
        return self.n_done

    def report(self):
        elapsed = time.time() - self.t0
        rate = self._rate()
        remaining = max(self.n_total - self.n_done, 0)
        eta = remaining / rate if rate > 1e-9 else float("inf")
        pct = 100.0 * self.n_done / self.n_total if self.n_total else 0.0
        line = (f"[{self.stage}] {self.n_done}/{self.n_total} ({pct:5.1f}%) | "
                f"{rate:6.2f} it/s | elapsed {fmt_dur(elapsed)} | ETA {fmt_dur(eta)}")
        self._emit(line)
        self._throughput.append({"t": round(elapsed, 1), "n": self.n_done, "rate": round(rate, 4)})

    def close(self, extra=None):
        elapsed = time.time() - self.t0
        self.report()
        self._emit(f"--- {self.stage} done {utcnow()}  {self.n_done}/{self.n_total} "
                   f"in {fmt_dur(elapsed)} ---")
        self._fh.close()
        rec = {
            "n_total": self.n_total,
            "n_done": self.n_done,
            "elapsed_s": round(elapsed, 1),
            "mean_rate_it_s": round(self.n_done / elapsed, 4) if elapsed > 0 else None,
            "throughput_samples": self._throughput[-200:],
            "progress_log": str(self.log_path),
        }
        if extra:
            rec.update(extra)
        log_params(self.stage, runtime=rec)
        return rec


def estimate_batch_runtime(seconds_per_item, n_items):
    """Projected batch runtime from preview throughput."""
    if not seconds_per_item or not np.isfinite(seconds_per_item):
        return "unknown"
    return fmt_dur(seconds_per_item * n_items)


# --- Compute plan ---

def resolve_compute_plan(policy=None, gpu_ids=None):
    policy = policy or GPU_POLICY
    ids = gpu_ids if gpu_ids is not None else GPU_IDS

    n_cuda = 0
    names = []
    if torch is not None and torch.cuda.is_available():
        n_cuda = torch.cuda.device_count()
        names = [torch.cuda.get_device_properties(i).name for i in range(n_cuda)]

    if ids is None:
        ids = list(range(n_cuda))
    else:
        ids = [i for i in ids if i < n_cuda]

    if policy == "cpu" or n_cuda == 0:
        mode, devices = "cpu", []
    elif policy == "single":
        mode, devices = "single", ids[:1]
    elif policy == "multi":
        mode, devices = ("multi", ids) if len(ids) > 1 else ("single", ids[:1])
    else:  # auto
        mode, devices = ("multi", ids) if len(ids) > 1 else ("single", ids[:1])

    plan = {
        "mode": mode,
        "devices": devices,
        "n_workers": max(len(devices), 1),
        "device_names": [names[i] for i in devices] if devices else [],
        "policy": policy,
        "n_cuda_visible": n_cuda,
    }
    return plan


COMPUTE_PLAN = resolve_compute_plan()

print("=" * 78)
print("COMPUTE PLAN")
print("=" * 78)
print(f"policy        : {COMPUTE_PLAN['policy']}")
print(f"mode          : {COMPUTE_PLAN['mode']}")
print(f"cuda devices  : {COMPUTE_PLAN['devices'] or 'none'}")
for i, nm in zip(COMPUTE_PLAN["devices"], COMPUTE_PLAN["device_names"]):
    print(f"   [{i}] {nm}")
print(f"workers       : {COMPUTE_PLAN['n_workers']}")
if COMPUTE_PLAN["mode"] == "cpu":
    print("!! CPU mode: SAM3 encode is ~260 s/image. A full-archive run is not feasible.")
print("=" * 78)


# --- Checkpoints ---

def checkpoint_path(stage):
    return DIR_TABLES / f"{stage}_checkpoint.csv"


def load_checkpoint(stage, key_col="item_id"):
    """Returns (DataFrame, set_of_completed_keys). Empty if absent or RESUME is off."""
    p = checkpoint_path(stage)
    if not RESUME or not p.exists():
        return pd.DataFrame(), set()
    try:
        df = pd.read_csv(p)
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        return pd.DataFrame(), set()
    done = set(df[key_col].astype(str)) if key_col in df.columns else set()
    return df, done


def append_checkpoint(stage, rows):
    """Append rows to the stage checkpoint. Header written only on creation."""
    if not rows:
        return
    p = checkpoint_path(stage)
    df = pd.DataFrame(rows)
    df.to_csv(p, mode="a", header=not p.exists(), index=False)


def count_checkpoint_rows(path):
    """Cheap liveness/progress probe used to poll subprocess workers."""
    p = Path(path)
    if not p.exists():
        return 0
    try:
        with open(p, "rb") as fh:
            n = sum(1 for _ in fh)
        return max(n - 1, 0)       # minus header
    except OSError:
        return 0


def count_checkpoint_items(path, key_col="stem"):
    """Progress probe counting distinct `key_col` values instead of raw CSV rows.

    Workers can write several rows per item (e.g. one per detection), so distinct keys
    give the number of finished items.
    """
    p = Path(path)
    if not p.exists():
        return 0
    try:
        df = pd.read_csv(p, usecols=lambda c: c == key_col)
        if key_col not in df.columns:
            return count_checkpoint_rows(path)
        return int(df[key_col].astype(str).nunique())
    except (pd.errors.EmptyDataError, pd.errors.ParserError, OSError, ValueError):
        return 0


# --- Provenance ---
# Stored in three places (npz array, PNG tEXt / JSON sidecar, CSV) so each file
# stays self-describing.

def provenance_record(stage, source_paths, params, extra=None):
    rec = {
        "run_id": RUN_ID,
        "stage": stage,
        "config_hash": CONFIG_HASH,
        "param_hash": hash_params(params),
        "created_utc": utcnow(),
    }
    if isinstance(source_paths, dict):
        rec.update({f"source_{k}": str(v) for k, v in source_paths.items()})
    else:
        rec["source_path"] = str(source_paths)
    if extra:
        rec.update(extra)
    return rec


def save_mask_npz(path, provenance=None, **arrays):
    """Save masks as compressed npz with an embedded provenance record."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = dict(arrays)
    if EMBED_PROVENANCE and provenance is not None:
        payload["provenance"] = np.array(json.dumps(provenance, default=str))
    np.savez_compressed(path, **payload)
    return path


def read_mask_npz(path):
    """Returns (dict_of_arrays, provenance_dict_or_None)."""
    with np.load(path, allow_pickle=False) as z:
        arrays = {k: z[k] for k in z.files if k != "provenance"}
        prov = None
        if "provenance" in z.files:
            try:
                prov = json.loads(str(z["provenance"]))
            except (json.JSONDecodeError, ValueError):
                prov = None
    return arrays, prov


def save_image_with_provenance(path, arr, provenance=None, fmt=None):
    """PNG gets tEXt chunks; JPEG falls back to a sidecar .json."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fmt = (fmt or path.suffix.lstrip(".") or CUTOUT_FORMAT).lower()
    img = Image.fromarray(arr)
    if fmt == "png":
        info = PngImagePlugin.PngInfo()
        if EMBED_PROVENANCE and provenance:
            for k, v in provenance.items():
                info.add_text(str(k), str(v))
        img.save(path, format="PNG", pnginfo=info, optimize=False)
    else:
        img.save(path, format="JPEG", quality=95)
        if EMBED_PROVENANCE and provenance:
            path.with_suffix(".json").write_text(json.dumps(provenance, indent=2, default=str))
    return path


def read_image_provenance(path):
    path = Path(path)
    if path.suffix.lower() == ".png":
        with Image.open(path) as im:
            return dict(im.text) if hasattr(im, "text") else {}
    side = path.with_suffix(".json")
    return json.loads(side.read_text()) if side.exists() else {}


def write_provenance_csv(stage, rows):
    if not rows:
        return None
    p = DIR_TABLES / f"{stage}_provenance.csv"
    pd.DataFrame(rows).to_csv(p, mode="a", header=not p.exists(), index=False)
    return p


# --- Widget helpers ---

def status_html(stage, n_scope=None, est=None, extra=""):
    locked = STAGE_PARAMS.get(stage)
    lock_txt = (f"<span style='color:#2a9d8f'>locked {locked['param_hash']}</span>"
                if locked else "<span style='color:#d1495b'>not locked</span>")
    parts = [f"<b>{stage}</b>", lock_txt]
    if n_scope is not None:
        parts.append(f"scope: {n_scope:,}")
    if est:
        parts.append(f"est. batch: {est}")
    if extra:
        parts.append(extra)
    return ("<div style='font-family:monospace;font-size:12px;padding:4px 8px;"
            "background:#f8fafc;border:1px solid #d1d5db;border-radius:6px'>"
            + " &nbsp;|&nbsp; ".join(parts) + "</div>")


def make_action_bar(on_preview, on_lock, on_batch, batch_label="Run batch"):
    """Uniform [Preview] [Lock params] [Run batch] bar with the enable-chain wired.

    Lock is disabled until a preview has rendered, batch until locked, so a batch
    always runs with the previewed settings.
    """
    b_prev = widgets.Button(description="Preview", button_style="info", icon="eye")
    b_lock = widgets.Button(description="Lock params", button_style="warning",
                            icon="lock", disabled=True)
    b_run = widgets.Button(description=batch_label, button_style="success",
                           icon="play", disabled=True)

    def _prev(_):
        on_preview()
        b_lock.disabled = False

    def _lock(_):
        on_lock()
        b_run.disabled = False

    def _run(_):
        b_run.disabled = True
        try:
            on_batch()
        finally:
            b_run.disabled = False

    b_prev.on_click(_prev)
    b_lock.on_click(_lock)
    b_run.on_click(_run)
    return widgets.HBox([b_prev, b_lock, b_run]), (b_prev, b_lock, b_run)


def invalidate_lock(stage, buttons):
    """Any control change re-arms the chain: lock and batch must be redone."""
    STAGE_PARAMS.pop(stage, None)
    _, b_lock, b_run = buttons
    b_lock.disabled = True
    b_run.disabled = True


log_params("_session", started_utc=utcnow(), compute_plan=COMPUTE_PLAN,
           python=sys.executable)
print(f"\nInfrastructure ready. Manifest -> {MANIFEST_PATH}")

In [ ]:
# === Cell 5: Archive Discovery and Frame Index ===
# Scans the archive without assuming a layout, maps modality roles and builds the frame
# table that records the RGB/HSI pairing.
# Roles are matched by directory name (a glob) because the raw archive nests one
# modality dir per capture session:
#     <root>/<date>/<session>/rgb/*.jpg
#     <root>/<date>/<session>/hyper/*.tiff
# while the preprocessed archive puts them at the top:
#     <root>/rgb/{train,val,test}/*.png
# The pattern "**/rgb" covers both.

CUBE_EXTS = {".tif", ".tiff", ".hdr", ".npy", ".npz", ".raw", ".img", ".dat", ".bin"}
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
SPLIT_TOKENS = {"train", "val", "valid", "validation", "test"}

MODALITY_ROLES = ["rgb", "hyper", "labels_rgb", "labels_hyper"]
_ROLE_HINTS = {
    "rgb": ("rgb", "color", "colour"),
    "hyper": ("hyper", "hsi", "spectral", "cube", "nir"),
    "labels_rgb": ("labels_rgb", "label_rgb", "labels_rgb_lt", "annotation_rgb", "masks_rgb"),
    "labels_hyper": ("labels_hyper", "labels_hyper_lt", "label_hyper", "masks_hyper"),
}


# --- I/O helpers ---

def cube_shape(path):
    """Shape without loading pixel data where possible. Returns (bands, H, W) or None."""
    p = Path(path)
    ext = p.suffix.lower()
    try:
        if ext in (".tif", ".tiff") and tifffile is not None:
            with tifffile.TiffFile(str(p)) as tf:
                shp = tf.series[0].shape
            return tuple(int(s) for s in shp)
        if ext == ".hdr" and envi is not None:
            img = envi.open(str(p))
            return (int(img.nbands), int(img.nrows), int(img.ncols))
        if ext == ".npy":
            with open(p, "rb") as fh:
                version = np.lib.format.read_magic(fh)
                shp, _, _ = np.lib.format._read_array_header(fh, version)
            return tuple(int(s) for s in shp)
    except Exception:
        return None
    return None


def load_cube(path, bands=None):
    """Load an HSI cube as (bands, H, W) float32. `bands` selects a subset."""
    p = Path(path)
    ext = p.suffix.lower()
    if ext in (".tif", ".tiff"):
        if tifffile is None:
            raise RuntimeError("tifffile not installed")
        arr = tifffile.imread(str(p))
    elif ext == ".hdr":
        if envi is None:
            raise RuntimeError("spectral (ENVI reader) not installed - `pip install spectral`")
        arr = np.asarray(envi.open(str(p)).load())      # (H, W, B)
        arr = np.transpose(arr, (2, 0, 1))
    elif ext == ".npy":
        arr = np.load(p)
    elif ext == ".npz":
        with np.load(p) as z:
            arr = z[list(z.files)[0]]
    else:
        raise ValueError(f"Unsupported cube format: {p.suffix}")

    arr = np.asarray(arr)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D cube, got shape {arr.shape} from {p.name}")
    # Move the band axis first (the axis matching N_BANDS, else the smallest).
    if arr.shape[0] != N_BANDS:
        cand = [i for i, s in enumerate(arr.shape) if s == N_BANDS]
        axis = cand[0] if cand else int(np.argmin(arr.shape))
        arr = np.moveaxis(arr, axis, 0)
    if bands is not None:
        arr = arr[bands]
    return arr.astype(np.float32)


def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def image_size(path):
    try:
        with Image.open(path) as im:
            return im.size          # (W, H)
    except Exception:
        return None


def hsi_false_colour(cube, band_idx=None, pct=(2, 98)):
    """Three-band composite for visual inspection, per-band contrast stretched."""
    n = cube.shape[0]
    if band_idx is None:
        band_idx = [int(n * f) for f in (0.15, 0.45, 0.80)]
    chans = []
    for b in band_idx:
        x = cube[min(b, n - 1)].astype(np.float64)
        lo, hi = np.percentile(x, pct)
        chans.append(np.clip((x - lo) / (hi - lo + 1e-9), 0, 1))
    return np.dstack(chans)


def stem_of(path):
    """ENVI ships stem.hdr + stem.raw; both must reduce to the same stem."""
    p = Path(path)
    s = p.stem
    return s[:-4] if s.lower().endswith(".raw") else s


def split_of(path, root):
    try:
        parts = [q.lower() for q in Path(path).relative_to(root).parts[:-1]]
    except ValueError:
        return ""
    for q in parts:
        if q in SPLIT_TOKENS:
            return "val" if q in ("valid", "validation") else q
    return ""


def session_of(base_dir, root):
    """Capture session = the path above the modality directory.

    '<root>/20230117/07/rgb' -> '20230117_07'. Frames within a session are strongly
    correlated, so the session is the grouping key for train/val splitting.
    """
    try:
        parts = Path(base_dir).relative_to(root).parts[:-1]
    except ValueError:
        return ""
    return "_".join(p for p in parts if p.lower() not in SPLIT_TOKENS)


# --- Discovery ---

def scan_tree(root, max_depth=6):
    """Inventory of every directory holding files, with counts by extension."""
    root = Path(root)
    cols = ["dir", "abs", "leaf", "depth", "n_files", "extensions", "example"]
    if not root.is_dir():
        return pd.DataFrame(columns=cols)
    rows = []
    for dirpath, dirnames, filenames in os.walk(root):
        d = Path(dirpath)
        try:
            rel = d.relative_to(root)
        except ValueError:
            continue
        depth = len(rel.parts)
        if depth > max_depth:
            dirnames[:] = []
            continue
        files = [f for f in filenames if not f.startswith(".")]
        if not files:
            continue
        exts = {}
        for f in files:
            e = Path(f).suffix.lower()
            exts[e] = exts.get(e, 0) + 1
        rows.append({
            "dir": str(rel) if rel.parts else ".",
            "abs": str(d),
            "leaf": d.name if rel.parts else ".",
            "depth": depth,
            "n_files": len(files),
            "extensions": ", ".join(f"{k or '<none>'}:{v}" for k, v in
                                    sorted(exts.items(), key=lambda kv: -kv[1])),
            "example": sorted(files)[0],
        })
    df = pd.DataFrame(rows, columns=cols)
    return df.sort_values("n_files", ascending=False).reset_index(drop=True)


def leaf_summary(scan_df):
    """Group directories by name (e.g. all directories called 'rgb')."""
    if scan_df.empty:
        return pd.DataFrame(columns=["leaf", "n_dirs", "n_files", "extensions", "example_dir"])
    rows = []
    for leaf, g in scan_df.groupby("leaf"):
        exts = {}
        for s in g["extensions"]:
            for part in str(s).split(", "):
                if ":" not in part:
                    continue
                k, v = part.rsplit(":", 1)
                exts[k] = exts.get(k, 0) + int(v)
        rows.append({"leaf": leaf, "n_dirs": len(g), "n_files": int(g["n_files"].sum()),
                     "extensions": ", ".join(f"{k}:{v}" for k, v in
                                             sorted(exts.items(), key=lambda kv: -kv[1])),
                     "example_dir": g.iloc[0]["dir"]})
    return pd.DataFrame(rows).sort_values("n_files", ascending=False).reset_index(drop=True)


def component_summary(root, max_depth=6):
    """Every distinct directory NAME anywhere in the tree, with the files beneath it.

    Files are counted recursively: in the preprocessed archive a modality dir holds only
    split subdirs ('rgb/train/*.png'), which a leaf-only view would miss.
    """
    root = Path(root)
    cols = ["name", "n_dirs", "n_files_below", "extensions", "img_frac", "cube_frac",
            "example_dir"]
    if not root.is_dir():
        return pd.DataFrame(columns=cols)
    by_name = {}
    for dirpath, dirnames, _ in os.walk(root):
        d = Path(dirpath)
        rel = d.relative_to(root)
        if not rel.parts:
            continue
        if len(rel.parts) > max_depth:
            dirnames[:] = []
            continue
        by_name.setdefault(d.name, []).append(d)

    rows = []
    for name, dirs in by_name.items():
        exts, n = {}, 0
        for d in dirs:
            for p in d.rglob("*"):
                if p.is_file() and not p.name.startswith("."):
                    n += 1
                    e = p.suffix.lower()
                    exts[e] = exts.get(e, 0) + 1
        if not n:
            continue
        img = sum(v for k, v in exts.items() if k in IMAGE_EXTS and k not in (".tif", ".tiff"))
        cube = sum(v for k, v in exts.items() if k in CUBE_EXTS)
        rows.append({
            "name": name, "n_dirs": len(dirs), "n_files_below": n,
            "extensions": ", ".join(f"{k}:{v}" for k, v in
                                    sorted(exts.items(), key=lambda kv: -kv[1])[:4]),
            "img_frac": round(img / n, 3), "cube_frac": round(cube / n, 3),
            "example_dir": str(dirs[0].relative_to(root)),
        })
    return (pd.DataFrame(rows, columns=cols)
            .sort_values("n_files_below", ascending=False).reset_index(drop=True))


def role_pattern_options(scan_df, comp_df=None):
    """Selectable patterns: name globs from every path component, then explicit paths."""
    opts = ["(none)"]
    if comp_df is not None and not comp_df.empty:
        for _, r in comp_df.iterrows():
            opts.append(f"**/{r['name']}")
    if scan_df is not None and not scan_df.empty:
        for d in list(scan_df["dir"])[:40]:
            if d != "." and d not in opts:
                opts.append(d)
    return opts


def guess_role_dirs(scan_df, comp_df=None, root=None):
    """Score directory names by keyword hints and extension purity.

    Purity separates modality dirs ('rgb' all images, 'hyper' all cubes) from split
    dirs such as 'train' that mix both.
    """
    if comp_df is None:
        comp_df = component_summary(root) if root is not None else pd.DataFrame()
    if comp_df.empty:
        return {r: None for r in MODALITY_ROLES}

    out = {}
    for role in MODALITY_ROLES:
        hints = _ROLE_HINTS[role]
        wants_cube = role in ("hyper", "labels_hyper")
        best, best_score = None, 0.0
        for _, r in comp_df.iterrows():
            nm = str(r["name"]).lower()
            score = 0.0
            if nm in hints:
                score += 20
            else:
                for h in hints:
                    if nm.startswith(h) or nm.endswith(h):
                        score += 8
                    elif h in nm:
                        score += 4
            if score <= 0:
                continue                       # no keyword support at all
            if nm in SPLIT_TOKENS:
                score -= 20                    # a split dir is never a modality
            is_label = any(t in nm for t in ("label", "mask", "annot"))
            if role in ("rgb", "hyper") and is_label:
                score -= 15
            if role in ("labels_rgb", "labels_hyper") and not is_label:
                score -= 15
            purity = float(r["cube_frac"] if wants_cube else r["img_frac"])
            score += 6.0 * purity - 4.0 * (1.0 - purity)
            score += min(r["n_files_below"], 5000) / 5000.0
            if score > best_score:
                best, best_score = f"**/{r['name']}", score
        out[role] = best
    return out


def resolve_role_dirs(root, pattern):
    root = Path(root)
    if not pattern or pattern == "(none)":
        return []
    if pattern == ".":
        return [root]
    return sorted({d for d in root.glob(pattern) if d.is_dir()})


def index_role(root, pattern, exts):
    """{stem: {path, split, session}} across every directory matching the pattern."""
    root = Path(root)
    found, dupes = {}, []
    for base in resolve_role_dirs(root, pattern):
        sess = session_of(base, root)
        for p in sorted(base.rglob("*")):
            if not p.is_file() or p.suffix.lower() not in exts:
                continue
            if p.suffix.lower() == ".raw" and p.with_suffix(".hdr").exists():
                continue                          # ENVI: index the .hdr, not the binary
            s = stem_of(p)
            if s in found:
                dupes.append(s)
                continue
            found[s] = {"path": str(p), "split": split_of(p, root),
                        "session": sess or split_of(p, root)}
    return found, dupes


ARCHIVE = {"scan": None, "leaves": None, "components": None, "roles": {},
           "frame_index": None, "layout": {}}


def build_frame_index(root, roles, reference_root=None, probe_shapes=True, progress=True):
    """Build the frame table, the pairing funnel and a layout summary."""
    root = Path(root)
    rgb_idx, rgb_dupes = index_role(root, roles.get("rgb"), IMAGE_EXTS)
    hyp_idx, hyp_dupes = index_role(root, roles.get("hyper"), CUBE_EXTS)
    lrgb_idx, _ = index_role(root, roles.get("labels_rgb"), IMAGE_EXTS)
    lhyp_idx, _ = index_role(root, roles.get("labels_hyper"), IMAGE_EXTS | CUBE_EXTS)

    ref_rgb, ref_hyp = {}, {}
    if reference_root and Path(reference_root).is_dir():
        ref_scan = scan_tree(reference_root, max_depth=4)
        ref_comp = component_summary(reference_root, max_depth=4)
        ref_roles = guess_role_dirs(ref_scan, ref_comp, reference_root)
        ref_rgb, _ = index_role(reference_root, ref_roles.get("rgb"), IMAGE_EXTS)
        ref_hyp, _ = index_role(reference_root, ref_roles.get("hyper"), CUBE_EXTS)

    all_stems = sorted(set(rgb_idx) | set(hyp_idx))
    rows = []
    rep = (ProgressReporter("stage0_index", len(all_stems), every=200, quiet=not progress)
           if probe_shapes else None)

    for s in all_stems:
        r = rgb_idx.get(s)
        h = hyp_idx.get(s)
        src = r or h or {}
        rec = {
            "stem": s,
            "session": src.get("session", ""),
            "split": src.get("split", ""),
            "rgb_path": r["path"] if r else "",
            "hyper_path": h["path"] if h else "",
            "labels_rgb_path": lrgb_idx.get(s, {}).get("path", ""),
            "labels_hyper_path": lhyp_idx.get(s, {}).get("path", ""),
            "ref_rgb_path": ref_rgb.get(s, {}).get("path", ""),
            "ref_hyper_path": ref_hyp.get(s, {}).get("path", ""),
            "has_rgb": bool(r),
            "has_hsi": bool(h),
        }
        if probe_shapes:
            if r:
                sz = image_size(r["path"])
                rec["rgb_w"], rec["rgb_h"] = sz if sz else (None, None)
            if h:
                shp = cube_shape(h["path"])
                if shp and len(shp) == 3:
                    b, hh, ww = shp
                    if b != N_BANDS and N_BANDS in shp:
                        order = list(shp)
                        bi = order.index(N_BANDS)
                        rest = [order[i] for i in range(3) if i != bi]
                        b, hh, ww = N_BANDS, rest[0], rest[1]
                    rec["hsi_bands"], rec["hsi_h"], rec["hsi_w"] = int(b), int(hh), int(ww)
        rec["in_scope"] = rec["has_rgb"] and rec["has_hsi"]
        rows.append(rec)
        if rep:
            rep.update()
    if rep:
        rep.close()

    df = pd.DataFrame(rows)
    funnel = [
        ("RGB files indexed", len(rgb_idx)),
        ("HSI cubes indexed", len(hyp_idx)),
        ("union of stems", len(all_stems)),
        ("paired RGB+HSI", int(df["in_scope"].sum()) if len(df) else 0),
        ("with reference RGB (256)", int((df["ref_rgb_path"] != "").sum()) if len(df) else 0),
        ("with reference HSI (256)", int((df["ref_hyper_path"] != "").sum()) if len(df) else 0),
    ]
    layout = {
        "input_root": str(root),
        "roles": roles,
        "reference_root": str(reference_root) if reference_root else None,
        "n_rgb": len(rgb_idx), "n_hyper": len(hyp_idx),
        "n_labels_rgb": len(lrgb_idx), "n_labels_hyper": len(lhyp_idx),
        "n_sessions": int(df["session"].nunique()) if len(df) else 0,
        "duplicate_stems_rgb": sorted(set(rgb_dupes))[:20],
        "duplicate_stems_hyper": sorted(set(hyp_dupes))[:20],
        "funnel": funnel,
    }
    return df, funnel, layout


# --- Widgets ---

w4_root = widgets.Text(value=str(INPUT_ROOT), description="Input root:",
                       style={"description_width": "110px"},
                       layout=widgets.Layout(width="760px"))
w4_use_ref = widgets.Checkbox(value=REFERENCE_PREPROCESSED_ROOT is not None,
                              description="Index reference 256 archive", indent=False)
w4_probe = widgets.Checkbox(value=True, description="Probe shapes (slower, recommended)",
                            indent=False)
w4_nsample = widgets.IntSlider(value=3, min=1, max=8, description="Preview N:",
                               style={"description_width": "110px"},
                               layout=widgets.Layout(width="380px"))
w4_roles = {r: widgets.Dropdown(options=["(none)"], value="(none)", description=f"{r}:",
                                style={"description_width": "110px"},
                                layout=widgets.Layout(width="520px"))
            for r in MODALITY_ROLES}

w4_scan_btn = widgets.Button(description="Scan archive", button_style="info", icon="search")
w4_status = widgets.HTML()
w4_out = widgets.Output()


def _refresh_role_options(scan_df, comp_df=None, root=None):
    opts = role_pattern_options(scan_df, comp_df)
    guesses = guess_role_dirs(scan_df, comp_df, root)
    for role, w in w4_roles.items():
        g = guesses.get(role)
        if g and g not in opts:
            opts = opts + [g]
        w.options = opts
        w.value = g if g in opts else "(none)"


def _roles_from_widgets():
    return {r: (None if w.value == "(none)" else w.value) for r, w in w4_roles.items()}


def on_scan(_=None):
    with w4_out:
        clear_output(wait=True)
        root = Path(w4_root.value)
        if not root.is_dir():
            print(f"!! Input root does not exist: {root}")
            return
        scan = scan_tree(root)
        ls = leaf_summary(scan)
        comp = component_summary(root)
        ARCHIVE["scan"], ARCHIVE["leaves"], ARCHIVE["components"] = scan, ls, comp
        _refresh_role_options(scan, comp, root)

        print(f"ARCHIVE: {root}")
        print(f"  {len(scan):,} directories containing files\n")
        print("DIRECTORY NAMES (roles are matched by name, so this is the table that matters)")
        print("img_frac / cube_frac = extension purity beneath that name; a real modality")
        print("directory is near 1.0 for one of them, a split directory is mixed.\n")
        display(HTML(comp.head(20).to_html(index=False)))
        if len(comp) > 20:
            print(f"... {len(comp) - 20} more distinct directory names")
        print("\nDEEPEST-POPULATED EXAMPLES")
        display(HTML(scan[["dir", "n_files", "extensions", "example"]].head(8).to_html(index=False)))
        print("\nRole patterns guessed below ('**/rgb' matches every directory named rgb, "
              "at any depth). Override if wrong, then press Preview.")


def render_preview4():
    with w4_out:
        clear_output(wait=True)
        root = Path(w4_root.value)
        roles = _roles_from_widgets()
        ref = REFERENCE_PREPROCESSED_ROOT if w4_use_ref.value else None

        df, funnel, layout = build_frame_index(root, roles, ref, probe_shapes=w4_probe.value)
        ARCHIVE.update({"frame_index": df, "roles": roles, "layout": layout})

        print("PAIRING FUNNEL")
        print("-" * 56)
        prev = None
        for name, n in funnel:
            delta = "" if prev is None else f"  ({n - prev:+d})"
            print(f"  {name:28s} {n:6,d}{delta}")
            prev = n
        print()

        if df.empty:
            print("!! No files indexed. Check the role patterns above.")
            return

        for key in ("duplicate_stems_rgb", "duplicate_stems_hyper"):
            if layout.get(key):
                print(f"!! duplicate stems in {key.split('_')[-1]}: {layout[key]}")
                print("   (same filename in more than one session - only the first was kept)")

        inv = []
        for role, col in (("rgb", "rgb_path"), ("hyper", "hyper_path"),
                          ("labels_rgb", "labels_rgb_path"),
                          ("labels_hyper", "labels_hyper_path")):
            n = int((df[col] != "").sum())
            ndirs = len(resolve_role_dirs(root, roles.get(role)))
            ex = df.loc[df[col] != "", col].iloc[0] if n else ""
            inv.append({"role": role, "pattern": roles.get(role) or "-", "n_dirs": ndirs,
                        "n_files": n, "example": Path(ex).name if ex else "-"})
        display(HTML(pd.DataFrame(inv).to_html(index=False)))

        print(f"\ncapture sessions : {df['session'].nunique():,}")
        top = df["session"].value_counts().head(5)
        for k, v in top.items():
            print(f"   {k or '<none>':20s} {v:4d} frames")

        if "rgb_w" in df.columns:
            sizes = df.loc[df["has_rgb"], ["rgb_w", "rgb_h"]].dropna()
            if len(sizes):
                uniq = sizes.drop_duplicates()
                print(f"\nRGB geometry  : {len(uniq)} distinct size(s)")
                for _, r in uniq.head(5).iterrows():
                    n = int(((sizes["rgb_w"] == r["rgb_w"]) & (sizes["rgb_h"] == r["rgb_h"])).sum())
                    print(f"   {int(r['rgb_w'])} x {int(r['rgb_h'])}   ({n:,} frames)")
                if len(uniq) > 1:
                    print("   !! non-constant RGB geometry - Cell 6 assumes constancy")
        if "hsi_w" in df.columns:
            hs = df.loc[df["has_hsi"], ["hsi_bands", "hsi_h", "hsi_w"]].dropna()
            if len(hs):
                uniq = hs.drop_duplicates()
                print(f"HSI geometry  : {len(uniq)} distinct shape(s)")
                for _, r in uniq.head(5).iterrows():
                    print(f"   {int(r['hsi_bands'])} bands x {int(r['hsi_h'])} x {int(r['hsi_w'])}")
                b = int(uniq.iloc[0]["hsi_bands"])
                if b != N_BANDS:
                    print(f"   !! band count {b} != configured N_BANDS {N_BANDS}")
                gain = (int(uniq.iloc[0]["hsi_h"]) * int(uniq.iloc[0]["hsi_w"])) / (256 * 256)
                print(f"   spatial pixels vs the 256x256 grid: {gain:.2f}x")

        scope = df[df["in_scope"]]
        if scope.empty:
            print("\n!! No RGB+HSI paired frames. Nothing downstream can run.")
            return
        n = min(w4_nsample.value, len(scope))
        sample = scope.sample(n=n, random_state=0)
        ncol = 3 if w4_use_ref.value else 2
        fig, axes = plt.subplots(n, ncol, figsize=(4.2 * ncol, 4.2 * n), squeeze=False)
        for ax_row, (_, row) in zip(axes, sample.iterrows()):
            try:
                ax_row[0].imshow(load_rgb(row["rgb_path"]))
                ax_row[0].set_title(f"RGB {row['stem']}", fontsize=8)
            except Exception as e:
                ax_row[0].text(0.5, 0.5, f"RGB read failed\n{e}", ha="center", fontsize=7)
            try:
                cube = load_cube(row["hyper_path"])
                ax_row[1].imshow(hsi_false_colour(cube))
                ax_row[1].set_title(f"HSI {cube.shape[1]}x{cube.shape[2]} false colour", fontsize=8)
            except Exception as e:
                ax_row[1].text(0.5, 0.5, f"cube read failed\n{e}", ha="center", fontsize=7)
            if ncol == 3:
                if row["ref_rgb_path"]:
                    ax_row[2].imshow(load_rgb(row["ref_rgb_path"]))
                    ax_row[2].set_title("reference RGB (256)", fontsize=8)
                else:
                    ax_row[2].text(0.5, 0.5, "no reference", ha="center", fontsize=8)
            for ax in ax_row:
                ax.axis("off")
        fig.suptitle("Modality Sample Check", y=1.005)
        plt.tight_layout()
        plt.show()

        w4_status.value = status_html("stage0_scope", n_scope=int(df["in_scope"].sum()))


def lock4():
    df = ARCHIVE.get("frame_index")
    if df is None or df.empty:
        print("!! Nothing to lock - run Preview first.")
        return
    scope = df[df["in_scope"]]
    params = {
        "input_root": str(w4_root.value),
        "roles": _roles_from_widgets(),
        "use_reference": bool(w4_use_ref.value),
        "n_scope": int(len(scope)),
        "n_sessions": int(scope["session"].nunique()),
    }
    lock_stage("stage0_scope", params, n_preview=int(w4_nsample.value))

    out_csv = DIR_TABLES / "frame_index.csv"
    df.to_csv(out_csv, index=False)
    (RESULTS_OUT_ROOT / "archive_layout.json").write_text(
        json.dumps(ARCHIVE["layout"], indent=2, default=str))
    globals()["FRAME_INDEX"] = df
    globals()["SCOPED_STEMS"] = list(scope["stem"])
    log_params("stage0_scope",
               outputs={"frame_index": str(out_csv),
                        "archive_layout": str(RESULTS_OUT_ROOT / "archive_layout.json")})
    print(f"\nframe index  -> {out_csv}")
    print(f"SCOPED_STEMS : {len(SCOPED_STEMS):,} frames with both RGB and HSI "
          f"across {scope['session'].nunique():,} sessions")
    w4_status.value = status_html("stage0_scope", n_scope=len(SCOPED_STEMS))


def batch4():
    print("Cell 5 has no batch step - the index is built during Preview and written on Lock.")


w4_bar, w4_buttons = make_action_bar(render_preview4, lock4, batch4, batch_label="(n/a)")
w4_scan_btn.on_click(on_scan)
for _w in [w4_root, w4_use_ref, w4_probe] + list(w4_roles.values()):
    _w.observe(lambda ch: invalidate_lock("stage0_scope", w4_buttons), names="value")

w4_status.value = status_html("stage0_scope")
display(widgets.VBox([
    widgets.HBox([w4_root, w4_scan_btn]),
    widgets.HBox([w4_use_ref, w4_probe, w4_nsample]),
    widgets.VBox(list(w4_roles.values())),
    w4_bar,
    w4_status,
]), w4_out)

on_scan()

In [ ]:
# === Cell 6: RGB to HSI Alignment ===
# The geometric transform is not fitted: it comes from the dataset authors' source and
# is corrected by two measured residual offsets.
#   spectralwaste/alignment/image_alignment.py :: multimodal_manual_align
#       rgb_align   = rgb[:1190, :]      (1200,1184) -> (1190,1184)
#       hyper_align = hyper[:, :605]     (600, 640)  -> (600, 605)
#       both then resize()d to a common square (skimage, order=0, nearest)
# Both crops are origin-anchored, so the composition is a per-axis scale (axes differ
# by 1.344%); the common output size cancels.
#
# The cameras are not co-located, so two residual offsets are measured here:
#   DX  cross-track, a single constant (-8 px), band-independent
#   DY  scan axis, per capture session
# The remaining ~3.3 px scene-depth parallax is handled by mask erosion (ERODE_PX)
# in the spectra extraction cell.

ALIGN = {"offsets": None, "dx": None, "source_csv": None, "method": None}


# --- Transform ---

def rgb_to_hsi_xy(x, y, dx=0.0, dy=0.0):
    """RGB pixel coords -> HSI pixel coords. skimage half-pixel convention."""
    xh = (np.asarray(x, dtype=np.float64) + 0.5) * ALIGN_SX - 0.5 + dx
    yh = (np.asarray(y, dtype=np.float64) + 0.5) * ALIGN_SY - 0.5 + dy
    return xh, yh


def hsi_to_rgb_xy(xh, yh, dx=0.0, dy=0.0):
    xr = (np.asarray(xh, dtype=np.float64) - dx + 0.5) / ALIGN_SX - 0.5
    yr = (np.asarray(yh, dtype=np.float64) - dy + 0.5) / ALIGN_SY - 0.5
    return xr, yr


class RgbToHsi:
    """Anisotropic scale + per-session offset. No fitting, no per-frame state."""

    def __init__(self, dx=0.0, dy=0.0, hsi_shape=None, session="", source="measured"):
        self.dx, self.dy = float(dx), float(dy)
        self.hsi_shape = hsi_shape or (HSI_NATIVE[1], HSI_NATIVE[0])
        self.session = session
        self.source = source

    def map_points(self, xs, ys):
        return rgb_to_hsi_xy(xs, ys, self.dx, self.dy)

    def map_mask(self, mask_rgb, out_shape=None):
        """Boolean mask in raw-RGB space -> boolean mask on the raw-HSI grid.

        RGB rows >= RGB_CROP_ROWS have no HSI counterpart and are dropped; objects
        touching that band are dropped in Stage 2 (drop_hsi_edge).
        """
        out_shape = out_shape or self.hsi_shape
        ys, xs = np.nonzero(mask_rgb)
        out = np.zeros(out_shape, dtype=bool)
        if xs.size == 0:
            return out
        keep = ys < RGB_CROP_ROWS
        xs, ys = xs[keep], ys[keep]
        if xs.size == 0:
            return out
        hx, hy = self.map_points(xs, ys)
        hx = np.rint(hx).astype(np.int64)
        hy = np.rint(hy).astype(np.int64)
        ok = (hx >= 0) & (hx < out_shape[1]) & (hy >= 0) & (hy < out_shape[0])
        out[hy[ok], hx[ok]] = True
        return out

    def warp_rgb(self, rgb_arr, out_shape=None):
        """Render an RGB frame on the HSI grid. Returns (image, valid_mask)."""
        out_shape = out_shape or self.hsi_shape
        H, W = out_shape
        yy, xx = np.indices((H, W))
        xr, yr = hsi_to_rgb_xy(xx, yy, self.dx, self.dy)
        xi, yi = np.rint(xr).astype(int), np.rint(yr).astype(int)
        ok = ((xi >= 0) & (xi < rgb_arr.shape[1]) &
              (yi >= 0) & (yi < min(RGB_CROP_ROWS, rgb_arr.shape[0])))
        out = np.zeros((H, W, 3), dtype=rgb_arr.dtype)
        out[ok] = rgb_arr[yi[ok], xi[ok]]
        return out, ok

    def to_dict(self):
        return {"session": self.session, "dx": self.dx, "dy": self.dy,
                "scale_x": ALIGN_SX, "scale_y": ALIGN_SY,
                "rgb_crop_rows": RGB_CROP_ROWS, "hsi_crop_cols": HSI_CROP_COLS,
                "source": self.source}


# --- Offset estimation ---

def _sobel_z(img2d):
    """Z-scored Sobel gradient magnitude; comparable across RGB frames and HSI bands."""
    x = np.asarray(img2d, dtype=np.float64)
    m = np.hypot(ndimage.sobel(x, axis=0), ndimage.sobel(x, axis=1))
    return (m - m.mean()) / (m.std() + 1e-8)


def _pct_u8(a):
    lo, hi = np.percentile(a, (1, 99))
    return np.clip((np.asarray(a, dtype=np.float64) - lo) / (hi - lo + 1e-9) * 255,
                   0, 255).astype(np.uint8)


def _offset_scan(gh, gr, core, axis, rng, other):
    """1-D edge-correlation scan along one axis. Returns the score curve."""
    out = []
    for v in rng:
        dy, dx = (v, other) if axis == 0 else (other, v)
        m = core & np.roll(np.roll(core, dy, 0), dx, 1)
        if not m.any():
            out.append(-np.inf)
            continue
        out.append(float(np.mean(gh[m] * np.roll(np.roll(gr, dy, 0), dx, 1)[m])))
    return np.asarray(out)


def _peak(curve, rng):
    """Argmax plus prominence (peak minus median); callers reject low-prominence peaks."""
    i = int(np.argmax(curve))
    finite = curve[np.isfinite(curve)]
    prom = float(curve[i] - np.median(finite)) if finite.size else 0.0
    return list(rng)[i], float(curve[i]), prom


def measure_frame_offset(row, dx, search, band_frac, erode_iter=24):
    """Estimate (dy, dx) for one frame against the current transform."""
    rgb = load_rgb(row["rgb_path"])
    cube = load_cube(row["hyper_path"])
    H, W = cube.shape[1], cube.shape[2]
    tf = RgbToHsi(dx=dx, dy=0.0, hsi_shape=(H, W))
    warp, ok = tf.warp_rgb(rgb, (H, W))
    gr = _sobel_z(_pct_u8(warp.mean(axis=2)))
    gh = _sobel_z(_pct_u8(cube[int(cube.shape[0] * band_frac)]))
    core = ndimage.binary_erosion(ok, iterations=int(erode_iter))
    if not core.any():
        raise RuntimeError("empty overlap core - erode_iter too large")

    # Alternating 1-D scans in two passes: the first y-scan uses the assumed dx, so a
    # second pass is needed when that assumption is off.
    rng = range(-int(search), int(search) + 1)
    bdy, bdx = 0, int(dx)
    prom_y = prom_x = 0.0
    peak = -np.inf
    for _ in range(2):
        cy = _offset_scan(gh, gr, core, 0, rng, bdx)
        bdy, _, prom_y = _peak(cy, rng)
        cx = _offset_scan(gh, gr, core, 1, rng, bdy)
        bdx, peak, prom_x = _peak(cx, rng)
    return {"dy": int(bdy), "dx": int(bdx), "peak": float(peak),
            "prom_y": prom_y, "prom_x": prom_x,
            "censored": bool(abs(bdy) >= search or abs(bdx) >= search)}


# --- Widgets ---

w5_dx = widgets.IntText(value=-8, description="DX cross-track:",
                        style={"description_width": "160px"},
                        layout=widgets.Layout(width="240px"))
w5_search = widgets.IntSlider(value=20, min=6, max=48, description="Search radius px:",
                              style={"description_width": "160px"},
                              layout=widgets.Layout(width="420px"))
w5_band = widgets.FloatSlider(value=0.25, min=0.05, max=0.95, step=0.05,
                              description="HSI band frac:", readout_format=".2f",
                              style={"description_width": "160px"},
                              layout=widgets.Layout(width="420px"))
w5_frames = widgets.IntSlider(value=9, min=1, max=20, description="Frames per session:",
                              style={"description_width": "160px"},
                              layout=widgets.Layout(width="420px"))
w5_erode = widgets.IntSlider(value=24, min=4, max=48, description="Overlap erode px:",
                             style={"description_width": "160px"},
                             layout=widgets.Layout(width="420px"))
w5_prom_min = widgets.FloatSlider(value=0.02, min=0.0, max=0.20, step=0.005,
                                  description="Min peak prominence:", readout_format=".3f",
                                  style={"description_width": "160px"},
                                  layout=widgets.Layout(width="440px"))
w5_reuse = widgets.Text(value="", description="Reuse RUN_ID:", placeholder="blank = measure now",
                        style={"description_width": "160px"},
                        layout=widgets.Layout(width="460px"))
w5_nprev = widgets.IntSlider(value=2, min=1, max=6, description="Preview frames:",
                             style={"description_width": "160px"},
                             layout=widgets.Layout(width="380px"))
w5_status = widgets.HTML()
w5_out = widgets.Output()


def _scoped():
    df = globals().get("FRAME_INDEX")
    return pd.DataFrame() if df is None else df[df["in_scope"]]


def _offsets_csv_for(run_id):
    return (RESULTS_OUT_ROOT.parent / str(run_id) / "tables" /
            "session_alignment_offsets.csv")


# --- Preview ---

def render_preview5():
    with w5_out:
        clear_output(wait=True)
        cand = _scoped()
        if cand.empty:
            print("!! No frames in scope - run Cell 5 first.")
            return

        print(f"scale x = {ALIGN_SX:.7f}   scale y = {ALIGN_SY:.7f}   "
              f"anisotropy = {(ALIGN_SX / ALIGN_SY - 1) * 100:+.3f}%")
        print(f"source  : authors' multimodal_manual_align "
              f"(rgb[:{RGB_CROP_ROWS}, :], hyper[:, :{HSI_CROP_COLS}])")
        print(f"DX      : {int(w5_dx.value):+d} px (measured constant, band-independent)")
        print()

        sample = cand.sample(n=min(int(w5_nprev.value), len(cand)), random_state=0)
        t0 = time.time()
        for _, row in sample.iterrows():
            try:
                res = measure_frame_offset(row, int(w5_dx.value), int(w5_search.value),
                                           float(w5_band.value), int(w5_erode.value))
            except Exception as e:
                print(f"!! {row['stem']}: {type(e).__name__}: {e}")
                continue

            rgb = load_rgb(row["rgb_path"])
            cube = load_cube(row["hyper_path"])
            H, W = cube.shape[1], cube.shape[2]
            tf = RgbToHsi(dx=int(w5_dx.value), dy=res["dy"], hsi_shape=(H, W),
                          session=row.get("session", ""))
            warp, ok = tf.warp_rgb(rgb, (H, W))
            hsi_vis = hsi_false_colour(cube)

            # Wide canvas and top margin keep panel labels and the suptitle apart.
            fig, axes = plt.subplots(1, 4, figsize=(_FW * 2.6, _FW * 0.85))
            axes[0].imshow(hsi_vis)
            _panel_label(axes[0], "a", "HSI False Colour")
            axes[1].imshow(warp)
            _panel_label(axes[1], "b", "RGB In HSI Geometry")

            g1 = _pct_u8(hsi_vis.mean(axis=2))
            g2 = _pct_u8(warp.mean(axis=2))
            tile = max(8, H // 16)
            yy, xx = np.indices((H, W))
            board = (((yy // tile) + (xx // tile)) % 2).astype(bool)
            axes[2].imshow(np.where(board, g1, g2), cmap="gray")
            _panel_label(axes[2], "c", "Checkerboard")

            axes[3].imshow(ok, cmap="gray")
            _panel_label(axes[3], "d", "Valid Overlap")
            for ax in axes:
                ax.axis("off")
            fig.suptitle(f"Alignment Preview - {row['stem']}   "
                         f"dy={res['dy']:+d}  dx={res['dx']:+d}", y=1.06, fontsize=13)
            plt.tight_layout(rect=[0, 0, 1, 0.90])
            plt.show()

            print(f"{row['stem']}  session {row.get('session', '')}")
            print(f"  measured offset : dy={res['dy']:+d}  dx={res['dx']:+d} px")
            print(f"  peak prominence : y {res['prom_y']:.3f}   x {res['prom_x']:.3f}"
                  f"   {'CENSORED' if res['censored'] else ''}")
            print(f"  valid overlap   : {ok.mean() * 100:.1f}% of the HSI grid")

        spf = (time.time() - t0) / max(len(sample), 1)
        n_sess = cand["session"].nunique()
        n_meas = n_sess * int(w5_frames.value)
        est = estimate_batch_runtime(spf, n_meas)
        w5_status.value = status_html("alignment", n_scope=len(cand), est=est)
        print(f"\n{n_sess} sessions x {int(w5_frames.value)} frames = {n_meas:,} measurements")
        print(f"projected batch : {est}")


# --- Lock and batch ---

def lock5():
    params = {
        "method": "authors_constants_plus_measured_offsets",
        "source_function": "spectralwaste.alignment.image_alignment.multimodal_manual_align",
        "rgb_crop_rows": int(RGB_CROP_ROWS), "hsi_crop_cols": int(HSI_CROP_COLS),
        "scale_x": float(ALIGN_SX), "scale_y": float(ALIGN_SY),
        "dx_px": int(w5_dx.value),
        "dy_granularity": "per_capture_session",
        "search_radius_px": int(w5_search.value),
        "hsi_band_frac": float(w5_band.value),
        "frames_per_session": int(w5_frames.value),
        "overlap_erode_px": int(w5_erode.value),
        "min_peak_prominence": float(w5_prom_min.value),
        "reuse_run_id": w5_reuse.value.strip(),
    }
    lock_stage("alignment", params, n_preview=int(w5_nprev.value))
    w5_status.value = status_html("alignment", n_scope=len(_scoped()))


def _write_alignment_methods(p, off, stats):
    """Write alignment_methods.md: transform, measurement settings and results of this run."""
    md = [
        "# Alignment Methods - Run " + RUN_ID,
        "",
        f"Generated {utcnow()} | CONFIG_HASH `{CONFIG_HASH}`",
        "",
        "## Transform",
        "",
        "RGB masks are mapped to the hyperspectral grid with the geometric transform",
        "published by the dataset authors, corrected by two measured residual terms.",
        "The authors' `multimodal_manual_align` crops each modality to a square",
        f"(`rgb[:{RGB_CROP_ROWS}, :]`, `hyper[:, :{HSI_CROP_COLS}]`, both anchored at the",
        "origin) and resizes both to a common grid. The common size cancels in the",
        "composition, giving a pure per-axis scale:",
        "",
        "```",
        f"x_hsi = (x_rgb + 0.5) * {ALIGN_SX:.7f} - 0.5 + DX",
        f"y_hsi = (y_rgb + 0.5) * {ALIGN_SY:.7f} - 0.5 + DY(session)",
        "```",
        "",
        f"The two axis scales differ by {(ALIGN_SX / ALIGN_SY - 1) * 100:.3f} %; the crops",
        "square up each modality independently and do not register the cameras against",
        "each other. The residual displacement that leaves was measured directly.",
        "",
        "| term | value | granularity | basis |",
        "|---|---|---|---|",
        f"| scale x | {ALIGN_SX:.7f} | constant | authors' source |",
        f"| scale y | {ALIGN_SY:.7f} | constant | authors' source |",
        f"| DX | {stats['dx']:+d} px | constant | measured, this run |",
        f"| DY | table | per capture session | measured, this run |",
        f"| residual | ~{stats['within_session_std']:.1f} px | - | scene-depth parallax |",
        "",
        "## Measurement",
        "",
        f"- Estimator: Sobel gradient-magnitude edge correlation, 1-D scans on each axis",
        f"- Search radius: +/- {stats['search']} px",
        f"- HSI band: fraction {stats['band_frac']:.2f} of {N_BANDS} "
        f"(~{WAVELENGTHS_NM[int(N_BANDS * stats['band_frac'])]:.0f} nm)",
        f"- Overlap eroded by {stats['erode']} px before correlation",
        f"- Frames per session: {stats['frames_per_session']} (median aggregated)",
        f"- Sessions measured: {stats['n_sessions']} of {stats['n_sessions_total']}",
        f"- Frames measured: {stats['n_frames']}",
        f"- Rejected on peak prominence < {stats['prom_min']:.3f}: {stats['n_rejected']}",
        "",
        "## Results",
        "",
        f"- DY range across sessions: {stats['dy_min']:+d} to {stats['dy_max']:+d} px",
        f"- DY between-session std: {stats['between_session_std']:.2f} px",
        f"- DY within-session std: {stats['within_session_std']:.2f} px",
        f"- DX median across sessions: {stats['dx_median']:+.1f} px "
        f"(IQR {stats['dx_iqr']:.1f})",
        "",
        "A per-session DY is required because a single global constant would be wrong by",
        "the full between-session range at the extremes. The within-session spread is",
        "scene-driven (frames in one session are seconds apart with different objects at",
        "different heights) and is not removable by any global transform.",
        "",
        "## Downstream consequences",
        "",
        f"- Region masks are eroded by {ERODE_PX_DEFAULT} px before hyperspectral sampling,",
        "  sized to the parallax residual. Misregistration contaminates boundary pixels;",
        "  interiors survive. The cost falls hardest on the smallest regions (caps) and is",
        "  monitored by the region-pixel-count histogram in Cell 13.",
        f"- Objects whose mask touches RGB row >= {RGB_CROP_ROWS} are dropped, not clipped",
        f"  ({(RGB_NATIVE[1] - RGB_CROP_ROWS) / RGB_NATIVE[1] * 100:.2f} % of frame height",
        "  has no hyperspectral counterpart). A dropped object is visible in the funnel; a",
        "  clipped one silently yields a spectrum from a partial mask.",
        "",
        "## Rejected approaches",
        "",
        "- **Fitting the transform against the shipped 256x256 preprocessed archive.**",
        "  That archive is nearest-neighbour downsampled and carries its own residual, so",
        "  anything fitted to it inherits that error as a floor.",
        "- **A single isotropic scale.** The two axes differ by 1.344 %; an isotropic fit",
        "  lands between them and produces error growing toward the frame edges.",
        "- **Per-frame offset estimation.** The whole-frame estimate follows whichever",
        "  object dominates the frame edges, so it chases parallax rather than removing it.",
        "- **A single global DY.** Falsified by the between-session spread above.",
        "",
        "## Source",
        "",
        "Casao et al., *SpectralWaste Dataset: Multimodal Data for Waste Sorting*,",
        "IROS 2024 (arXiv:2403.18033); `ferpb/spectralwaste-dataset`,",
        "`spectralwaste/alignment/image_alignment.py`.",
        "",
    ]
    p.write_text("\n".join(md))


def batch5():
    p = require_locked("alignment")
    cand = _scoped()
    if cand.empty:
        print("!! Nothing in scope.")
        return
    out_csv = DIR_TABLES / "session_alignment_offsets.csv"

    with w5_out:
        clear_output(wait=True)

        # --- Reuse a previous run's offsets ---
        reuse = p.get("reuse_run_id", "").strip()
        if reuse:
            src = _offsets_csv_for(reuse)
            if not src.exists():
                print(f"!! No offsets table at {src}")
                return
            off = pd.read_csv(src)
            off.to_csv(out_csv, index=False)
            reused_dx = (int(off["dx_applied"].iloc[0])
                         if "dx_applied" in off.columns and len(off) else int(p["dx_px"]))
            ALIGN.update({"offsets": off, "dx": reused_dx,
                          "source_csv": str(src), "method": f"reused:{reuse}"})
            print(f"Reused {len(off):,} session offsets from run {reuse}")
            print(f"  {src}\n  -> {out_csv}")
            n_sess_total = cand["session"].astype(str).nunique()
            log_params("alignment", outputs={"offsets_csv": str(out_csv)},
                       results={"n_sessions": int(len(off)), "n_sessions_total": int(n_sess_total),
                                "dx": reused_dx, "reused_from": reuse})
            return

        # --- Measure ---
        sessions = sorted(cand["session"].astype(str).unique())
        per_sess = int(p["frames_per_session"])
        jobs = []
        for s in sessions:
            g = cand[cand["session"].astype(str) == s].sort_values("stem")
            step = max(1, len(g) // per_sess)
            jobs.append((s, g.iloc[::step].head(per_sess)))
        n_total = sum(len(g) for _, g in jobs)

        rep = ProgressReporter("alignment", n_total)
        rows = []
        for s, g in jobs:
            for _, row in g.iterrows():
                try:
                    r = measure_frame_offset(row, int(p["dx_px"]), int(p["search_radius_px"]),
                                             float(p["hsi_band_frac"]),
                                             int(p["overlap_erode_px"]))
                    rows.append({"session": s, "stem": row["stem"], **r})
                except Exception as e:
                    rows.append({"session": s, "stem": row["stem"],
                                 "error": f"{type(e).__name__}: {e}"})
                rep.update()
        rep.close()

        frames = pd.DataFrame(rows)
        frames.to_csv(DIR_TABLES / "frame_alignment_measurements.csv", index=False)

        ok = frames[frames.get("dy").notna()] if "dy" in frames.columns else pd.DataFrame()
        if ok.empty:
            print("!! Every measurement failed - inspect frame_alignment_measurements.csv")
            return
        n_before = len(ok)
        ok = ok[(~ok["censored"]) & (ok["prom_y"] >= float(p["min_peak_prominence"]))]
        n_rejected = n_before - len(ok)
        if ok.empty:
            print("!! Every measurement was rejected. Lower the prominence floor.")
            return

        agg = (ok.groupby("session")
                 .agg(n_frames=("dy", "size"), dy=("dy", "median"), dx=("dx", "median"),
                      dy_std=("dy", "std"), dx_std=("dx", "std"),
                      prom_y=("prom_y", "median"), peak=("peak", "median"))
                 .reset_index())
        agg["dy"] = agg["dy"].round().astype(int)
        agg["dx"] = agg["dx"].round().astype(int)
        agg["dx_applied"] = int(p["dx_px"])
        agg["scale_x"], agg["scale_y"] = ALIGN_SX, ALIGN_SY
        agg["rgb_crop_rows"], agg["hsi_crop_cols"] = RGB_CROP_ROWS, HSI_CROP_COLS
        agg["run_id"] = RUN_ID
        agg.to_csv(out_csv, index=False)

        ALIGN.update({"offsets": agg, "dx": int(p["dx_px"]),
                      "source_csv": str(out_csv), "method": p["method"]})

        within = float(ok.groupby("session")["dy"].std().median())
        between = float(agg["dy"].std())
        stats = {
            "dx": int(p["dx_px"]), "search": int(p["search_radius_px"]),
            "band_frac": float(p["hsi_band_frac"]), "erode": int(p["overlap_erode_px"]),
            "frames_per_session": per_sess, "prom_min": float(p["min_peak_prominence"]),
            "n_sessions": int(len(agg)), "n_sessions_total": len(sessions),
            "n_frames": int(len(ok)), "n_rejected": int(n_rejected),
            "dy_min": int(agg["dy"].min()), "dy_max": int(agg["dy"].max()),
            "within_session_std": within, "between_session_std": between,
            "dx_median": float(agg["dx"].median()),
            "dx_iqr": float(agg["dx"].quantile(.75) - agg["dx"].quantile(.25)),
        }

        print(f"sessions measured : {len(agg):,} of {len(sessions):,}")
        print(f"frames used       : {len(ok):,}  ({n_rejected:,} rejected)")
        print(f"DY range          : {stats['dy_min']:+d} to {stats['dy_max']:+d} px")
        print(f"DY std            : within-session {within:.2f}  "
              f"between-session {between:.2f} px")
        print(f"DX median         : {stats['dx_median']:+.1f} px "
              f"(IQR {stats['dx_iqr']:.1f}) - applied constant {int(p['dx_px']):+d}")
        if between <= within:
            print("\n!! Between-session spread does not exceed within-session spread.")
            print("   A per-session table is not justified by this measurement - check the")
            print("   prominence floor and the frames-per-session count before trusting it.")

        # QA figure
        fig, axes = plt.subplots(1, 3, figsize=(_FW, _FW / 3))
        axes[0].hist(agg["dy"], bins=range(int(agg["dy"].min()), int(agg["dy"].max()) + 2),
                     color=C_EDT_ALIGN, edgecolor="white", linewidth=0.5)
        _panel_label(axes[0], "a", "Scan-Axis Offset By Session")
        axes[0].set_xlabel("DY (px)")
        axes[0].set_ylabel("Sessions")

        axes[1].hist(agg["dx"], bins=range(int(agg["dx"].min()), int(agg["dx"].max()) + 2),
                     color=C_GRID_ALIGN, edgecolor="white", linewidth=0.5)
        axes[1].axvline(int(p["dx_px"]), color="black", ls="--", lw=1.2, label="Applied")
        _panel_label(axes[1], "b", "Cross-Track Offset By Session")
        axes[1].set_xlabel("DX (px)")
        axes[1].set_ylabel("Sessions")
        axes[1].legend(loc="upper right", frameon=False)

        axes[2].scatter(agg["n_frames"], agg["dy_std"], s=18, color=C_EDT_ALIGN)
        axes[2].axhline(within, color="black", ls="--", lw=1.2, label="Median")
        _panel_label(axes[2], "c", "Within-Session Spread")
        axes[2].set_xlabel("Frames Measured")
        axes[2].set_ylabel("DY Std (px)")
        axes[2].legend(loc="upper right", frameon=False)
        fig.suptitle("Alignment Offset Measurement", y=1.02)
        plt.tight_layout()
        fig.savefig(DIR_FIGURES / "alignment_offsets.png", dpi=150, bbox_inches="tight")
        plt.show()

        meth = RESULTS_OUT_ROOT / "alignment_methods.md"
        _write_alignment_methods(meth, agg, stats)
        (RESULTS_OUT_ROOT / "alignment_parameters.json").write_text(
            json.dumps({"params": p, "stats": stats,
                        "transform": {"scale_x": ALIGN_SX, "scale_y": ALIGN_SY,
                                      "rgb_crop_rows": RGB_CROP_ROWS,
                                      "hsi_crop_cols": HSI_CROP_COLS,
                                      "dx_px": int(p["dx_px"]),
                                      "erode_px_default": ERODE_PX_DEFAULT},
                        "run_id": RUN_ID, "config_hash": CONFIG_HASH},
                       indent=2, default=str))
        print(f"\noffsets    -> {out_csv}")
        print(f"methods    -> {meth}")
        print(f"parameters -> {RESULTS_OUT_ROOT / 'alignment_parameters.json'}")

        log_params("alignment",
                   outputs={"offsets_csv": str(out_csv),
                            "frame_measurements_csv":
                                str(DIR_TABLES / "frame_alignment_measurements.csv"),
                            "methods_md": str(meth),
                            "qa_figure": str(DIR_FIGURES / "alignment_offsets.png")},
                   results=stats)


# --- Consumer API ---

def load_alignment(run_id=None):
    """Reload a measured offsets table without re-running the batch."""
    src = _offsets_csv_for(run_id) if run_id else (DIR_TABLES / "session_alignment_offsets.csv")
    if not src.exists():
        raise FileNotFoundError(f"No alignment offsets at {src}")
    off = pd.read_csv(src)
    dx = int(off["dx_applied"].iloc[0]) if "dx_applied" in off.columns else int(w5_dx.value)
    ALIGN.update({"offsets": off, "dx": dx, "source_csv": str(src),
                  "method": f"loaded:{run_id or RUN_ID}"})
    print(f"loaded {len(off):,} session offsets from {src}")
    return off


def get_transform(stem=None, session=None, hsi_shape=None):
    """Transform for one frame. Falls back to the median DY if the session is unmeasured."""
    off = ALIGN.get("offsets")
    if off is None:
        p = (DIR_TABLES / "session_alignment_offsets.csv")
        if p.exists():
            off = load_alignment()
        else:
            raise RuntimeError("No alignment measured - run Cell 6 batch first.")
    if session is None and stem is not None:
        fi = globals().get("FRAME_INDEX")
        hit = fi[fi["stem"] == stem] if fi is not None else None
        session = str(hit["session"].iloc[0]) if hit is not None and len(hit) else None

    dx = ALIGN.get("dx", 0)
    row = off[off["session"].astype(str) == str(session)] if session is not None else off.iloc[0:0]
    if len(row):
        return RgbToHsi(dx=dx, dy=float(row["dy"].iloc[0]), hsi_shape=hsi_shape,
                        session=str(session), source="session")
    return RgbToHsi(dx=dx, dy=float(off["dy"].median()), hsi_shape=hsi_shape,
                    session=str(session or ""), source="median_fallback")


w5_bar, w5_buttons = make_action_bar(render_preview5, lock5, batch5,
                                     batch_label="Measure session offsets")
for _w in (w5_dx, w5_search, w5_band, w5_frames, w5_erode, w5_prom_min, w5_reuse):
    _w.observe(lambda ch: invalidate_lock("alignment", w5_buttons), names="value")

w5_status.value = status_html("alignment")
display(widgets.VBox([
    widgets.HBox([w5_dx, w5_search]),
    widgets.HBox([w5_band, w5_frames]),
    widgets.HBox([w5_erode, w5_prom_min]),
    widgets.HBox([w5_reuse, w5_nprev]),
    w5_bar,
    w5_status,
]), w5_out)

In [ ]:
# === Cell 7: SAM3 Runtime and GPU Dispatch ===
# In-process SAM3 for previews; batches run as subprocess workers (1..N GPUs) so the
# kernel holds no CUDA context during long runs. No tunables.
# Workers are a standalone .py started with Popen (multiprocessing spawn fails in
# Jupyter) via sys.executable, with one shard file and one log file per worker.

SAM3_WORKER_SRC = r'''#!/usr/bin/env python
"""Generated by 20260818_Spectral_Waste_Preprocessing.ipynb - do not edit by hand.
This copy is an artifact of the run that produced it."""
import argparse, json, sys, time
from pathlib import Path
import numpy as np
from PIL import Image

_state = {"pred": None, "last_image": None, "imgsz": None}


def get_predictor(weights, imgsz, conf):
    if _state["pred"] is None or _state["imgsz"] != imgsz:
        from ultralytics.models.sam import SAM3SemanticPredictor
        overrides = {"conf": conf, "task": "segment", "mode": "predict",
                     "model": str(weights), "imgsz": int(imgsz),
                     "save": False, "verbose": False}
        _state["pred"] = SAM3SemanticPredictor(overrides=overrides)
        _state["imgsz"] = imgsz
        _state["last_image"] = None
    return _state["pred"]


def run_sam3(image_path, prompts, conf, weights, imgsz):
    """Encoder cache: set_image dominates runtime, decode is cheap. Only re-encode
    when the image actually changes, so multi-prompt on one frame is near-free."""
    pred = get_predictor(weights, imgsz, conf)
    pred.args.conf = float(conf)
    if _state["last_image"] != str(image_path):
        pred.set_image(str(image_path))
        _state["last_image"] = str(image_path)
    return pred(text=list(prompts))[0]


def masks_to_bool(result):
    """(N, H, W) bool at ORIGINAL image resolution."""
    if result.masks is None or len(result.masks) == 0:
        return np.zeros((0, *result.orig_shape), dtype=bool)
    data = result.masks.data.cpu().numpy()
    H, W = result.orig_shape
    if data.shape[1:] != (H, W):
        out = np.zeros((data.shape[0], H, W), dtype=bool)
        for i, m in enumerate(data):
            im = Image.fromarray((m > 0.5).astype(np.uint8) * 255)
            out[i] = np.asarray(im.resize((W, H), Image.NEAREST)) > 127
        return out
    return data > 0.5


def _lookup_name(names, cls_id):
    """result.names is usually {cls_id: name}, but some SAM3 predictors return it as
    a plain list indexed by cls_id - .get() doesn't exist on a list, hence the crash.
    Handle both."""
    if isinstance(names, dict):
        return str(names.get(cls_id, cls_id))
    try:
        return str(names[cls_id])
    except (IndexError, TypeError):
        return str(cls_id)


def det_meta(result):
    if result.boxes is None or len(result.boxes) == 0:
        return [], [], []
    confs = [float(c) for c in result.boxes.conf.cpu().numpy()]
    clsid = [int(c) for c in result.boxes.cls.cpu().numpy()]
    names = [_lookup_name(result.names, c) for c in clsid]
    return confs, clsid, names


def save_npz(path, provenance=None, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = dict(arrays)
    if provenance is not None:
        payload["provenance"] = np.array(json.dumps(provenance, default=str))
    np.savez_compressed(path, **payload)


# ------------------------------------------------------------------ stage 1
def do_stage1(items, p, mask_dir, writer):
    weights, imgsz, conf = p["weights"], p["imgsz"], p["conf"]
    prompts = p["prompts"]
    min_area = int(p.get("min_area_px", 0))
    for it in items:
        t0 = time.time()
        try:
            res = run_sam3(it["rgb_path"], prompts, conf, weights, imgsz)
            masks = masks_to_bool(res)
            confs, clsid, names = det_meta(res)
            n_kept = 0
            for i in range(masks.shape[0]):
                m = masks[i]
                area = int(m.sum())
                if area < min_area:
                    continue
                ys, xs = np.nonzero(m)
                mp = Path(mask_dir) / f"{it['stem']}_inst{n_kept:03d}.npz"
                save_npz(mp, mask=m,
                         provenance={"stem": it["stem"], "source_rgb": it["rgb_path"],
                                     "prompt": names[i] if i < len(names) else "",
                                     "conf": confs[i] if i < len(confs) else None,
                                     "sam3_conf": conf, "imgsz": imgsz,
                                     "prompts": prompts, "run_id": p.get("run_id")})
                writer({"stem": it["stem"], "split": it.get("split", ""),
                        "instance_idx": n_kept,
                        "prompt": names[i] if i < len(names) else "",
                        "confidence": confs[i] if i < len(confs) else None,
                        "mask_area_px": area,
                        "x1": int(xs.min()), "y1": int(ys.min()),
                        "x2": int(xs.max()), "y2": int(ys.max()),
                        "img_w": int(m.shape[1]), "img_h": int(m.shape[0]),
                        "rgb_path": it["rgb_path"], "mask_path": str(mp),
                        "item_id": it["stem"], "error": ""})
                n_kept += 1
            if n_kept == 0:
                writer({"stem": it["stem"], "split": it.get("split", ""),
                        "instance_idx": -1, "prompt": "", "confidence": None,
                        "mask_area_px": 0, "rgb_path": it["rgb_path"], "mask_path": "",
                        "item_id": it["stem"], "error": ""})
        except Exception as e:
            writer({"stem": it["stem"], "instance_idx": -1, "item_id": it["stem"],
                    "error": f"{type(e).__name__}: {e}"})
        finally:
            _ = time.time() - t0


# ------------------------------------------------------------------ stage 3
def do_stage3(items, p, mask_dir, writer):
    """cap/label sub-segmentation on cutouts -> body/cap/label pseudo-GT.

    whole_object_mask is REPLAYED from the stage-1 mask through the canvas geometry,
    never derived from cutout pixel colours (dark object regions would read as
    background). Verified pixel-exact against stage 2 output in the previous pipeline.
    """
    weights, imgsz = p["weights"], p["imgsz"]
    cap_prompt, label_prompt = p["cap_prompt"], p["label_prompt"]
    # cap and label get their OWN confidence threshold - the two prompts detect very
    # different-looking regions (a whole cap vs. a printed label) and a single shared
    # conf was never a real constraint, just an accident of one widget doing two jobs.
    conf_map = {"cap": p["cap_conf"], "label": p["label_conf"]}
    for it in items:
        rec = {"item_id": it["cutout_id"], "cutout_id": it["cutout_id"],
               "stem": it.get("stem", ""), "cutout_path": it["cutout_path"], "error": ""}
        try:
            with np.load(it["whole_mask_path"], allow_pickle=False) as z:
                src_mask = z["mask"]
            ch, cw = int(it["canvas_h"]), int(it["canvas_w"])
            whole = np.zeros((ch, cw), dtype=bool)
            ys, xs = np.nonzero(src_mask)
            cy = ys - int(it["mask_bbox_y1"]) + int(it["canvas_offset_y"])
            cx = xs - int(it["mask_bbox_x1"]) + int(it["canvas_offset_x"])
            ok = (cy >= 0) & (cy < ch) & (cx >= 0) & (cx < cw)
            whole[cy[ok], cx[ok]] = True
            whole_area = int(whole.sum())
            rec["whole_area_px"] = whole_area
            if whole_area == 0:
                rec["error"] = "empty whole-object mask"
                writer(rec)
                continue

            region = {}
            for key, prompt in (("cap", cap_prompt), ("label", label_prompt)):
                try:
                    res = run_sam3(it["cutout_path"], [prompt], conf_map[key], weights, imgsz)
                    m = masks_to_bool(res)
                    confs, _, _ = det_meta(res)
                    union = m.any(axis=0) if m.shape[0] else np.zeros_like(whole)
                    if union.shape != whole.shape:
                        im = Image.fromarray(union.astype(np.uint8) * 255)
                        union = np.asarray(im.resize((cw, ch), Image.NEAREST)) > 127
                    union &= whole
                    region[key] = union
                    rec[f"{key}_n_det"] = int(m.shape[0])
                    rec[f"{key}_conf"] = float(max(confs)) if confs else np.nan
                    rec[f"{key}_frac"] = float(union.sum()) / whole_area
                except Exception as e:
                    region[key] = np.zeros_like(whole)
                    rec[f"{key}_n_det"] = 0
                    rec[f"{key}_conf"] = np.nan
                    rec[f"{key}_frac"] = 0.0
                    rec[f"{key}_error"] = f"{type(e).__name__}: {e}"

            body = whole & ~region["cap"] & ~region["label"]
            rec["body_frac"] = float(body.sum()) / whole_area
            rec["cap_present"] = bool(region["cap"].any())

            prov = {"cutout_id": it["cutout_id"], "source_cutout": it["cutout_path"],
                    "source_whole_mask": it["whole_mask_path"],
                    "cap_prompt": cap_prompt, "label_prompt": label_prompt,
                    "cap_conf": conf_map["cap"], "label_conf": conf_map["label"],
                    "imgsz": imgsz, "run_id": p.get("run_id")}
            mp = Path(mask_dir) / f"{it['cutout_id']}_regions.npz"
            save_npz(mp, body=body, cap=region["cap"], label=region["label"],
                     whole=whole, provenance=prov)
            rec["regions_mask_path"] = str(mp)
        except Exception as e:
            rec["error"] = f"{type(e).__name__}: {e}"
        writer(rec)


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", required=True, choices=["stage1", "stage3"])
    ap.add_argument("--items", required=True)
    ap.add_argument("--params", required=True)
    ap.add_argument("--out-csv", required=True)
    ap.add_argument("--mask-dir", required=True)
    args = ap.parse_args()

    items = json.loads(Path(args.items).read_text())
    p = json.loads(Path(args.params).read_text())
    Path(args.mask_dir).mkdir(parents=True, exist_ok=True)
    out = Path(args.out_csv)
    out.parent.mkdir(parents=True, exist_ok=True)

    import csv
    # FIXED per-stage header, not inferred from whichever row happens to be written
    # first. do_stage3 emits a "thin" row (no regions_mask_path at all) whenever an
    # object's whole-mask replay is empty; if that thin row landed first, the old
    # code (fieldnames=list(row.keys())) locked the CSV header to it and
    # extrasaction="ignore" silently dropped regions_mask_path/cap_frac/label_frac/etc.
    # from every later row - even fully successful ones. A fixed superset plus
    # restval="" makes the header order-independent.
    STAGE_FIELDS = {
        "stage1": ["stem", "split", "instance_idx", "prompt", "confidence", "mask_area_px",
                   "x1", "y1", "x2", "y2", "img_w", "img_h", "rgb_path", "mask_path",
                   "item_id", "error"],
        "stage3": ["item_id", "cutout_id", "stem", "cutout_path", "whole_area_px",
                   "cap_n_det", "cap_conf", "cap_frac", "cap_error",
                   "label_n_det", "label_conf", "label_frac", "label_error",
                   "body_frac", "cap_present", "regions_mask_path", "error"],
    }
    state = {"fh": None, "w": None}

    def writer(row):
        if state["w"] is None:
            state["fh"] = open(out, "w", newline="", buffering=1)
            state["w"] = csv.DictWriter(state["fh"], fieldnames=STAGE_FIELDS[args.stage],
                                        extrasaction="ignore", restval="")
            state["w"].writeheader()
        state["w"].writerow(row)
        state["fh"].flush()

    t0 = time.time()
    print(f"[worker] stage={args.stage} items={len(items)} device_env="
          f"{__import__('os').environ.get('CUDA_VISIBLE_DEVICES', 'unset')}", flush=True)
    if args.stage == "stage1":
        do_stage1(items, p, args.mask_dir, writer)
    else:
        do_stage3(items, p, args.mask_dir, writer)
    if state["fh"]:
        state["fh"].close()
    print(f"[worker] done in {time.time() - t0:.1f}s", flush=True)


if __name__ == "__main__":
    main()
'''


def write_worker_script():
    """Write the worker script into the run directory as part of the run's outputs."""
    p = DIR_WORKERS / "sam3_worker.py"
    p.write_text(SAM3_WORKER_SRC)
    p.chmod(0o755)
    return p


WORKER_PATH = write_worker_script()


# --- In-process SAM3 (previews only) ---

_PREVIEW = {"pred": None, "last_image": None, "imgsz": None}


def preview_predictor(imgsz, conf):
    if _PREVIEW["pred"] is None or _PREVIEW["imgsz"] != imgsz:
        from ultralytics.models.sam import SAM3SemanticPredictor
        overrides = {"conf": conf, "task": "segment", "mode": "predict",
                     "model": str(SAM3_WEIGHTS), "imgsz": int(imgsz),
                     "save": False, "verbose": False}
        _PREVIEW["pred"] = SAM3SemanticPredictor(overrides=overrides)
        _PREVIEW["imgsz"] = imgsz
        _PREVIEW["last_image"] = None
    return _PREVIEW["pred"]


def sam3_preview(image_path, prompts, conf, imgsz):
    pred = preview_predictor(imgsz, conf)
    pred.args.conf = float(conf)
    if _PREVIEW["last_image"] != str(image_path):
        pred.set_image(str(image_path))
        _PREVIEW["last_image"] = str(image_path)
    return pred(text=list(prompts))[0]


def release_preview_model():
    """Free the notebook's preview model and CUDA cache before a batch."""
    _PREVIEW["pred"] = None
    _PREVIEW["last_image"] = None
    _PREVIEW["imgsz"] = None
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


def result_masks_bool(result):
    if result.masks is None or len(result.masks) == 0:
        return np.zeros((0, *result.orig_shape), dtype=bool)
    data = result.masks.data.cpu().numpy()
    H, W = result.orig_shape
    if data.shape[1:] != (H, W):
        out = np.zeros((data.shape[0], H, W), dtype=bool)
        for i, m in enumerate(data):
            im = Image.fromarray((m > 0.5).astype(np.uint8) * 255)
            out[i] = np.asarray(im.resize((W, H), Image.NEAREST)) > 127
        return out
    return data > 0.5


def _lookup_name(names, cls_id):
    """Class name from result.names, which may be a dict or a list.

    Identical copy of the helper in SAM3_WORKER_SRC; keep both in sync.
    """
    if isinstance(names, dict):
        return str(names.get(cls_id, cls_id))
    try:
        return str(names[cls_id])
    except (IndexError, TypeError):
        return str(cls_id)


def result_meta(result):
    if result.boxes is None or len(result.boxes) == 0:
        return [], []
    confs = [float(c) for c in result.boxes.conf.cpu().numpy()]
    names = [_lookup_name(result.names, int(c)) for c in result.boxes.cls.cpu().numpy()]
    return confs, names


# --- Dispatch ---

def run_sam3_batch(stage, items, params, mask_dir, key_col="item_id"):
    """Shard `items` across available GPUs, run the worker, aggregate, report ETA.

    Round-robin sharding keeps per-worker cost balanced when item cost varies.
    Progress is polled from the worker checkpoint CSVs.
    """
    plan = resolve_compute_plan()
    n_workers = plan["n_workers"] if plan["mode"] == "multi" else 1
    devices = plan["devices"] or [None]

    # Resume: skip items already in the checkpoint.
    _, done = load_checkpoint(stage, key_col=key_col)
    todo = [it for it in items if str(it.get(key_col, it.get("stem"))) not in done]
    if done:
        print(f"[{stage}] resume: {len(done):,} already done, {len(todo):,} remaining")
    if not todo:
        print(f"[{stage}] nothing to do.")
        return pd.read_csv(checkpoint_path(stage)) if checkpoint_path(stage).exists() else pd.DataFrame()

    release_preview_model()
    Path(mask_dir).mkdir(parents=True, exist_ok=True)
    shard_dir = DIR_WORKERS / f"{stage}_shards"
    shard_dir.mkdir(parents=True, exist_ok=True)

    params = dict(params)
    params.setdefault("run_id", RUN_ID)
    params["weights"] = str(SAM3_WEIGHTS)
    params_path = shard_dir / "params.json"
    params_path.write_text(json.dumps(params, indent=2, default=str))

    shards = [todo[i::n_workers] for i in range(n_workers)]
    procs, shard_csvs, logs = [], [], []
    for wi, shard in enumerate(shards):
        if not shard:
            continue
        sp = shard_dir / f"shard_{wi:02d}.json"          # one file per worker
        sp.write_text(json.dumps(shard, default=str))
        oc = shard_dir / f"out_{wi:02d}.csv"
        lg = DIR_LOGS / f"{stage}_worker{wi:02d}.log"
        shard_csvs.append(oc)
        logs.append(lg)

        env = dict(os.environ)
        dev = devices[wi % len(devices)]
        if dev is not None:
            env["CUDA_VISIBLE_DEVICES"] = str(dev)
        cmd = [sys.executable, str(WORKER_PATH),          # same interpreter as the kernel
               "--stage", stage, "--items", str(sp), "--params", str(params_path),
               "--out-csv", str(oc), "--mask-dir", str(mask_dir)]
        fh = open(lg, "w", buffering=1)
        procs.append((wi, subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env), fh, oc))
        print(f"[{stage}] worker {wi} -> GPU {dev}  {len(shard):,} items  log={lg.name}")

    rep = ProgressReporter(stage, len(todo))
    try:
        while any(pr.poll() is None for _, pr, _, _ in procs):
            time.sleep(WORKER_POLL_S)
            # Count distinct stems, not rows: stage 1 writes one row per detection.
            rep.set_done(sum(count_checkpoint_items(c, key_col="stem") for c in shard_csvs))
            rep.report()
    finally:
        for wi, pr, fh, _ in procs:
            pr.wait()
            fh.close()

    failed = [(wi, pr.returncode, logs[i]) for i, (wi, pr, _, _) in enumerate(procs)
              if pr.returncode != 0]
    for wi, rc, lg in failed:
        print(f"!! worker {wi} exited {rc} - see {lg}")
        try:
            tail = lg.read_text().strip().splitlines()[-15:]
            for line in tail:
                print(f"     {line}")
        except OSError:
            pass

    frames = [pd.read_csv(c) for c in shard_csvs if c.exists() and c.stat().st_size > 0]
    new = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if len(new):
        append_checkpoint(stage, new.to_dict("records"))
    rep.set_done(len(new), force=True)
    rep.close(extra={"n_workers": len(procs), "mode": plan["mode"],
                     "devices": plan["devices"], "failed_workers": len(failed)})

    ck = checkpoint_path(stage)
    return pd.read_csv(ck) if ck.exists() else new


print(f"SAM3 worker written -> {WORKER_PATH}")
print(f"dispatch mode       : {COMPUTE_PLAN['mode']} ({COMPUTE_PLAN['n_workers']} worker(s))")

In [ ]:
# === Cell 8: Stage 1: SAM3 Object Detection ===
# Whole-object detection on full-resolution RGB (caps are only a few pixels wide on the
# 256 grid). The preview can compare prompts on the same frame sample.

STAGE1 = {"preview": None, "instances": None, "sec_per_frame": None}

PROMPT_PRESETS = {
    "bottle+container (recommended)": "bottle, container",
    "bottle": "bottle",
    "container": "container",
    "packaging": "packaging",
    "labeled bottles (previous run)": "labeled bottles",
    "wide": "bottle, container, packaging, jar, jug, tray",
}

w7_preset = widgets.Dropdown(options=list(PROMPT_PRESETS), value="bottle+container (recommended)",
                             description="Preset:", style={"description_width": "110px"},
                             layout=widgets.Layout(width="420px"))
w7_prompts = widgets.Text(value=PROMPT_PRESETS["bottle+container (recommended)"],
                          description="Prompts:", style={"description_width": "110px"},
                          layout=widgets.Layout(width="620px"))
w7_conf = widgets.FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05,
                              description="Conf:", readout_format=".2f",
                              style={"description_width": "110px"},
                              layout=widgets.Layout(width="380px"))
w7_imgsz = widgets.Dropdown(options=[644, 1008, 1288, 1652], value=644,
                            description="imgsz:", style={"description_width": "110px"},
                            layout=widgets.Layout(width="240px"))
w7_minarea = widgets.IntText(value=200, description="Min area px:",
                             style={"description_width": "110px"},
                             layout=widgets.Layout(width="220px"))
w7_nprev = widgets.IntSlider(value=4, min=1, max=12, description="Preview N:",
                             style={"description_width": "110px"},
                             layout=widgets.Layout(width="360px"))
w7_compare = widgets.Checkbox(value=False, indent=False,
                              description="Compare each prompt separately (settles D1)")
w7_seed = widgets.IntText(value=0, description="Seed:", style={"description_width": "110px"},
                          layout=widgets.Layout(width="180px"))
w7_status = widgets.HTML()
w7_out = widgets.Output()


def _prompts_list():
    return [s.strip() for s in w7_prompts.value.split(",") if s.strip()]


def _on_preset(change):
    w7_prompts.value = PROMPT_PRESETS[change["new"]]


w7_preset.observe(_on_preset, names="value")


def _scope_df():
    df = globals().get("FRAME_INDEX")
    if df is None:
        return pd.DataFrame()
    return df[df["in_scope"]]


def render_preview7():
    with w7_out:
        clear_output(wait=True)
        scope = _scope_df()
        if scope.empty:
            print("!! No frames in scope - lock Cell 5 first.")
            return
        prompts = _prompts_list()
        if not prompts:
            print("!! No prompts given.")
            return

        sample = scope.sample(n=min(w7_nprev.value, len(scope)), random_state=int(w7_seed.value))

        modes = [[p] for p in prompts] if w7_compare.value else [prompts]
        stats = []
        t_frames = []

        for mode_prompts in modes:
            label = "+".join(mode_prompts)
            n_rows = len(sample)
            fig, axes = plt.subplots(int(np.ceil(n_rows / 2)), 2,
                                     figsize=(_FW * 1.9, _FW * 0.95 * np.ceil(n_rows / 2)),
                                     squeeze=False)
            axes = axes.reshape(-1)
            tot_det, frames_hit = 0, 0
            for ax, (_, row) in zip(axes, sample.iterrows()):
                t0 = time.time()
                try:
                    res = sam3_preview(row["rgb_path"], mode_prompts, w7_conf.value, w7_imgsz.value)
                    masks = result_masks_bool(res)
                except Exception as e:
                    ax.text(0.5, 0.5, f"SAM3 failed\n{e}", ha="center", fontsize=7)
                    ax.axis("off")
                    continue
                dt = time.time() - t0
                t_frames.append(dt)

                n_kept = int(sum(masks[i].sum() >= int(w7_minarea.value)
                                 for i in range(masks.shape[0])))
                tot_det += n_kept
                frames_hit += int(n_kept > 0)

                # Standard Ultralytics rendering of all raw detections (before the
                # min-area filter); the title reports kept vs. raw counts.
                try:
                    plotted = res.plot(line_width=2, conf=True, labels=True)
                    ax.imshow(plotted[:, :, ::-1])
                except Exception:
                    ax.imshow(load_rgb(row["rgb_path"]))
                ax.set_title(f"{row['stem']}  n={n_kept} kept / {masks.shape[0]} raw  ({dt:.1f}s)",
                            fontsize=8)
                ax.axis("off")
            for ax in axes[len(sample):]:
                ax.axis("off")
            _titled_legend(fig, f"Stage 1 Preview - prompts: {label}  conf={w7_conf.value:.2f}  "
                                f"imgsz={w7_imgsz.value}", top_rect=0.92)
            plt.show()

            stats.append({"prompts": label, "n_frames": len(sample), "n_detections": tot_det,
                          "frames_with_det": frames_hit,
                          "pct_frames_hit": round(100 * frames_hit / len(sample), 1),
                          "det_per_frame": round(tot_det / len(sample), 2)})

        print("\nPROMPT COMPARISON (identical sample)")
        display(HTML(pd.DataFrame(stats).to_html(index=False)))

        spf = float(np.median(t_frames)) if t_frames else None
        STAGE1["sec_per_frame"] = spf
        n_scope = len(scope)
        est = estimate_batch_runtime(spf, n_scope)
        plan = resolve_compute_plan()
        if plan["mode"] == "multi" and spf:
            est = f"{estimate_batch_runtime(spf / plan['n_workers'], n_scope)} " \
                  f"({plan['n_workers']} GPUs)"
        print(f"\nmedian {spf:.2f} s/frame -> projected batch over {n_scope:,} frames: {est}"
              if spf else "\nno timing captured")
        if w7_imgsz.value > 644:
            print(f"note: imgsz={w7_imgsz.value} is "
                  f"{(w7_imgsz.value / 644) ** 2:.1f}x the pixels of the 644 baseline; "
                  f"the encoder dominates runtime.")
        w7_status.value = status_html("stage1", n_scope=n_scope, est=est)


def lock7():
    params = {
        "prompts": _prompts_list(),
        "conf": float(w7_conf.value),
        "imgsz": int(w7_imgsz.value),
        "min_area_px": int(w7_minarea.value),
    }
    lock_stage("stage1", params, n_preview=int(w7_nprev.value))
    w7_status.value = status_html("stage1", n_scope=len(_scope_df()),
                                  est=estimate_batch_runtime(STAGE1["sec_per_frame"],
                                                             len(_scope_df())))


def batch7():
    p = require_locked("stage1")
    scope = _scope_df()
    if scope.empty:
        print("!! No frames in scope.")
        return
    items = [{"stem": r["stem"], "split": r["split"], "rgb_path": r["rgb_path"]}
             for _, r in scope.iterrows()]
    with w7_out:
        clear_output(wait=True)
        print(f"Stage 1: {len(items):,} frames | prompts={p['prompts']} "
              f"conf={p['conf']} imgsz={p['imgsz']}")
        df = run_sam3_batch("stage1", items, p, DIR_STAGE1 / "masks", key_col="item_id")
        if df.empty:
            print("!! No output produced.")
            return

        det = df[df["instance_idx"] >= 0] if "instance_idx" in df.columns else df
        out_csv = DIR_STAGE1 / "instances.csv"
        det.to_csv(out_csv, index=False)
        STAGE1["instances"] = det

        n_frames = df["stem"].nunique()
        n_hit = det["stem"].nunique()
        errs = df[df.get("error", "").astype(str).str.len() > 0] if "error" in df.columns else df.iloc[:0]
        print(f"\nframes processed  : {n_frames:,}")
        print(f"frames with >=1   : {n_hit:,}  ({100 * n_hit / max(n_frames, 1):.1f}%)")
        print(f"instances         : {len(det):,}  ({len(det) / max(n_frames, 1):.2f}/frame)")
        print(f"errors            : {len(errs):,}")
        if len(det):
            print("\nby prompt:")
            display(HTML(det.groupby("prompt")
                         .agg(n=("mask_area_px", "size"),
                              median_conf=("confidence", "median"),
                              median_area=("mask_area_px", "median"))
                         .reset_index().to_html(index=False)))
        log_params("stage1", outputs={"instances": str(out_csv),
                                      "masks": str(DIR_STAGE1 / "masks")},
                   results={"n_frames": int(n_frames), "n_frames_with_det": int(n_hit),
                            "n_instances": int(len(det)), "n_errors": int(len(errs))})
        print(f"\ninstances -> {out_csv}")


w7_bar, w7_buttons = make_action_bar(render_preview7, lock7, batch7,
                                     batch_label="Run Stage 1 batch")
for _w in (w7_prompts, w7_conf, w7_imgsz, w7_minarea, w7_compare):
    _w.observe(lambda ch: invalidate_lock("stage1", w7_buttons), names="value")

w7_status.value = status_html("stage1")
display(widgets.VBox([
    widgets.HBox([w7_preset, w7_prompts]),
    widgets.HBox([w7_conf, w7_imgsz, w7_minarea]),
    widgets.HBox([w7_nprev, w7_seed, w7_compare]),
    w7_bar,
    w7_status,
]), w7_out)

In [ ]:
# === Cell 9: Stage 2: Cutout Generation ===
# Mask-isolated object on a black canvas at CANVAS_ASPECT_W:CANVAS_ASPECT_H (CPU only).
# mask_bbox_* and canvas_offset_* map cutout pixels back to the original frame and are
# used by Stage 3 and the HSI step.

STAGE2 = {"cutouts": None, "sec_per_item": None}


def make_cutout(rgb_arr, mask, aspect_w, aspect_h, canvas_scale):
    ys, xs = np.nonzero(mask)
    if xs.size == 0:
        return None, None
    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())
    obj_w, obj_h = max(x2 - x1, 1), max(y2 - y1, 1)

    tight = np.zeros((obj_h, obj_w, 3), dtype=np.uint8)
    local = mask[y1:y2, x1:x2]
    tight[local] = rgb_arr[y1:y2, x1:x2][local]

    target_ratio = aspect_w / aspect_h
    canvas_h = max(obj_h, obj_w / target_ratio) * canvas_scale
    canvas_w = canvas_h * target_ratio
    canvas_h, canvas_w = int(canvas_h), int(canvas_w)

    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    off_x, off_y = (canvas_w - obj_w) // 2, (canvas_h - obj_h) // 2
    canvas[off_y:off_y + obj_h, off_x:off_x + obj_w] = tight

    coord_map = {"mask_bbox_x1": x1, "mask_bbox_y1": y1,
                 "mask_bbox_x2": x2, "mask_bbox_y2": y2,
                 "canvas_offset_x": int(off_x), "canvas_offset_y": int(off_y),
                 "canvas_w": canvas_w, "canvas_h": canvas_h,
                 "obj_w": obj_w, "obj_h": obj_h,
                 "fill_frac": float(mask.sum()) / (canvas_h * canvas_w)}
    return canvas, coord_map


w8_aspect = widgets.HTML(
    f"<div style='font-family:monospace;font-size:12px;padding:6px'>"
    f"canvas aspect fixed at <b>{CANVAS_ASPECT_W}:{CANVAS_ASPECT_H}</b> "
    f"(deployment inference shape {YOLO_INFER_H}x{YOLO_INFER_W})</div>")
w8_scale = widgets.FloatSlider(value=2.0, min=1.0, max=8.0, step=0.1, description="Canvas scale:",
                               readout_format=".1f", style={"description_width": "110px"},
                               layout=widgets.Layout(width="420px"))
w8_minarea = widgets.IntText(value=200, description="Min area px:",
                             style={"description_width": "110px"},
                             layout=widgets.Layout(width="220px"))
w8_fmt = widgets.Dropdown(options=["png", "jpg"], value=CUTOUT_FORMAT, description="Format:",
                          style={"description_width": "110px"}, layout=widgets.Layout(width="200px"))
w8_nprev = widgets.IntSlider(value=6, min=1, max=16, description="Preview N:",
                             style={"description_width": "110px"},
                             layout=widgets.Layout(width="360px"))
w8_seed = widgets.IntText(value=0, description="Seed:", style={"description_width": "110px"},
                          layout=widgets.Layout(width="180px"))
w8_drop_edge = widgets.Checkbox(
    value=True, indent=False,
    description=f"Drop objects touching RGB row >= {RGB_CROP_ROWS} (no HSI counterpart)")
w8_status = widgets.HTML()
w8_out = widgets.Output()


def _instances_df():
    df = STAGE1.get("instances")
    if df is None:
        p = DIR_STAGE1 / "instances.csv"
        if p.exists():
            df = pd.read_csv(p)
            STAGE1["instances"] = df
    return df if df is not None else pd.DataFrame()


def render_preview8():
    with w8_out:
        clear_output(wait=True)
        inst = _instances_df()
        if inst.empty:
            print("!! No Stage 1 instances - run Cell 8 batch first.")
            return
        inst = inst[inst["mask_area_px"] >= int(w8_minarea.value)]
        if inst.empty:
            print("!! Min area filter removed every instance.")
            return

        sample = inst.sample(n=min(w8_nprev.value, len(inst)), random_state=int(w8_seed.value))
        n = len(sample)
        ncol = 3
        nrow = int(np.ceil(n / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(_FW * 0.62 * ncol, _FW * 0.40 * nrow),
                                 squeeze=False)
        axes = axes.reshape(-1)
        sizes, fills, times = [], [], []
        for ax, (_, row) in zip(axes, sample.iterrows()):
            t0 = time.time()
            try:
                arrays, _ = read_mask_npz(row["mask_path"])
                mask = arrays["mask"]
                rgb = load_rgb(row["rgb_path"])
                canvas, cm = make_cutout(rgb, mask, CANVAS_ASPECT_W, CANVAS_ASPECT_H,
                                         w8_scale.value)
            except Exception as e:
                ax.text(0.5, 0.5, f"failed\n{e}", ha="center", fontsize=7)
                ax.axis("off")
                continue
            times.append(time.time() - t0)
            if canvas is None:
                ax.axis("off")
                continue
            sizes.append((cm["canvas_w"], cm["canvas_h"]))
            fills.append(cm["fill_frac"])
            ax.imshow(canvas)
            ax.set_title(f"{row['stem']} #{int(row['instance_idx'])}\n"
                         f"{cm['canvas_w']}x{cm['canvas_h']}  fill={cm['fill_frac']:.3f}",
                         fontsize=7)
            ax.axis("off")
        for ax in axes[n:]:
            ax.axis("off")
        fig.suptitle(f"Stage 2 Preview - aspect {CANVAS_ASPECT_W}:{CANVAS_ASPECT_H}  "
                     f"scale {w8_scale.value:.1f}", y=1.01)
        plt.tight_layout()
        plt.show()

        if sizes:
            w = np.array([s[0] for s in sizes])
            h = np.array([s[1] for s in sizes])
            print(f"canvas width  : median {np.median(w):.0f} px  range {w.min()}-{w.max()}")
            print(f"canvas height : median {np.median(h):.0f} px  range {h.min()}-{h.max()}")
            print(f"object fill   : median {np.median(fills):.4f}")
        spf = float(np.median(times)) if times else None
        STAGE2["sec_per_item"] = spf
        est = estimate_batch_runtime(spf, len(inst))
        w8_status.value = status_html("stage2", n_scope=len(inst), est=est)
        print(f"\nprojected batch over {len(inst):,} instances: {est}")


def lock8():
    params = {"aspect_w": int(CANVAS_ASPECT_W), "aspect_h": int(CANVAS_ASPECT_H),
              "canvas_scale": float(w8_scale.value), "min_area_px": int(w8_minarea.value),
              "format": w8_fmt.value, "drop_hsi_edge": bool(w8_drop_edge.value),
              "rgb_crop_rows": int(RGB_CROP_ROWS)}
    lock_stage("stage2", params, n_preview=int(w8_nprev.value))
    w8_status.value = status_html("stage2", n_scope=len(_instances_df()))


def batch8():
    p = require_locked("stage2")
    inst = _instances_df()
    inst = inst[inst["mask_area_px"] >= p["min_area_px"]]
    if inst.empty:
        print("!! Nothing to cut out.")
        return
    out_dir = DIR_CUTOUTS / "cutouts"
    out_dir.mkdir(parents=True, exist_ok=True)
    # Capture session per stem, stored per cutout.
    _fi = globals().get("FRAME_INDEX")
    sess_map = (dict(zip(_fi["stem"], _fi["session"]))
                if _fi is not None and "session" in _fi.columns else {})

    with w8_out:
        clear_output(wait=True)
        rep = ProgressReporter("stage2", len(inst))
        rows, prov, n_skip, n_edge = [], [], 0, 0
        for _, row in inst.iterrows():
            try:
                arrays, _ = read_mask_npz(row["mask_path"])
                rgb = load_rgb(row["rgb_path"])
                # RGB rows >= RGB_CROP_ROWS have no HSI counterpart; such objects are
                # dropped rather than clipped
                if p.get("drop_hsi_edge", True):
                    _ys = np.nonzero(arrays["mask"])[0]
                    if _ys.size and int(_ys.max()) >= RGB_CROP_ROWS:
                        n_edge += 1
                        rep.update()
                        continue
                canvas, cm = make_cutout(rgb, arrays["mask"], p["aspect_w"], p["aspect_h"],
                                         p["canvas_scale"])
                if canvas is None:
                    n_skip += 1
                    rep.update()
                    continue
                cid = f"{row['stem']}_inst{int(row['instance_idx']):03d}"
                cpath = out_dir / f"{cid}_cutout.{p['format']}"
                pr = provenance_record("stage2",
                                       {"rgb": row["rgb_path"], "mask": row["mask_path"]},
                                       p, {"cutout_id": cid, "stem": row["stem"],
                                           "stage1_prompt": row.get("prompt", ""),
                                           "stage1_conf": row.get("confidence", "")})
                save_image_with_provenance(cpath, canvas, pr, fmt=p["format"])
                rows.append({"cutout_id": cid, "stem": row["stem"], "split": row.get("split", ""),
                             "session": sess_map.get(row["stem"], ""),
                             "instance_idx": int(row["instance_idx"]),
                             "stage1_prompt": row.get("prompt", ""),
                             "stage1_confidence": row.get("confidence", np.nan),
                             "whole_area_px": int(row["mask_area_px"]),
                             "rgb_path": row["rgb_path"], "whole_mask_path": row["mask_path"],
                             "cutout_path": str(cpath), **cm, "item_id": cid})
                prov.append(pr | {"output_path": str(cpath)})
            except Exception as e:
                rows.append({"cutout_id": f"{row['stem']}_inst{row['instance_idx']}",
                             "stem": row["stem"], "error": f"{type(e).__name__}: {e}"})
            rep.update()
        rep.close()

        df = pd.DataFrame(rows)
        out_csv = DIR_CUTOUTS / "cutouts.csv"
        df.to_csv(out_csv, index=False)
        write_provenance_csv("stage2", prov)
        STAGE2["cutouts"] = df
        ok = df[df.get("error", "").astype(str).str.len() == 0] if "error" in df.columns else df
        print(f"\ncutouts written : {len(ok):,}")
        print(f"skipped (empty) : {n_skip:,}")
        print(f"dropped at HSI edge (y >= {RGB_CROP_ROWS}) : {n_edge:,}")
        print(f"cutouts.csv     -> {out_csv}")
        log_params("stage2", outputs={"cutouts_csv": str(out_csv), "cutouts_dir": str(out_dir)},
                   results={"n_cutouts": int(len(ok)), "n_skipped": int(n_skip),
                            "n_dropped_hsi_edge": int(n_edge)})


w8_bar, w8_buttons = make_action_bar(render_preview8, lock8, batch8,
                                     batch_label="Run Stage 2 batch")
for _w in (w8_scale, w8_minarea, w8_fmt, w8_drop_edge):
    _w.observe(lambda ch: invalidate_lock("stage2", w8_buttons), names="value")

w8_status.value = status_html("stage2")
display(widgets.VBox([
    w8_aspect,
    widgets.HBox([w8_scale]),
    widgets.HBox([w8_minarea, w8_fmt, w8_nprev, w8_seed]),
    widgets.HBox([w8_drop_edge]),
    w8_bar,
    w8_status,
]), w8_out)

In [ ]:
# === Cell 10: Stage 3: Cap/Label Pseudo-GT ===
# SAM3 cap and label sub-segmentation on cutouts; body = whole - cap - label (cutout
# coordinates). Stores raw per-region diagnostics without filtering; all thresholds are
# applied in the Filter Tuning cell.
# The whole-object mask is replayed from the Stage 1 mask via the canvas geometry, not
# derived from cutout pixel colours (dark object regions would read as background).

STAGE3 = {"pseudo_gt": None, "sec_per_item": None}

w9_cap_prompt = widgets.Text(value="cap", description="Cap prompt:",
                             style={"description_width": "130px"},
                             layout=widgets.Layout(width="380px"))
w9_label_prompt = widgets.Text(value="label", description="Label prompt:",
                               style={"description_width": "130px"},
                               layout=widgets.Layout(width="380px"))
w9_cap_conf = widgets.FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05,
                                  description="Cap conf:", readout_format=".2f",
                                  style={"description_width": "130px"},
                                  layout=widgets.Layout(width="380px"))
w9_label_conf = widgets.FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05,
                                    description="Label conf:", readout_format=".2f",
                                    style={"description_width": "130px"},
                                    layout=widgets.Layout(width="380px"))
w9_imgsz = widgets.Dropdown(options=[644, 1008, 1288], value=644, description="imgsz:",
                            style={"description_width": "130px"},
                            layout=widgets.Layout(width="240px"))
w9_nprev = widgets.IntSlider(value=6, min=1, max=16, description="Preview N:",
                             style={"description_width": "130px"},
                             layout=widgets.Layout(width="360px"))
w9_seed = widgets.IntText(value=0, description="Seed:", style={"description_width": "130px"},
                          layout=widgets.Layout(width="200px"))
w9_status = widgets.HTML()
w9_out = widgets.Output()


def _cutouts_df():
    df = STAGE2.get("cutouts")
    if df is None:
        p = DIR_CUTOUTS / "cutouts.csv"
        if p.exists():
            df = pd.read_csv(p)
            STAGE2["cutouts"] = df
    if df is None:
        return pd.DataFrame()
    return df[df["cutout_path"].notna()] if "cutout_path" in df.columns else df


def replay_whole_mask(row):
    """Stage-1 mask -> cutout coordinates, via the Stage-2 canvas geometry."""
    arrays, _ = read_mask_npz(row["whole_mask_path"])
    src = arrays["mask"]
    ch, cw = int(row["canvas_h"]), int(row["canvas_w"])
    whole = np.zeros((ch, cw), dtype=bool)
    ys, xs = np.nonzero(src)
    cy = ys - int(row["mask_bbox_y1"]) + int(row["canvas_offset_y"])
    cx = xs - int(row["mask_bbox_x1"]) + int(row["canvas_offset_x"])
    ok = (cy >= 0) & (cy < ch) & (cx >= 0) & (cx < cw)
    whole[cy[ok], cx[ok]] = True
    return whole


def render_preview9():
    with w9_out:
        clear_output(wait=True)
        cuts = _cutouts_df()
        if cuts.empty:
            print("!! No cutouts - run Cell 9 batch first.")
            return
        sample = cuts.sample(n=min(w9_nprev.value, len(cuts)), random_state=int(w9_seed.value))

        n = len(sample)
        ncol = 3
        nrow = int(np.ceil(n / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(_FW * 0.62 * ncol, _FW * 0.42 * nrow),
                                 squeeze=False)
        axes = axes.reshape(-1)
        recs, times = [], []
        for ax, (_, row) in zip(axes, sample.iterrows()):
            t0 = time.time()
            try:
                whole = replay_whole_mask(row)
                wa = int(whole.sum())
                regions, meta = {}, {}
                conf_map = {"cap": w9_cap_conf.value, "label": w9_label_conf.value}
                for key, prompt in (("cap", w9_cap_prompt.value), ("label", w9_label_prompt.value)):
                    res = sam3_preview(row["cutout_path"], [prompt], conf_map[key], w9_imgsz.value)
                    m = result_masks_bool(res)
                    confs, _ = result_meta(res)
                    union = m.any(axis=0) if m.shape[0] else np.zeros_like(whole)
                    if union.shape != whole.shape:
                        im = Image.fromarray(union.astype(np.uint8) * 255)
                        union = np.asarray(im.resize((whole.shape[1], whole.shape[0]),
                                                     Image.NEAREST)) > 127
                    union &= whole
                    regions[key] = union
                    meta[f"{key}_n_det"] = int(m.shape[0])
                    meta[f"{key}_conf"] = float(max(confs)) if confs else np.nan
                    meta[f"{key}_frac"] = float(union.sum()) / max(wa, 1)
                body = whole & ~regions["cap"] & ~regions["label"]
                meta["body_frac"] = float(body.sum()) / max(wa, 1)
            except Exception as e:
                ax.text(0.5, 0.5, f"failed\n{e}", ha="center", fontsize=7)
                ax.axis("off")
                continue
            times.append(time.time() - t0)

            ax.imshow(np.array(Image.open(row["cutout_path"]).convert("RGB")))
            ax.contourf(body, levels=[0.5, 1], colors=[C_BODY], alpha=0.35)
            ax.contourf(regions["label"], levels=[0.5, 1], colors=[C_LABEL], alpha=0.35)
            ax.contourf(regions["cap"], levels=[0.5, 1], colors=[C_CAP], alpha=0.35)
            ax.set_title(f"{row['cutout_id']}\nlbl={meta['label_frac']:.2f}"
                         f"({meta['label_conf']:.2f}) cap={meta['cap_frac']:.2f}"
                         f"({meta['cap_conf']:.2f}) body={meta['body_frac']:.2f}", fontsize=7)
            ax.axis("off")
            recs.append(meta)
        for ax in axes[n:]:
            ax.axis("off")
        handles = [Line2D([0], [0], color=REGION_COLORS[c], lw=4, label=c.title())
                   for c in REGION_ORDER]
        _titled_legend(fig, f"Stage 3 Preview - cap='{w9_cap_prompt.value}' "
                            f"({w9_cap_conf.value:.2f})  label='{w9_label_prompt.value}' "
                            f"({w9_label_conf.value:.2f})", handles=handles, top_rect=0.88)
        plt.show()

        if recs:
            r = pd.DataFrame(recs)
            print(f"median label frac : {r['label_frac'].median():.3f}   "
                  f"median cap frac : {r['cap_frac'].median():.3f}   "
                  f"median body frac : {r['body_frac'].median():.3f}")
            print("(no filtering here - all thresholds are set and applied in Cell 11)")
        spf = float(np.median(times)) if times else None
        STAGE3["sec_per_item"] = spf
        plan = resolve_compute_plan()
        eff = spf / plan["n_workers"] if (spf and plan["mode"] == "multi") else spf
        est = estimate_batch_runtime(eff, len(cuts))
        w9_status.value = status_html("stage3", n_scope=len(cuts), est=est)
        print(f"\nmedian {spf:.2f} s/cutout -> projected batch over {len(cuts):,}: {est}"
              if spf else "")


def lock9():
    params = {"cap_prompt": w9_cap_prompt.value.strip(),
              "label_prompt": w9_label_prompt.value.strip(),
              "cap_conf": float(w9_cap_conf.value), "label_conf": float(w9_label_conf.value),
              "imgsz": int(w9_imgsz.value)}
    lock_stage("stage3", params, n_preview=int(w9_nprev.value))
    w9_status.value = status_html("stage3", n_scope=len(_cutouts_df()))


def batch9():
    p = require_locked("stage3")
    cuts = _cutouts_df()
    if cuts.empty:
        print("!! No cutouts.")
        return
    need = ["cutout_id", "cutout_path", "whole_mask_path", "mask_bbox_x1", "mask_bbox_y1",
            "canvas_offset_x", "canvas_offset_y", "canvas_w", "canvas_h", "stem"]
    items = [{**{k: r[k] for k in need}, "item_id": r["cutout_id"]} for _, r in cuts.iterrows()]

    with w9_out:
        clear_output(wait=True)
        print(f"Stage 3: {len(items):,} cutouts | cap='{p['cap_prompt']}' "
              f"({p['cap_conf']})  label='{p['label_prompt']}' ({p['label_conf']}) "
              f"imgsz={p['imgsz']}")
        df = run_sam3_batch("stage3", items, p, DIR_PSEUDOGT / "masks", key_col="item_id")
        if df.empty:
            print("!! No output produced.")
            return

        # Raw diagnostic columns are stored unfiltered.
        merged = cuts.merge(df, on="cutout_id", how="inner", suffixes=("", "_s3"))
        out_csv = DIR_PSEUDOGT / "pseudo_gt.csv"
        merged.to_csv(out_csv, index=False)
        STAGE3["pseudo_gt"] = merged

        n = len(merged)
        print(f"\nprocessed        : {n:,}")
        print(f"label detected   : {int((merged['label_frac'] > 0).sum()):,}")
        print(f"cap detected     : {int(merged['cap_present'].fillna(False).sum()):,}")
        errs = merged[merged.get("error", "").astype(str).str.len() > 0]
        print(f"errors           : {len(errs):,}")

        fig, axes = plt.subplots(1, 3, figsize=(_FW * 1.8, _FW * 0.42))
        for ax, col, c in zip(axes, ("label_frac", "cap_frac", "body_frac"),
                              (C_LABEL, C_CAP, C_BODY)):
            ax.hist(merged[col].dropna(), bins=40, color=c, alpha=0.85,
                    edgecolor="black", linewidth=0.3)
            ax.set_title(col.replace("_", " ").title())
            ax.set_xlabel("Area Fraction")
            ax.set_ylabel("Objects")
        _titled_legend(fig, "Stage 3 Region Area Fractions (raw, unfiltered - "
                            "thresholds are set in Cell 11)", top_rect=0.86)
        fig.savefig(DIR_FIGURES / "stage3_fractions.png", dpi=150, bbox_inches="tight")
        plt.show()

        log_params("stage3", outputs={"pseudo_gt": str(out_csv),
                                      "masks": str(DIR_PSEUDOGT / "masks")},
                   results={"n_processed": int(n),
                            "n_label_detected": int((merged["label_frac"] > 0).sum()),
                            "n_cap_detected": int(merged["cap_present"].fillna(False).sum()),
                            "n_errors": int(len(errs))})
        print(f"\npseudo_gt -> {out_csv}")


w9_bar, w9_buttons = make_action_bar(render_preview9, lock9, batch9,
                                     batch_label="Run Stage 3 batch")
for _w in (w9_cap_prompt, w9_label_prompt, w9_cap_conf, w9_label_conf, w9_imgsz):
    _w.observe(lambda ch: invalidate_lock("stage3", w9_buttons), names="value")

w9_status.value = status_html("stage3")
display(widgets.VBox([
    widgets.HBox([w9_cap_prompt, w9_label_prompt]),
    widgets.HBox([w9_cap_conf, w9_label_conf]),
    widgets.HBox([w9_imgsz, w9_nprev, w9_seed]),
    w9_bar,
    w9_status,
]), w9_out)

In [ ]:
# === Cell 11: Stage 4: Filter Tuning and YOLO Export ===
# Applies all thresholds to the stored Stage 3 columns and exports a single flat split
# at YOLO_INFER_W x YOLO_INFER_H. The set is for evaluation only (data.yaml: val only).

STAGE4 = {"filtered": None, "export": None}
try:
    import cv2
except ImportError:
    cv2 = None
w10_s1conf = widgets.FloatSlider(value=0.70, min=0.0, max=1.0, step=0.01,
                                 description="body(stage1) conf >=:", readout_format=".2f",
                                 style={"description_width": "150px"},
                                 layout=widgets.Layout(width="410px"))
w10_lconf = widgets.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.01,
                                description="label conf >=:", readout_format=".2f",
                                style={"description_width": "150px"},
                                layout=widgets.Layout(width="410px"))
w10_cconf = widgets.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.01,
                                description="cap conf >=:", readout_format=".2f",
                                style={"description_width": "150px"},
                                layout=widgets.Layout(width="410px"))
w10_bfrac = widgets.FloatSlider(value=0.0, min=0.0, max=0.9, step=0.01,
                                description="min body frac:", readout_format=".2f",
                                style={"description_width": "150px"},
                                layout=widgets.Layout(width="410px"))
w10_lfrac = widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.01,
                                description="min label frac:", readout_format=".2f",
                                style={"description_width": "150px"},
                                layout=widgets.Layout(width="410px"))
w10_cfrac = widgets.FloatSlider(value=0.0, min=0.0, max=0.9, step=0.01,
                                description="min cap frac:", readout_format=".2f",
                                style={"description_width": "150px"},
                                layout=widgets.Layout(width="410px"))
w10_lndet_min = widgets.IntText(value=1, description="min label_n_det:",
                                style={"description_width": "150px"},
                                layout=widgets.Layout(width="250px"))
w10_lndet_max = widgets.IntText(value=1, description="max label_n_det:",
                                style={"description_width": "150px"},
                                layout=widgets.Layout(width="250px"))
w10_cndet_min = widgets.IntText(value=1, description="min cap_n_det:",
                                style={"description_width": "150px"},
                                layout=widgets.Layout(width="250px"))
w10_cndet_max = widgets.IntText(value=1, description="max cap_n_det:",
                                style={"description_width": "150px"},
                                layout=widgets.Layout(width="250px"))
w10_gallery = widgets.IntSlider(value=12, min=0, max=48, description="Gallery N:",
                                style={"description_width": "140px"},
                                layout=widgets.Layout(width="380px"))
w10_eps = widgets.FloatSlider(value=0.002, min=0.0005, max=0.01, step=0.0005,
                              description="Polygon epsilon:", readout_format=".4f",
                              style={"description_width": "140px"},
                              layout=widgets.Layout(width="400px"))
w10_minpoly = widgets.IntText(value=24, description="Min poly px2:",
                              style={"description_width": "140px"},
                              layout=widgets.Layout(width="240px"))
w10_size = widgets.HTML(
    f"<div style='font-family:monospace;font-size:12px;padding:6px'>"
    f"export size fixed at <b>{YOLO_INFER_W}x{YOLO_INFER_H}</b> (w x h), single flat split"
    f"</div>")
w10_status = widgets.HTML()
w10_out = widgets.Output()
def _pseudo_gt_df():
    df = STAGE3.get("pseudo_gt")
    if df is None:
        p = DIR_PSEUDOGT / "pseudo_gt.csv"
        if p.exists():
            df = pd.read_csv(p)
            STAGE3["pseudo_gt"] = df
    return df if df is not None else pd.DataFrame()
def _filtered():
    """Locked, filtered object set. Read from disk if not in memory."""
    df = STAGE4.get("filtered")
    if df is None:
        p = DIR_TABLES / "filtered_objects.csv"
        if p.exists():
            df = pd.read_csv(p)
            STAGE4["filtered"] = df
    return df if df is not None else pd.DataFrame()
def apply_filter(df, p):
    """Apply ten thresholds: confidence floor per region (body from Stage 1, label/cap
    from Stage 3), area-fraction floor per region, detection-count bounds for label and
    cap. Missing values fail a minimum (fillna 0) and pass a maximum (fillna 99)."""
    m = (
        (df["stage1_confidence"].fillna(0) >= p["stage1_conf_min"])
        & (df["label_conf"].fillna(0) >= p["label_conf_min"])
        & (df["cap_conf"].fillna(0) >= p["cap_conf_min"])
        & (df["body_frac"].fillna(0) >= p["min_body_area_frac"])
        & (df["label_frac"].fillna(0) >= p["min_label_area_frac"])
        & (df["cap_frac"].fillna(0) >= p["min_cap_area_frac"])
        & (df["label_n_det"].fillna(0) >= p["label_n_det_min"])
        & (df["label_n_det"].fillna(99) <= p["label_n_det_max"])
        & (df["cap_n_det"].fillna(0) >= p["cap_n_det_min"])
        & (df["cap_n_det"].fillna(99) <= p["cap_n_det_max"])
    )
    return df[m]
def _params10():
    return {"stage1_conf_min": float(w10_s1conf.value),
            "label_conf_min": float(w10_lconf.value),
            "cap_conf_min": float(w10_cconf.value),
            "min_body_area_frac": float(w10_bfrac.value),
            "min_label_area_frac": float(w10_lfrac.value),
            "min_cap_area_frac": float(w10_cfrac.value),
            "label_n_det_min": int(w10_lndet_min.value),
            "label_n_det_max": int(w10_lndet_max.value),
            "cap_n_det_min": int(w10_cndet_min.value),
            "cap_n_det_max": int(w10_cndet_max.value),
            "export_w": int(YOLO_INFER_W), "export_h": int(YOLO_INFER_H),
            "polygon_epsilon_frac": float(w10_eps.value),
            "min_polygon_area_px2": int(w10_minpoly.value)}
def mask_to_polygons(mask, min_area_px=None, epsilon_frac=None):
    """Boolean mask -> list of pixel-space polygons.

    Filters on contour area, not vertex count (CHAIN_APPROX_SIMPLE gives 4 points for a
    rectangle).
    """
    min_area_px = int(w10_minpoly.value) if min_area_px is None else min_area_px
    epsilon_frac = float(w10_eps.value) if epsilon_frac is None else epsilon_frac
    m = (np.asarray(mask).astype(np.uint8)) * 255
    if not m.any():
        return []
    if cv2 is not None:
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        out = []
        for c in cnts:
            if len(c) < 3 or cv2.contourArea(c) < min_area_px:
                continue
            eps = epsilon_frac * cv2.arcLength(c, True)
            simplified = cv2.approxPolyDP(c, eps, True)
            if len(simplified) < 3:            # over-simplified: keep the original
                simplified = c
            out.append(simplified.reshape(-1, 2).astype(np.float64))
        return out
    # Fallback without cv2: per-component bounding box polygon
    lbl, n = ndimage.label(mask)
    out = []
    for i in range(1, n + 1):
        ys, xs = np.nonzero(lbl == i)
        if xs.size < min_area_px:
            continue
        out.append(np.array([[xs.min(), ys.min()], [xs.max(), ys.min()],
                             [xs.max(), ys.max()], [xs.min(), ys.max()]], dtype=np.float64))
    return out
def resize_bool(mask, w, h):
    im = Image.fromarray(np.asarray(mask).astype(np.uint8) * 255)
    return np.asarray(im.resize((w, h), Image.NEAREST)) > 127
def write_yolo_label(path, regions, img_w, img_h):
    """Write YOLO segmentation lines for already-resized region masks.

    A region split into disconnected pieces writes several lines of the same class.
    """
    lines = []
    for name in REGION_ORDER:
        mask = regions.get(name)
        if mask is None or not mask.any():
            continue
        cid = CLASS_MAP[name]
        for poly in mask_to_polygons(mask):
            norm = poly.copy()
            norm[:, 0] /= img_w
            norm[:, 1] /= img_h
            norm = np.clip(norm, 0.0, 1.0)
            coords = " ".join(f"{v:.6f}" for v in norm.reshape(-1))
            lines.append(f"{cid} {coords}")
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text("\n".join(lines) + ("\n" if lines else ""))
    return len(lines)
def render_preview10():
    with w10_out:
        clear_output(wait=True)
        df = _pseudo_gt_df()
        if df.empty:
            print("!! No pseudo-GT - run Cell 10 batch first.")
            return
        p = _params10()
        kept = apply_filter(df, p)
        print(f"PASS: {len(kept):,} / {len(df):,}  ({100 * len(kept) / max(len(df), 1):.1f}%)")
        if len(kept):
            print(f"  cap present among kept : {100 * kept['cap_present'].fillna(False).mean():.1f}%")
            print(f"  median stage1 conf     : {kept['stage1_confidence'].median():.3f}")
            print(f"  median label frac      : {kept['label_frac'].median():.3f}")
            print(f"  median body frac       : {kept['body_frac'].median():.3f}")
            if "session" in kept.columns:
                display(HTML(kept.groupby("session").size().reset_index(name="kept")
                             .to_html(index=False)))
        n_nolabel = int((df["label_frac"].fillna(0) == 0).sum())
        print(f"  label_frac == 0 in pool: {n_nolabel:,} "
              f"({100 * n_nolabel / max(len(df), 1):.1f}%)")
        # Sensitivity: sweep each threshold independently
        sweeps = [
            ("stage1_conf_min", np.arange(0.0, 1.0, 0.05)),
            ("label_conf_min", np.arange(0.0, 1.0, 0.05)),
            ("cap_conf_min", np.arange(0.0, 1.0, 0.05)),
            ("min_body_area_frac", np.arange(0.0, 0.9, 0.05)),
            ("min_label_area_frac", np.arange(0.0, 0.5, 0.025)),
            ("min_cap_area_frac", np.arange(0.0, 0.5, 0.025)),
            ("label_n_det_max", np.arange(1, 9)),
            ("cap_n_det_max", np.arange(1, 9)),
        ]
        nrow_sw = 2
        ncol_sw = int(np.ceil(len(sweeps) / nrow_sw))
        fig, axes = plt.subplots(nrow_sw, ncol_sw, figsize=(_FW * 2.0, _FW * 0.75), squeeze=False)
        axes = axes.reshape(-1)
        for ax, (key, vals) in zip(axes, sweeps):
            ys = []
            for v in vals:
                q = dict(p)
                q[key] = v
                ys.append(len(apply_filter(df, q)))
            ax.plot(vals, ys, color="#2a9d8f", lw=1.5)
            ax.axvline(p[key], color="#d1495b", ls="--", lw=1)
            ax.set_title(key.replace("_", " "), fontsize=9)
            ax.set_xlabel("threshold")
            ax.set_ylabel("Objects Passing")
        for ax in axes[len(sweeps):]:
            ax.axis("off")
        _titled_legend(fig, "Filter Sensitivity - Each Threshold Swept Independently",
                       top_rect=0.90)
        fig.savefig(DIR_FIGURES / "filter_sensitivity.png", dpi=150, bbox_inches="tight")
        plt.show()
        if len(kept) and w10_gallery.value:
            n = min(int(w10_gallery.value), len(kept))
            sample = kept.sample(n=n, random_state=0)
            ncol = 4
            nrow = int(np.ceil(n / ncol))
            fig, axes = plt.subplots(nrow, ncol, figsize=(_FW * 0.48 * ncol, _FW * 0.34 * nrow),
                                     squeeze=False)
            axes = axes.reshape(-1)
            for ax, (_, row) in zip(axes, sample.iterrows()):
                # Separate try blocks: a mask read failure still shows the image.
                try:
                    ax.imshow(np.array(Image.open(row["cutout_path"]).convert("RGB")))
                except Exception:
                    ax.text(0.5, 0.5, f"{row.get('cutout_id', '?')}\nimage failed", ha="center",
                            va="center", fontsize=6, color="#d1495b")
                    ax.axis("off")
                    continue

                mp = row.get("regions_mask_path")
                if pd.isna(mp) or not mp:
                    ax.set_title(f"{row.get('cutout_id', '?')}\nno regions mask recorded",
                                 fontsize=6, color="#d1495b")
                    ax.axis("off")
                    continue

                try:
                    arrays, _ = read_mask_npz(mp)
                    # Body first so smaller label/cap polygons draw on top
                    for key in ("body", "label", "cap"):
                        m = arrays.get(key)
                        if m is None or not m.any():
                            continue
                        for poly in mask_to_polygons(m):
                            ax.add_patch(MplPolygon(poly, closed=True,
                                                    facecolor=REGION_COLORS[key], alpha=0.40,
                                                    edgecolor=REGION_COLORS[key], linewidth=1.0))
                    ax.set_title(f"{str(row['cutout_id'])[:26]}\n"
                                f"body {row.get('body_frac', float('nan')):.2f}  "
                                f"label {row.get('label_frac', float('nan')):.2f}"
                                f"({row.get('label_conf', float('nan')):.2f})  "
                                f"cap {row.get('cap_frac', float('nan')):.2f}", fontsize=6)
                except Exception as e:
                    ax.set_title(f"{row.get('cutout_id', '?')}\nmask read failed: "
                                 f"{type(e).__name__}", fontsize=6, color="#d1495b")
                ax.axis("off")
            for ax in axes[n:]:
                ax.axis("off")
            handles = [Line2D([0], [0], color=REGION_COLORS[c], lw=4, label=c.title())
                       for c in ("body", "label", "cap")]
            _titled_legend(fig, "Filtered Objects - Polygon Masks", handles=handles,
                           top_rect=0.90)
            plt.show()
        STAGE4["filtered"] = kept
        w10_status.value = status_html("filter", n_scope=len(kept))
def lock10():
    p = _params10()
    df = _pseudo_gt_df()
    kept = apply_filter(df, p)
    STAGE4["filtered"] = kept
    out_csv = DIR_TABLES / "filtered_objects.csv"
    kept.to_csv(out_csv, index=False)
    lock_stage("filter", p, n_preview=int(w10_gallery.value))
    log_params("filter", outputs={"filtered_objects": str(out_csv)},
               results={"n_pool": int(len(df)), "n_kept": int(len(kept))})
    print(f"filtered_objects -> {out_csv}  ({len(kept):,} objects)")
    w10_status.value = status_html("filter", n_scope=len(kept))
def batch10():
    """Export the pseudo-GT as a YOLO segmentation dataset, single flat split."""
    p = require_locked("filter")
    kept = STAGE4.get("filtered")
    if kept is None or kept.empty:
        kept = _filtered()
    if kept is None or kept.empty:
        print("!! Nothing to export.")
        return
    with w10_out:
        clear_output(wait=True)
        img_dir = DIR_YOLO_DS / "images"
        lbl_dir = DIR_YOLO_DS / "labels"
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        print(f"Exporting {len(kept):,} objects at {YOLO_INFER_W}x{YOLO_INFER_H}")
        rep = ProgressReporter("yolo_export", len(kept))
        rows, prov, n_lines = [], [], 0
        for _, row in kept.iterrows():
            rec = {"cutout_id": row["cutout_id"], "stem": row.get("stem", ""),
                   "source": "spectralwaste", "session": row.get("session", ""),
                   "split": row.get("split", ""),
                   "stage1_confidence": row.get("stage1_confidence", np.nan), "error": ""}
            try:
                arrays, _ = read_mask_npz(row["regions_mask_path"])
                src = Image.open(row["cutout_path"]).convert("RGB")
                img_r = src.resize((YOLO_INFER_W, YOLO_INFER_H), Image.BILINEAR)
                regions = {k: resize_bool(arrays[k], YOLO_INFER_W, YOLO_INFER_H)
                           for k in REGION_ORDER if k in arrays}
                img_dst = img_dir / f"{row['cutout_id']}.png"
                lbl_dst = lbl_dir / f"{row['cutout_id']}.txt"
                img_r.save(img_dst, format="PNG")
                nl = write_yolo_label(lbl_dst, regions, YOLO_INFER_W, YOLO_INFER_H)
                if nl == 0:
                    rec["error"] = "no polygons after simplification"
                n_lines += nl
                rec.update({"image_path": str(img_dst), "label_path": str(lbl_dst),
                            "n_lines": nl,
                            **{f"n_{k}_polys": len(mask_to_polygons(regions[k]))
                               for k in regions}})
                prov.append(provenance_record(
                    "yolo_export",
                    {"cutout": row["cutout_path"], "regions": row["regions_mask_path"],
                     "rgb": row.get("rgb_path", "")}, p,
                    {"cutout_id": row["cutout_id"], "session": row.get("session", ""),
                     "output_image": str(img_dst), "output_label": str(lbl_dst),
                     "pseudo_gt": "body/cap/label from SAM3 prompts"}))
            except Exception as e:
                rec["error"] = f"{type(e).__name__}: {e}"
                print(f"!! {row['cutout_id']}: {e}")
            rows.append(rec)
            rep.update()
        rep.close()

        exp = pd.DataFrame(rows)
        out_csv = DIR_YOLO_DS / "export_index.csv"
        exp.to_csv(out_csv, index=False)
        ok = exp[exp["error"] == ""]
        STAGE4["export"] = ok

        yaml_txt = (f"# SpectralWaste adapted to the KI-Sort body/cap/label convention\n"
                    f"# run_id: {RUN_ID}   config_hash: {CONFIG_HASH}\n"
                    f"# Evaluation set - single split, no train/val partition.\n"
                    f"path: {DIR_YOLO_DS}\nval: images\n\n"
                    f"nc: {len(CLASS_MAP)}\n"
                    f"names:\n" + "".join(f"  {v}: {k}\n"
                                          for k, v in sorted(CLASS_MAP.items(),
                                                             key=lambda kv: kv[1])))
        (DIR_YOLO_DS / "data.yaml").write_text(yaml_txt)

        (DIR_YOLO_DS / "DATASET_CARD.md").write_text("\n".join([
            "# SpectralWaste - KI-Sort Adapted Segmentation Set", "",
            f"- run_id: `{RUN_ID}`", f"- config hash: `{CONFIG_HASH}`",
            f"- source dataset: `{INPUT_ROOT}`",
            f"- objects exported: {len(ok):,}",
            f"- image size: {YOLO_INFER_W} x {YOLO_INFER_H} (w x h), deployment "
            f"inference shape",
            "- single flat split; evaluation only",
            "- capture session per object is recorded in `export_index.csv` as `session`",
            "", "## Label provenance", "",
            "| element | provenance |", "|---|---|",
            "| whole-object extent | SAM3 concept prompt (Stage 1), not human annotation |",
            "| body / cap / label split | SAM3 concept prompts (Stage 3); "
            "body = whole - cap - label |",
            "| polymer / material class | absent; SpectralWaste ships no polymer labels |",
            "", "## Filter thresholds", "",
            "```json", json.dumps(p, indent=2, default=str), "```", "",
        ]))
        write_provenance_csv("yolo_export", prov)

        print(f"\nYOLO dataset -> {DIR_YOLO_DS}")
        print(f"  exported {len(ok):,}/{len(exp):,} objects | {n_lines:,} polygon rows")
        if len(ok) and "session" in ok.columns:
            display(HTML(ok.groupby("session")
                         .agg(n=("cutout_id", "size"), mean_lines=("n_lines", "mean"))
                         .reset_index().to_html(index=False)))
        log_params("yolo_export",
                   outputs={"dataset": str(DIR_YOLO_DS),
                            "data_yaml": str(DIR_YOLO_DS / "data.yaml"),
                            "export_index": str(out_csv)},
                   results={"n_exported": int(len(ok)), "n_in": int(len(exp)),
                            "n_polygons": n_lines, "split": "single (evaluation only)"})
        w10_status.value = status_html("filter", n_scope=len(kept))
w10_bar, w10_buttons = make_action_bar(render_preview10, lock10, batch10,
                                       batch_label="Export YOLO dataset")
for _w in (w10_s1conf, w10_lconf, w10_cconf, w10_bfrac, w10_lfrac, w10_cfrac,
          w10_lndet_min, w10_lndet_max, w10_cndet_min, w10_cndet_max,
          w10_eps, w10_minpoly):
    _w.observe(lambda ch: invalidate_lock("filter", w10_buttons), names="value")
w10_status.value = status_html("filter")
display(widgets.VBox([
    w10_size,
    widgets.HBox([w10_s1conf, w10_lconf, w10_cconf]),
    widgets.HBox([w10_bfrac, w10_lfrac, w10_cfrac]),
    widgets.HBox([w10_lndet_min, w10_lndet_max]),
    widgets.HBox([w10_cndet_min, w10_cndet_max]),
    widgets.HBox([w10_eps, w10_minpoly]),
    widgets.HBox([w10_gallery]),
    w10_bar,
    w10_status,
]), w10_out)

In [ ]:
# === Cell 12: YOLO Visual QA Preview ===
# Runs the deployment segmentation model on sampled exported images and shows prediction
# beside pseudo-GT. No metrics; scoring is done in the benchmark notebook.

QA = {"model": None}


def load_yolo():
    if QA["model"] is None:
        from ultralytics import YOLO
        if not SEG_MODEL_DEPLOY.exists():
            raise FileNotFoundError(f"Deployment weights not found: {SEG_MODEL_DEPLOY}")
        m = YOLO(str(SEG_MODEL_DEPLOY))
        for name, cid in CLASS_MAP.items():
            got = m.names.get(cid)
            if got != name:
                print(f"!! CLASS_MAP mismatch at id {cid}: config says '{name}', "
                      f"model says '{got}'.")
        QA["model"] = m
    return QA["model"]


def _export_df():
    df = STAGE4.get("export")
    if df is None:
        p = DIR_YOLO_DS / "export_index.csv"
        if p.exists():
            d = pd.read_csv(p)
            d["error"] = d["error"].fillna("")
            df = d[d["error"] == ""]
            STAGE4["export"] = df
    return df if df is not None else pd.DataFrame()


def parse_label_file(path, w, h):
    """YOLO polygon label file -> {region: [pixel-space polygons]}."""
    out = {k: [] for k in CLASS_MAP}
    for line in Path(path).read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        cid = int(float(parts[0]))
        name = CLASS_NAMES.get(cid)
        if name is None:
            continue
        pts = np.array([float(v) for v in parts[1:]], dtype=np.float64).reshape(-1, 2)
        pts[:, 0] *= w
        pts[:, 1] *= h
        out[name].append(pts)
    return out


def _draw_polys(ax, polys_by_region, lw=1.3, alpha=0.25):
    for region, polys in polys_by_region.items():
        for p in polys:
            if len(p) < 3:
                continue
            closed = np.vstack([p, p[:1]])
            ax.fill(closed[:, 0], closed[:, 1], color=REGION_COLORS[region], alpha=alpha)
            ax.plot(closed[:, 0], closed[:, 1], color=REGION_COLORS[region], lw=lw)


w11_conf = widgets.FloatSlider(value=0.25, min=0.01, max=0.95, step=0.01,
                               description="YOLO conf:", readout_format=".2f",
                               style={"description_width": "120px"},
                               layout=widgets.Layout(width="360px"))
w11_iou = widgets.FloatSlider(value=0.50, min=0.1, max=0.9, step=0.05,
                              description="YOLO iou:", readout_format=".2f",
                              style={"description_width": "120px"},
                              layout=widgets.Layout(width="360px"))
w11_session = widgets.Dropdown(options=["all"], value="all", description="Session:",
                               style={"description_width": "120px"},
                               layout=widgets.Layout(width="300px"))
w11_nprev = widgets.IntSlider(value=4, min=1, max=10, description="Preview N:",
                              style={"description_width": "120px"},
                              layout=widgets.Layout(width="360px"))
w11_seed = widgets.IntText(value=0, description="Seed:", style={"description_width": "120px"},
                           layout=widgets.Layout(width="200px"))
w11_btn = widgets.Button(description="Run visual QA", button_style="info", icon="eye")
w11_refresh = widgets.Button(description="Refresh sessions", icon="refresh")
w11_out = widgets.Output()


def _refresh_sessions(_=None):
    exp = _export_df()
    opts = ["all"]
    if len(exp) and "session" in exp.columns:
        opts += sorted(str(s) for s in exp["session"].dropna().unique() if str(s))
    w11_session.options = opts
    if w11_session.value not in opts:
        w11_session.value = "all"


w11_refresh.on_click(_refresh_sessions)


def run_qa(_):
    with w11_out:
        clear_output(wait=True)
        exp = _export_df()
        if exp.empty:
            print("!! Nothing exported - run Cell 11 export first.")
            return
        if w11_session.value != "all" and "session" in exp.columns:
            exp = exp[exp["session"].astype(str) == w11_session.value]
        if exp.empty:
            print("!! No exported objects for that session.")
            return
        model = load_yolo()
        sample = exp.sample(n=min(w11_nprev.value, len(exp)), random_state=int(w11_seed.value))
        n = len(sample)
        fig, axes = plt.subplots(n, 2, figsize=(_FW * 1.9, _FW * 0.42 * n), squeeze=False)
        counts = []
        for i, (_, r) in enumerate(sample.iterrows()):
            rgb = np.array(Image.open(r["image_path"]).convert("RGB"))
            h, w = rgb.shape[:2]

            gt = parse_label_file(r["label_path"], w, h)
            axes[i][0].imshow(rgb)
            _draw_polys(axes[i][0], gt)
            axes[i][0].set_title(f"$\\bf{{(a)}}$ pseudo-GT  {str(r['cutout_id'])[:28]}")
            axes[i][0].axis("off")

            res = model.predict(source=r["image_path"], conf=float(w11_conf.value),
                                iou=float(w11_iou.value), imgsz=YOLO_INFER_W,
                                rect=True, verbose=False)[0]
            pred = {k: [] for k in CLASS_MAP}
            if res.masks is not None and len(res.masks):
                cls = res.boxes.cls.cpu().numpy().astype(int)
                for poly, cid in zip(res.masks.xy, cls):
                    name = CLASS_NAMES.get(int(cid))
                    if name is not None and len(poly) >= 3:
                        pred[name].append(np.asarray(poly, dtype=np.float64))
            axes[i][1].imshow(rgb)
            _draw_polys(axes[i][1], pred)
            axes[i][1].set_title("$\\bf{(b)}$ prediction  "
                                 + "  ".join(f"{k}:{len(v)}" for k, v in pred.items()))
            axes[i][1].axis("off")
            counts.append({"item": str(r["cutout_id"])[:34],
                           **{f"gt_{k}": len(v) for k, v in gt.items()},
                           **{f"pred_{k}": len(v) for k, v in pred.items()}})
        handles = [Line2D([0], [0], color=REGION_COLORS[c], lw=6, label=c.title())
                   for c in REGION_ORDER]
        _titled_legend(fig, f"Visual QA - Deployment Model on Exported Crops  "
                            f"conf={w11_conf.value:.2f}", handles=handles, ncol=3,
                       top_rect=0.90)
        plt.show()
        display(HTML(pd.DataFrame(counts).to_html(index=False)))
        print("\nRegion counts only. No IoU, mAP or aggregate score is computed here.")


w11_btn.on_click(run_qa)
_refresh_sessions()
display(widgets.VBox([
    widgets.HBox([w11_conf, w11_iou]),
    widgets.HBox([w11_session, w11_refresh]),
    widgets.HBox([w11_nprev, w11_seed]),
    w11_btn,
]), w11_out)

In [ ]:
# === Cell 13: True-GT Spectra Extraction and Export ===
# Per-region HSI spectra from human-verified true-GT labels (not Stage 3 pseudo-GT).
# Chain: true-GT polygon -> cutout px -> raw RGB px -> raw HSI grid px -> cube index.
#   yolo-seg -> cutout :  denormalised against the object's own canvas_w x canvas_h
#                         (yolo_dataset/images/ is a resized copy of the cutout canvas)
#   cutout   -> rgb    :  x_rgb = x_cut - canvas_offset_x + mask_bbox_x1
#   rgb      -> hsi    :  RgbToHsi (authors' scale, constant DX, per-session DY)
#
# YOLO-seg polygons cannot encode holes, so the body is the full outer silhouette and
# region priority is applied here: body_valid = body AND NOT label AND NOT cap.
# Masks are eroded by `erode_px` before projection; regions with >= 1 HSI px are kept.
#
# Spectra are extracted as raw DN; preprocessing is applied at export in this order:
#     cut  : wavelength-range restriction
#     IARR : DN(b) / mean_DN(b), dataset-wide mean over all true-GT-pool frames
#     SNV  : per-spectrum standardisation over the retained band range (ddof=0)

# --- Source run ---
TRUE_GT_RUN_ID = "20260820_115620_raw_labeled_v1"
ALIGN_RUN_ID = "20260819_065958_raw_labeled_HSI_alignment_table"

SRC_DATA_ROOT = OUT_ROOT / "data" / TRUE_GT_RUN_ID
SRC_RESULTS_ROOT = OUT_ROOT / "results" / TRUE_GT_RUN_ID
SRC_YOLO_DS = SRC_DATA_ROOT / "yolo_dataset"
TRUE_GT_LABELS_DIR = SRC_YOLO_DS / "labels_true_gt"
TRUE_GT_INDEX_CSV = SRC_YOLO_DS / "export_index.csv"

# --- Export target ---
SPECTRA_EXPORT_PARENT = OUT_ROOT / "data"
SPECTRA_EXPORT_PREFIX = "preprocessed_spectra"
IARR_REF_CACHE = SPECTRA_EXPORT_PARENT / f"true_gt_iarr_reference_{TRUE_GT_RUN_ID}.csv"

# Spectrum files: two columns, semicolon-separated, no header, no extension
# (same format as the IR preprocessing notebook).
SPECTRUM_FILE_EXT = ""
SPECTRUM_SEP = ";"
SPECTRUM_FLOAT_FMT = "%.10f"

GEOM_COLS = ["cutout_id", "stem", "session", "split", "canvas_offset_x", "canvas_offset_y",
             "mask_bbox_x1", "mask_bbox_y1", "canvas_w", "canvas_h", "cutout_path"]

TRUEGT = {"pool": None, "geometry": None, "iarr_reference": None, "last_export": None,
          "frame_index": None}

import traceback

try:
    import cv2 as _cv2
except ImportError:
    _cv2 = None


# --- NaN-safe coercion ---
# Values used in Path(), open() or string comparisons pass through these helpers, since
# pandas may keep missing values as NaN after .astype(str) and Path(nan) raises.

def _is_missing(v):
    try:
        return v is None or bool(pd.isna(v))
    except (TypeError, ValueError):
        return False


def _as_text(v):
    """Missing -> empty string, everything else -> str."""
    if _is_missing(v):
        return ""
    s = str(v)
    return "" if s in ("nan", "None", "<NA>", "NaT") else s


def _safe_stem(v):
    s = _as_text(v)
    if not s:
        return ""
    try:
        return Path(s).stem
    except Exception:
        return ""


# --- Source loading ---

def _frame_index():
    """The true-GT source run's frame_index.csv, restricted to frames with an HSI cube.

    Uses the source run's table rather than the live FRAME_INDEX (which belongs to the
    current RUN_ID); FRAME_INDEX is only a fallback when the file is missing.
    """
    if TRUEGT.get("frame_index") is not None:
        return TRUEGT["frame_index"]
    p = SRC_RESULTS_ROOT / "tables" / "frame_index.csv"
    if not p.exists():
        fi = globals().get("FRAME_INDEX")
        if fi is None:
            raise FileNotFoundError(
                f"No frame_index.csv for the true-GT source run at {p}, and no "
                f"FRAME_INDEX in memory. Check TRUE_GT_RUN_ID, or run Cell 5.")
    else:
        fi = pd.read_csv(p)
    fi = fi.copy()
    if "hyper_path" not in fi.columns:
        raise KeyError(f"frame index has no hyper_path column: {list(fi.columns)}")
    fi["hyper_path"] = fi["hyper_path"].map(_as_text)
    fi = fi[fi["hyper_path"] != ""].drop_duplicates("stem").reset_index(drop=True)
    if fi.empty:
        raise RuntimeError(f"No frames with a usable hyper_path in {p}")
    TRUEGT["frame_index"] = fi
    return fi


def _frame_row(stem):
    fi = _frame_index()
    hit = fi[fi["stem"].map(_as_text) == _as_text(stem)]
    if not len(hit):
        raise KeyError(f"frame {stem!r} has no row with a usable hyper_path in the "
                       f"{TRUE_GT_RUN_ID} frame index")
    return hit.iloc[0]


def ensure_alignment():
    """Load the measured session offsets from ALIGN_RUN_ID without re-running the alignment."""
    if ALIGN.get("offsets") is not None:
        return ALIGN["offsets"]
    try:
        return load_alignment(ALIGN_RUN_ID)
    except FileNotFoundError:
        root = RESULTS_OUT_ROOT.parent / ALIGN_RUN_ID
        hits = sorted(root.rglob("session_alignment_offsets.csv"))
        if not hits:
            raise FileNotFoundError(f"No session_alignment_offsets.csv under {root}")
        off = pd.read_csv(hits[0])
        dx = int(off["dx_applied"].iloc[0]) if "dx_applied" in off.columns else 0
        ALIGN.update({"offsets": off, "dx": dx, "source_csv": str(hits[0]),
                      "method": f"loaded:{ALIGN_RUN_ID}"})
        print(f"loaded {len(off):,} session offsets from {hits[0]}")
        return off


def load_geometry():
    """cutout_id -> canvas geometry and frame identity, from the source run."""
    if TRUEGT["geometry"] is not None:
        return TRUEGT["geometry"]
    for p in (SRC_RESULTS_ROOT / "tables" / "filtered_objects.csv",
              SRC_DATA_ROOT / "stage2_cutouts" / "cutouts.csv"):
        if not p.exists():
            continue
        df = pd.read_csv(p)
        have = [c for c in GEOM_COLS if c in df.columns]
        if not {"cutout_id", "stem", "canvas_w", "canvas_h",
                "canvas_offset_x", "mask_bbox_x1"}.issubset(set(have)):
            continue
        df = df[have].drop_duplicates("cutout_id").reset_index(drop=True)
        for c in ("session", "split"):
            if c not in df.columns:
                df[c] = ""
        df["geometry_source"] = str(p)
        TRUEGT["geometry"] = df
        return df
    raise FileNotFoundError(
        f"No geometry table found. Looked for filtered_objects.csv under "
        f"{SRC_RESULTS_ROOT / 'tables'} and cutouts.csv under "
        f"{SRC_DATA_ROOT / 'stage2_cutouts'}.")


def _detect_index_key(idx, label_stems):
    """Column in export_index.csv whose values match labels_true_gt/<stem>.txt names."""
    best, best_hits = None, -1
    for c in idx.columns:
        vals = idx[c].map(_as_text)
        stems = idx[c].map(_safe_stem)
        hits = int(vals.isin(label_stems).sum()
                   + stems[stems != ""].isin(label_stems).sum())
        if hits > best_hits:
            best, best_hits = c, hits
    return best, best_hits


def load_true_gt_pool():
    """Accepted true-GT objects joined to their canvas geometry."""
    if TRUEGT["pool"] is not None:
        return TRUEGT["pool"]
    if not TRUE_GT_LABELS_DIR.is_dir():
        raise FileNotFoundError(f"No labels_true_gt at {TRUE_GT_LABELS_DIR}")
    label_paths = {p.stem: p for p in sorted(TRUE_GT_LABELS_DIR.glob("*.txt"))}
    if not label_paths:
        raise FileNotFoundError(f"labels_true_gt is empty: {TRUE_GT_LABELS_DIR}")

    geom = load_geometry()
    if TRUE_GT_INDEX_CSV.exists():
        idx = pd.read_csv(TRUE_GT_INDEX_CSV)
        keycol, hits = _detect_index_key(idx, set(label_paths))
        if "true_gt_keep" in idx.columns:
            keep_mask = idx["true_gt_keep"].astype(str).str.lower().isin({"true", "1"})
            idx = idx[keep_mask]
        idx = idx.copy()
        idx["_stem"] = idx[keycol].map(
            lambda v: _as_text(v) if _as_text(v) in label_paths else _safe_stem(v))
        accepted = [s for s in idx["_stem"] if s in label_paths]
        print(f"export_index join key : {keycol}  ({hits:,} stem matches)")
    else:
        accepted = list(label_paths)
        print(f"!! No export_index.csv at {TRUE_GT_INDEX_CSV} - taking every "
              f"labels_true_gt/*.txt on disk.")

    pool = geom[geom["cutout_id"].map(_as_text).isin(set(accepted))].copy()
    pool["true_gt_label_abs"] = pool["cutout_id"].map(
        lambda s: str(label_paths[_as_text(s)]))
    missing = sorted(set(accepted) - set(pool["cutout_id"].map(_as_text)))
    if missing:
        print(f"!! {len(missing):,} accepted objects have no geometry row and were "
              f"dropped, e.g. {missing[:3]}")
    TRUEGT["pool"] = pool.reset_index(drop=True)
    return TRUEGT["pool"]


# --- Mask construction ---

def parse_yolo_seg(txt_path, width, height):
    """YOLO-seg polygon file -> per-region boolean masks at (height, width).

    Coordinates are denormalised against the caller-supplied size (the object's native
    cutout canvas, not the 640x384 export size).
    """
    if _cv2 is None:
        raise RuntimeError("cv2 is required to rasterise true-GT polygons "
                           "(use the yolo-train env, or install opencv).")
    acc = {k: np.zeros((int(height), int(width)), dtype=np.uint8) for k in REGION_ORDER}
    text = Path(txt_path).read_text().strip()
    for line in text.splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        name = CLASS_NAMES.get(int(float(parts[0])))
        if name not in acc:
            continue
        pts = np.asarray([float(v) for v in parts[1:]], dtype=np.float64)
        if pts.size % 2:
            pts = pts[:-1]
        pts = pts.reshape(-1, 2)
        pts[:, 0] *= float(width)
        pts[:, 1] *= float(height)
        _cv2.fillPoly(acc[name], [np.rint(pts).astype(np.int32)], 1)
    return {k: v.astype(bool) for k, v in acc.items()}


def apply_region_priority(masks):
    """body_valid = body_raw AND NOT label_union AND NOT cap_union."""
    body = masks["body"] & ~masks["label"] & ~masks["cap"]
    return {"body": body, "cap": masks["cap"], "label": masks["label"]}


def cutout_mask_to_hsi(mask_cut, row, transform, hsi_shape, erode_px=0):
    """Boolean mask in cutout space -> boolean mask on the raw HSI grid.

    `erode_px` is in HSI grid pixels. Erosion is applied in cutout space (1:1 with RGB)
    before projection, since the projected nearest-neighbour mask can contain holes; the
    iteration count is scaled by 1/ALIGN_SX to convert HSI px to RGB px.
    """
    if erode_px > 0 and mask_cut.any():
        iters = max(1, int(round(float(erode_px) / ALIGN_SX)))
        eroded = ndimage.binary_erosion(mask_cut, iterations=iters)
        if eroded.any():                 # keep the uneroded mask if erosion empties it
            mask_cut = eroded
    ys, xs = np.nonzero(mask_cut)
    out = np.zeros(hsi_shape, dtype=bool)
    if xs.size == 0:
        return out
    x_rgb = xs - int(row["canvas_offset_x"]) + int(row["mask_bbox_x1"])
    y_rgb = ys - int(row["canvas_offset_y"]) + int(row["mask_bbox_y1"])
    keep = y_rgb < RGB_CROP_ROWS         # no HSI counterpart above this row
    x_rgb, y_rgb = x_rgb[keep], y_rgb[keep]
    if x_rgb.size == 0:
        return out
    hx, hy = transform.map_points(x_rgb, y_rgb)
    hx = np.rint(hx).astype(np.int64)
    hy = np.rint(hy).astype(np.int64)
    ok = (hx >= 0) & (hx < hsi_shape[1]) & (hy >= 0) & (hy < hsi_shape[0])
    out[hy[ok], hx[ok]] = True
    return out


def extract_true_gt_one(row, erode_px, cube=None):
    """Returns (raw spectra dict, hsi px counts, cutout masks, hsi masks, cube, transform).

    A region with zero surviving HSI pixels yields spectra[key] = None.
    """
    frame = _frame_row(row["stem"])
    if cube is None:
        cube = load_cube(_as_text(frame["hyper_path"]))
    hsi_shape = cube.shape[1:]
    tf = get_transform(_as_text(row["stem"]),
                       session=(_as_text(row.get("session")) or None),
                       hsi_shape=hsi_shape)

    label_txt = _as_text(row.get("true_gt_label_abs"))
    if not label_txt:
        raise ValueError(f"{row.get('cutout_id')}: no true-GT label path on this row")
    masks_cut = parse_yolo_seg(label_txt, int(row["canvas_w"]), int(row["canvas_h"]))
    masks_cut = apply_region_priority(masks_cut)

    spectra, counts, hsi_masks = {}, {}, {}
    for key in REGION_ORDER:
        mc = masks_cut[key]
        if not mc.any():
            counts[key] = 0
            hsi_masks[key] = np.zeros(hsi_shape, dtype=bool)
            spectra[key] = None
            continue
        hm = cutout_mask_to_hsi(mc, row, tf, hsi_shape, erode_px=erode_px)
        n = int(hm.sum())
        counts[key] = n
        hsi_masks[key] = hm
        spectra[key] = cube[:, hm].mean(axis=1) if n >= 1 else None
    return spectra, counts, masks_cut, hsi_masks, cube, tf


# --- IARR reference ---

def get_iarr_reference(pool, force=False):
    """Dataset-wide per-band mean DN over every distinct frame in the true-GT pool.

    Computed on the true-GT pool (not the pseudo-GT pool); cached to disk keyed on
    TRUE_GT_RUN_ID.
    """
    if not force and TRUEGT["iarr_reference"] is not None:
        return TRUEGT["iarr_reference"]
    if not force and IARR_REF_CACHE.exists():
        ref = pd.read_csv(IARR_REF_CACHE)
        TRUEGT["iarr_reference"] = ref
        return ref
    stems = sorted(set(pool["stem"].map(_as_text)) - {""})
    print(f"building IARR reference over {len(stems):,} distinct true-GT frames ...")
    rep = ProgressReporter("truegt_iarr_reference", len(stems))
    band_sum, n = None, 0
    for s in stems:
        try:
            cube = load_cube(_as_text(_frame_row(s)["hyper_path"]))
            m = cube.reshape(cube.shape[0], -1).mean(axis=1)
            band_sum = m if band_sum is None else band_sum + m
            n += 1
        except Exception as e:
            print(f"!! {s}: {type(e).__name__}: {e}")
        rep.update()
    rep.close()
    if band_sum is None:
        raise RuntimeError("No cubes readable - IARR reference could not be built.")
    ref = pd.DataFrame({"band_idx": np.arange(len(band_sum)),
                        "wavelength_nm": np.linspace(WL_MIN_NM, WL_MAX_NM, len(band_sum)),
                        "mean_dn": band_sum / n,
                        "n_frames": n})
    IARR_REF_CACHE.parent.mkdir(parents=True, exist_ok=True)
    ref.to_csv(IARR_REF_CACHE, index=False)
    TRUEGT["iarr_reference"] = ref
    print(f"IARR reference -> {IARR_REF_CACHE}  ({n:,} frames)")
    return ref


# --- Preprocessing ---

def preprocess_spectrum(wl, dn, ref_dn=None, cut=None, do_iarr=False, do_snv=False):
    """Applied in order: cut -> IARR -> SNV. Returns (wavelengths, values)."""
    wl = np.asarray(wl, dtype=np.float64)
    y = np.asarray(dn, dtype=np.float64)
    m = np.ones(wl.size, dtype=bool)
    if cut is not None:
        lo, hi = min(cut), max(cut)
        m = (wl >= lo) & (wl <= hi)
    x, y = wl[m], y[m]
    if do_iarr:
        if ref_dn is None:
            raise ValueError("IARR requested without a reference spectrum.")
        d = np.asarray(ref_dn, dtype=np.float64)[m]
        d = np.where(np.abs(d) < 1e-8, np.nan, d)
        y = y / d
    if do_snv:
        mu = float(np.nanmean(y))
        sd = float(np.nanstd(y))          # ddof=0, project convention
        y = (y - mu) / (sd + 1e-9)
    return x, y


def write_spectrum_file(path, wl, values):
    """Write two numeric columns, semicolon-separated, no header."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"wavelength_nm": wl, "value": values}).to_csv(
        path, sep=SPECTRUM_SEP, header=False, index=False,
        float_format=SPECTRUM_FLOAT_FMT)


# --- Widgets ---

w12_erode = widgets.IntSlider(value=ERODE_PX_DEFAULT, min=0, max=20,
                              description="mask erode HSI px:",
                              style={"description_width": "150px"},
                              layout=widgets.Layout(width="430px"))
w12_iarr = widgets.Checkbox(value=False, indent=False, description="IARR correction")
w12_snv = widgets.Checkbox(value=False, indent=False, description="SNV normalisation")
w12_wlmin = widgets.FloatSlider(value=WL_MIN_NM, min=WL_MIN_NM, max=WL_MAX_NM, step=5,
                                description="cut wl min (nm):", readout_format=".0f",
                                style={"description_width": "150px"},
                                layout=widgets.Layout(width="430px"))
w12_wlmax = widgets.FloatSlider(value=WL_MAX_NM, min=WL_MIN_NM, max=WL_MAX_NM, step=5,
                                description="cut wl max (nm):", readout_format=".0f",
                                style={"description_width": "150px"},
                                layout=widgets.Layout(width="430px"))
w12_stack = widgets.Checkbox(value=False, indent=False, description="Offset-stack spectra")
w12_seed = widgets.IntText(value=0, description="Preview seed:",
                           style={"description_width": "150px"},
                           layout=widgets.Layout(width="250px"))
w12_btn_prev = widgets.Button(description="Preview random object", button_style="info",
                              icon="image", layout=widgets.Layout(width="220px"))
w12_btn_export = widgets.Button(description="Export all spectra", button_style="success",
                                icon="save", layout=widgets.Layout(width="200px"))
w12_status = widgets.HTML()
w12_out = widgets.Output()


def _cut_range():
    """Cut is active whenever the sliders narrow the full acquisition range."""
    lo, hi = float(w12_wlmin.value), float(w12_wlmax.value)
    lo, hi = min(lo, hi), max(lo, hi)
    active = (lo > WL_MIN_NM) or (hi < WL_MAX_NM)
    return ((lo, hi) if active else None), active


def _variant_tag():
    cut, cut_active = _cut_range()
    bits = []
    if cut_active:
        bits.append(f"cut{int(cut[0])}-{int(cut[1])}")
    if w12_iarr.value:
        bits.append("iarr")
    if w12_snv.value:
        bits.append("snv")
    return "+".join(bits) if bits else "raw"


def _status12(extra=""):
    try:
        n = len(load_true_gt_pool())
    except Exception:
        n = 0
    w12_status.value = (
        f"<div style='font-family:monospace;font-size:12px;padding:6px'>"
        f"true-GT pool <b>{n:,}</b> objects &nbsp;|&nbsp; variant <b>{_variant_tag()}</b>"
        f" &nbsp;|&nbsp; erode <b>{int(w12_erode.value)}</b> px{extra}</div>")


def render_preview12(_=None):
    with w12_out:
        clear_output(wait=True)
        try:
            ensure_alignment()
            pool = load_true_gt_pool()
        except Exception as e:
            print(f"!! {type(e).__name__}: {e}")
            traceback.print_exc()
            return
        if pool.empty:
            print("!! No accepted true-GT objects with usable geometry.")
            return

        cut, cut_active = _cut_range()
        ref = None
        if w12_iarr.value:
            try:
                ref = get_iarr_reference(pool)["mean_dn"].to_numpy()
            except Exception as e:
                print(f"!! IARR reference unavailable: {type(e).__name__}: {e}")
                traceback.print_exc()
                return

        row = pool.sample(n=1, random_state=int(w12_seed.value)).iloc[0]
        t0 = time.time()
        try:
            spectra, counts, masks_cut, hsi_masks, cube, tf = extract_true_gt_one(
                row, int(w12_erode.value))
        except Exception as e:
            print(f"!! {row.get('cutout_id')}: {type(e).__name__}: {e}")
            traceback.print_exc()
            return
        dt = time.time() - t0
        wl_full = np.linspace(WL_MIN_NM, WL_MAX_NM, cube.shape[0])

        fig, axes = plt.subplots(1, 3, figsize=(_FW * 2.0, _FW * 0.55))
        try:
            axes[0].imshow(np.array(
                Image.open(_as_text(row.get("cutout_path"))).convert("RGB")))
        except Exception:
            axes[0].imshow(np.zeros((int(row["canvas_h"]), int(row["canvas_w"]), 3),
                                    dtype=np.uint8))
        for k in REGION_ORDER:
            if masks_cut[k].any():
                axes[0].contourf(masks_cut[k], levels=[0.5, 1], colors=[REGION_COLORS[k]],
                                 alpha=0.35)
        axes[0].set_title("True GT in Cutout Space")
        axes[0].axis("off")

        axes[1].imshow(hsi_false_colour(cube))
        for k, m in hsi_masks.items():
            if m.any():
                axes[1].contourf(m, levels=[0.5, 1], colors=[REGION_COLORS[k]], alpha=0.50)
        axes[1].set_title(f"Projected Onto HSI Grid {cube.shape[1]}x{cube.shape[2]}")
        axes[1].axis("off")

        curves = {}
        for k in REGION_ORDER:
            if spectra.get(k) is None:
                continue
            x, y = preprocess_spectrum(wl_full, spectra[k], ref_dn=ref, cut=cut,
                                       do_iarr=bool(w12_iarr.value),
                                       do_snv=bool(w12_snv.value))
            curves[k] = (x, y)
        if curves:
            spans = [np.nanmax(y) - np.nanmin(y) for _, y in curves.values()]
            step = float(np.nanmax(spans)) * 1.25 if spans else 1.0
            off = 0.0
            for k in REGION_ORDER:
                if k not in curves:
                    continue
                x, y = curves[k]
                yy = y + (off if w12_stack.value else 0.0)
                axes[2].plot(x, yy, color=REGION_COLORS[k], lw=1.5,
                             label=f"{k.title()} (n={counts[k]})")
                if w12_stack.value:
                    axes[2].text(x[0], yy[0], f" {k.title()}", color=REGION_COLORS[k],
                                 fontsize=_FONT_SIZE - 1, va="bottom")
                    off += step
            axes[2].legend(loc="best")
        axes[2].set_xlabel("Wavelength (nm)")
        if w12_snv.value:
            ylab = "SNV Standardised"
        elif w12_iarr.value:
            ylab = "IARR Relative Reflectance"
        else:
            ylab = "Digital Number (DN)"
        axes[2].set_ylabel(ylab)
        if w12_stack.value:
            axes[2].set_yticks([])
        axes[2].set_title("Per-Region Spectrum")

        fig.suptitle(f"{row['cutout_id']}   variant {_variant_tag()}   "
                     f"erode {int(w12_erode.value)} px", y=1.03)
        plt.tight_layout()
        plt.show()

        print(f"session          : {row.get('session', '')}   "
              f"alignment source: {tf.source}")
        print(f"canvas           : {int(row['canvas_w'])}x{int(row['canvas_h'])} px "
              f"(true-GT polygons denormalised against this, not 640x384)")
        print("region pixel counts (no minimum floor; a region is dropped only at 0 px):")
        for k in REGION_ORDER:
            n_cut = int(masks_cut[k].sum())
            flag = "   << empty, no file written" if counts[k] == 0 else ""
            print(f"  {k:6s} cutout {n_cut:8,d} px | HSI {counts[k]:6,d} px{flag}")
        print(f"\n{dt:.2f} s/object -> projected export over {len(pool):,} objects: "
              f"{estimate_batch_runtime(dt, len(pool))}")
        _status12()


def export12(_=None):
    with w12_out:
        clear_output(wait=True)
        try:
            ensure_alignment()
            pool = load_true_gt_pool()
        except Exception as e:
            print(f"!! {type(e).__name__}: {e}")
            traceback.print_exc()
            return
        if pool.empty:
            print("!! Nothing to export.")
            return

        cut, cut_active = _cut_range()
        do_iarr, do_snv = bool(w12_iarr.value), bool(w12_snv.value)
        erode_px = int(w12_erode.value)
        ref_df = get_iarr_reference(pool) if do_iarr else None
        ref = ref_df["mean_dn"].to_numpy() if ref_df is not None else None

        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_dir = SPECTRA_EXPORT_PARENT / f"{SPECTRA_EXPORT_PREFIX}_{stamp}"
        params_path = SPECTRA_EXPORT_PARENT / f"{SPECTRA_EXPORT_PREFIX}_{stamp}.txt"
        out_dir.mkdir(parents=True, exist_ok=False)

        print(f"exporting {len(pool):,} objects -> {out_dir}")
        rep = ProgressReporter("truegt_spectra_export", len(pool))
        meta_rows, failures = [], []
        n_files = 0
        for _, row in pool.iterrows():
            try:
                spectra, counts, masks_cut, _hm, cube, tf = extract_true_gt_one(row, erode_px)
                wl_full = np.linspace(WL_MIN_NM, WL_MAX_NM, cube.shape[0])
                for k in REGION_ORDER:
                    rec = {"cutout_id": row["cutout_id"], "stem": row["stem"],
                           "session": row.get("session", ""), "split": row.get("split", ""),
                           "region": k,
                           "n_px_cutout": int(masks_cut[k].sum()),
                           "n_px_hsi": int(counts.get(k, 0)),
                           "canvas_w": int(row["canvas_w"]), "canvas_h": int(row["canvas_h"]),
                           "align_source": tf.source, "written": False, "file_name": ""}
                    s = spectra.get(k)
                    if s is not None and counts.get(k, 0) >= 1:
                        x, y = preprocess_spectrum(wl_full, s, ref_dn=ref, cut=cut,
                                                   do_iarr=do_iarr, do_snv=do_snv)
                        fname = f"{row['cutout_id']}_{k}{SPECTRUM_FILE_EXT}"
                        write_spectrum_file(out_dir / fname, x, y)
                        rec["written"] = True
                        rec["file_name"] = fname
                        n_files += 1
                    meta_rows.append(rec)
            except Exception as e:
                failures.append((str(row["cutout_id"]), f"{type(e).__name__}: {e}"))
                for k in REGION_ORDER:
                    meta_rows.append({"cutout_id": row["cutout_id"], "stem": row["stem"],
                                      "session": row.get("session", ""),
                                      "split": row.get("split", ""), "region": k,
                                      "n_px_cutout": 0, "n_px_hsi": 0,
                                      "canvas_w": int(row.get("canvas_w", 0)),
                                      "canvas_h": int(row.get("canvas_h", 0)),
                                      "align_source": "", "written": False,
                                      "file_name": "", "error": f"{type(e).__name__}: {e}"})
            rep.update()
        rep.close()

        meta = pd.DataFrame(meta_rows)
        meta_path = out_dir / "true_gt_spectra_meta.csv"
        meta.to_csv(meta_path, index=False)
        if ref_df is not None:
            ref_df.to_csv(out_dir / "iarr_reference_spectrum.csv", index=False)

        n_obj = int(meta["cutout_id"].nunique())
        n_fb = int((meta["align_source"] == "median_fallback").sum())
        lines = [
            "=== SpectralWaste true-GT per-region spectra export ===",
            f"timestamp                 : {stamp}",
            f"utc                       : {utcnow()}",
            f"config_hash               : {CONFIG_HASH}",
            f"notebook_run_id           : {RUN_ID}",
            "",
            "--- source ---",
            f"true_gt_run_id            : {TRUE_GT_RUN_ID}",
            f"true_gt_labels_dir        : {TRUE_GT_LABELS_DIR}",
            f"export_index_csv          : {TRUE_GT_INDEX_CSV}",
            f"geometry_source           : {pool['geometry_source'].iloc[0]}",
            f"alignment_run_id          : {ALIGN_RUN_ID}",
            f"alignment_offsets_csv     : {ALIGN.get('source_csv', '')}",
            f"alignment_dx_px           : {ALIGN.get('dx')}",
            f"align_scale_x             : {ALIGN_SX:.7f}",
            f"align_scale_y             : {ALIGN_SY:.7f}",
            "",
            "--- geometry / masking ---",
            "polygon_denorm_reference  : per-object canvas_w x canvas_h (not 640x384)",
            "region_priority           : body_valid = body AND NOT label AND NOT cap",
            f"erode_px_hsi              : {erode_px}",
            "erode_units               : hsi_grid_px",
            "erode_rationale           : sized to measured parallax residual (~3.3 px)",
            f"rgb_crop_rows             : {RGB_CROP_ROWS}",
            "min_region_px             : none (every region with >=1 HSI px is exported)",
            "",
            "--- preprocessing (applied in this order) ---",
            f"cut_applied               : {cut_active}",
            f"cut_wl_min_nm             : {cut[0] if cut_active else WL_MIN_NM}",
            f"cut_wl_max_nm             : {cut[1] if cut_active else WL_MAX_NM}",
            f"iarr_applied              : {do_iarr}",
            f"iarr_reference_scope      : dataset-wide mean over true-GT-pool frames",
            f"iarr_reference_n_frames   : {int(ref_df['n_frames'].iloc[0]) if ref_df is not None else ''}",
            f"iarr_reference_csv        : {IARR_REF_CACHE if ref_df is not None else ''}",
            f"snv_applied               : {do_snv}",
            "snv_ddof                  : 0",
            f"variant_tag               : {_variant_tag()}",
            "",
            "--- output ---",
            f"output_folder             : {out_dir}",
            f"file_format               : two columns, sep='{SPECTRUM_SEP}', no header, "
            f"float_format='{SPECTRUM_FLOAT_FMT}'",
            "file_columns              : wavelength_nm; value",
            f"file_naming               : <cutout_id>_<region>{SPECTRUM_FILE_EXT or ' (no extension)'}",
            f"meta_csv                  : {meta_path}",
            f"n_objects                 : {n_obj}",
            f"n_spectrum_files          : {n_files}",
            f"n_median_fallback_rows    : {n_fb}",
            f"n_failed_objects          : {len(failures)}",
        ]
        for k in REGION_ORDER:
            sub = meta[meta["region"] == k]
            lines.append(f"n_written_{k:<15s}: {int(sub['written'].sum())}")
        if failures:
            lines.append("failed_objects:")
            lines.extend([f"- {cid}: {err}" for cid, err in failures])
        else:
            lines.append("failed_objects            : none")
        params_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

        TRUEGT["last_export"] = out_dir
        print(f"\nobjects          : {n_obj:,}")
        print(f"spectrum files   : {n_files:,}")
        for k in REGION_ORDER:
            sub = meta[meta["region"] == k]
            v = sub["n_px_hsi"]
            print(f"  {k:6s} written {int(sub['written'].sum()):4,d} | "
                  f"median {v[v > 0].median() if (v > 0).any() else 0:7.1f} HSI px | "
                  f"empty {int((v == 0).sum()):,}")
        if n_fb:
            print(f"!! {n_fb:,} rows used the MEDIAN DY - their capture session was never "
                  f"measured in the alignment run.")
        if failures:
            print(f"!! {len(failures):,} objects failed; see the parameters file.")
        print(f"\nspectra    -> {out_dir}")
        print(f"meta       -> {meta_path}")
        print(f"parameters -> {params_path}")

        fig, ax = plt.subplots(figsize=(_FW, _FW * 0.45))
        for k in REGION_ORDER:
            v = meta.loc[meta["region"] == k, "n_px_hsi"]
            v = v[v > 0]
            if len(v):
                ax.hist(v, bins=np.logspace(0, np.log10(max(v.max(), 10)), 40),
                        alpha=0.55, color=REGION_COLORS[k], label=k.title())
        ax.set_xscale("log")
        ax.set_xlabel("Region Size on HSI Grid (px)")
        ax.set_ylabel("Objects")
        ax.set_title("True-GT Spectral Sampling Adequacy")
        ax.legend(loc="upper right")
        plt.tight_layout()
        fig.savefig(DIR_FIGURES / f"true_gt_region_pixels_{stamp}.png", dpi=150,
                    bbox_inches="tight")
        plt.show()
        _status12()


w12_btn_prev.on_click(render_preview12)
w12_btn_export.on_click(export12)
for _w in (w12_erode, w12_iarr, w12_snv, w12_wlmin, w12_wlmax):
    _w.observe(lambda ch: _status12(), names="value")

_status12()
display(widgets.VBox([
    widgets.HBox([w12_erode, w12_iarr, w12_snv]),
    widgets.HBox([w12_wlmin, w12_wlmax]),
    widgets.HBox([w12_seed, w12_stack]),
    widgets.HBox([w12_btn_prev, w12_btn_export]),
    w12_status,
]), w12_out)


In [ ]:
# === Cell 14: Aggregate Spectral Analysis ===
# Per-region mean spectra from an exported preprocessed_spectra_<timestamp>/ folder.
# No correction is applied here; the folder carries the cut/IARR/SNV variant chosen at
# export.

AGG = {"folder": None, "long": None, "meta": None, "last": None}


def list_spectra_exports():
    if not SPECTRA_EXPORT_PARENT.is_dir():
        return []
    return sorted([p for p in SPECTRA_EXPORT_PARENT.iterdir()
                   if p.is_dir() and p.name.startswith(SPECTRA_EXPORT_PREFIX)],
                  key=lambda p: p.name, reverse=True)


def read_spectrum_file(path):
    """Read a two-column, semicolon-separated spectrum file without header."""
    df = pd.read_csv(path, sep=SPECTRUM_SEP, header=None,
                     names=["wavelength_nm", "value"], engine="python")
    df["wavelength_nm"] = pd.to_numeric(df["wavelength_nm"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.dropna().reset_index(drop=True)


def _split_region(stem):
    """'<cutout_id>_<region>' -> (cutout_id, region); None if the suffix is not a region."""
    for k in REGION_ORDER:
        if stem.endswith(f"_{k}"):
            return stem[: -(len(k) + 1)], k
    return None, None


def load_spectra_folder(folder, force=False):
    """All spectrum files in an export folder -> long DataFrame + its meta table."""
    folder = Path(folder)
    if not force and AGG["folder"] == str(folder) and AGG["long"] is not None:
        return AGG["long"], AGG["meta"]
    reserved = {"true_gt_spectra_meta.csv", "iarr_reference_spectrum.csv"}
    rows = []
    for p in sorted(folder.iterdir()):
        if not p.is_file() or p.name in reserved:
            continue
        cid, region = _split_region(p.stem if p.suffix else p.name)
        if region is None:
            continue
        try:
            df = read_spectrum_file(p)
        except Exception:
            continue
        rows.append(pd.DataFrame({"cutout_id": cid, "region": region,
                                  "band_idx": np.arange(len(df)),
                                  "wavelength_nm": df["wavelength_nm"].to_numpy(),
                                  "value": df["value"].to_numpy()}))
    long = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
        columns=["cutout_id", "region", "band_idx", "wavelength_nm", "value"])

    mp = folder / "true_gt_spectra_meta.csv"
    meta = pd.read_csv(mp) if mp.exists() else pd.DataFrame()
    AGG.update({"folder": str(folder), "long": long, "meta": meta})
    return long, meta


def spectra_matrix(long, region):
    """(objects x bands) matrix for one region, plus its wavelength axis and object ids."""
    g = long[long["region"] == region]
    if g.empty:
        return None, None, None
    piv = g.pivot_table(index="cutout_id", columns="band_idx", values="value")
    wl = g.groupby("band_idx")["wavelength_nm"].first().reindex(piv.columns).to_numpy()
    return piv.index.to_numpy(), piv.to_numpy(dtype=np.float64), wl


w13_folder = widgets.Dropdown(options=[(p.name, str(p)) for p in list_spectra_exports()],
                              description="Export folder:",
                              style={"description_width": "130px"},
                              layout=widgets.Layout(width="520px"))
w13_refresh = widgets.Button(description="Refresh", icon="refresh",
                             layout=widgets.Layout(width="120px"))
w13_regions = widgets.SelectMultiple(options=list(REGION_ORDER), value=tuple(REGION_ORDER),
                                     description="Regions:",
                                     style={"description_width": "130px"},
                                     layout=widgets.Layout(width="300px", height="80px"))
w13_minpx = widgets.IntSlider(value=1, min=1, max=500, description="min HSI px:",
                              style={"description_width": "130px"},
                              layout=widgets.Layout(width="400px"))
w13_stacked = widgets.Checkbox(value=False, indent=False, description="Stacked / offset")
w13_std = widgets.Checkbox(value=True, indent=False, description="Show std band")
w13_btn = widgets.Button(description="Render", button_style="info", icon="chart-line")
w13_save = widgets.Button(description="Save figure + CSV", icon="save")
w13_out = widgets.Output()


def _refresh13(_=None):
    opts = [(p.name, str(p)) for p in list_spectra_exports()]
    w13_folder.options = opts
    if opts:
        w13_folder.value = opts[0][1]


def _px_filter(long, meta, min_px):
    """Drop object x region pairs below a pixel count, using the export's meta table."""
    if meta is None or meta.empty or min_px <= 1 or "n_px_hsi" not in meta.columns:
        return long
    ok = meta[meta["n_px_hsi"] >= min_px][["cutout_id", "region"]].drop_duplicates()
    return long.merge(ok, on=["cutout_id", "region"], how="inner")


def render13(_=None):
    with w13_out:
        clear_output(wait=True)
        if not w13_folder.value:
            print("!! No export folder found - run Cell 13's export first.")
            return
        folder = Path(w13_folder.value)
        long, meta = load_spectra_folder(folder)
        if long.empty:
            print(f"!! No spectrum files in {folder}")
            return
        long = _px_filter(long, meta, int(w13_minpx.value))
        long = long[long["region"].isin(list(w13_regions.value))]
        if long.empty:
            print("!! No spectra match these settings.")
            return

        fig, ax = plt.subplots(figsize=(_FW, _FW * 0.55))
        mats, offset, step = {}, 0.0, 0.0
        for region in REGION_ORDER:
            ids, M, wl = spectra_matrix(long, region)
            if M is not None:
                mats[region] = (ids, M, wl)
        if w13_stacked.value and mats:
            spans = [np.nanmax(np.nanmean(M, axis=0)) - np.nanmin(np.nanmean(M, axis=0))
                     for _i, M, _w in mats.values()]
            step = float(np.nanmax(spans)) * 1.25 if spans else 1.0

        for region in REGION_ORDER:
            if region not in mats:
                continue
            _ids, M, wl = mats[region]
            mu = np.nanmean(M, axis=0)
            sd = np.nanstd(M, axis=0)          # ddof=0, project convention
            y = mu + (offset if w13_stacked.value else 0.0)
            ax.plot(wl, y, color=REGION_COLORS[region], lw=1.6,
                    label=f"{region.title()} (n={M.shape[0]})")
            if w13_std.value:
                ax.fill_between(wl, y - sd, y + sd, color=REGION_COLORS[region],
                                alpha=0.18, linewidth=0)
            if w13_stacked.value:
                ax.text(wl[0], y[0], f" {region.title()}", color=REGION_COLORS[region],
                        fontsize=_FONT_SIZE - 1, va="bottom")
                offset += step

        ax.set_xlabel("Wavelength (nm)")
        ax.set_ylabel("Signal")
        if w13_stacked.value:
            ax.set_yticks([])
        else:
            ax.legend(loc="best")
        ax.set_title("SpectralWaste True-GT Per-Region Mean Spectra")
        plt.tight_layout()
        plt.show()

        rows = [{"region": r, "n_objects": int(M.shape[0]), "n_bands": int(M.shape[1]),
                 "wl_min_nm": round(float(np.nanmin(wl)), 1),
                 "wl_max_nm": round(float(np.nanmax(wl)), 1),
                 "mean_level": round(float(np.nanmean(M)), 4)}
                for r, (_i, M, wl) in mats.items()]
        display(HTML(pd.DataFrame(rows).to_html(index=False)))
        print(f"folder: {folder.name}")
        pt = SPECTRA_EXPORT_PARENT / f"{folder.name}.txt"
        if pt.exists():
            for line in pt.read_text().splitlines():
                if line.startswith(("variant_tag", "iarr_applied", "snv_applied",
                                    "cut_applied", "erode_px_hsi")):
                    print("  " + line)
        AGG["last"] = (mats, folder)


def save13(_=None):
    with w13_out:
        last = AGG.get("last")
        if not last:
            print("Render first.")
            return
        mats, folder = last
        rows = []
        for region, (_ids, M, wl) in mats.items():
            mu, sd = np.nanmean(M, axis=0), np.nanstd(M, axis=0)
            for i, w_ in enumerate(wl):
                rows.append({"region": region, "wavelength_nm": float(w_),
                             "mean": float(mu[i]), "std": float(sd[i]),
                             "n_objects": int(M.shape[0])})
        p = DIR_TABLES / f"true_gt_aggregate_spectra_{folder.name}.csv"
        pd.DataFrame(rows).to_csv(p, index=False)
        f = DIR_FIGURES / f"true_gt_aggregate_spectra_{folder.name}.png"
        plt.gcf().savefig(f, dpi=150, bbox_inches="tight")
        log_params("truegt_aggregate", figure_params={
            "export_folder": str(folder), "regions": list(w13_regions.value),
            "min_px": int(w13_minpx.value), "stacked": bool(w13_stacked.value)},
            outputs={"csv": str(p), "figure": str(f)})
        print(f"saved -> {p}\nsaved -> {f}")


w13_refresh.on_click(_refresh13)
w13_btn.on_click(render13)
w13_save.on_click(save13)
display(widgets.VBox([
    widgets.HBox([w13_folder, w13_refresh]),
    widgets.HBox([w13_regions, w13_minpx]),
    widgets.HBox([w13_stacked, w13_std]),
    widgets.HBox([w13_btn, w13_save]),
]), w13_out)

In [ ]:
# === Cell 15: PCA Cluster Screening ===
# Exploratory PCA on per-object mean spectra (one row per cutout_id x region) from an
# exported folder, to screen for material heterogeneity within a region. Colour defaults
# to session (confound check), or region when Scope=full; no cluster count is imposed.
# The 3D plot is rendered as inline HTML (fig.show() requires nbformat in the kernel).

import uuid

try:
    from sklearn.decomposition import PCA as _SkPCA
except ImportError:
    _SkPCA = None

try:
    import plotly.graph_objects as go
except ImportError:
    go = None

w13b_folder = widgets.Dropdown(options=[(p.name, str(p)) for p in list_spectra_exports()],
                               description="Export folder:",
                               style={"description_width": "130px"},
                               layout=widgets.Layout(width="520px"))
w13b_refresh = widgets.Button(description="Refresh", icon="refresh",
                              layout=widgets.Layout(width="120px"))
w13b_scope = widgets.Dropdown(options=["body", "label", "cap", "full"], value="body",
                              description="Scope:", style={"description_width": "130px"},
                              layout=widgets.Layout(width="260px"))
w13b_minpx = widgets.IntSlider(value=1, min=1, max=500, description="min HSI px:",
                               style={"description_width": "130px"},
                               layout=widgets.Layout(width="400px"))
w13b_colorby = widgets.Dropdown(options=["session", "region", "split", "mean_level_bin"],
                                value="session", description="Colour by:",
                                style={"description_width": "130px"},
                                layout=widgets.Layout(width="300px"))
w13b_npcs = widgets.IntSlider(value=3, min=2, max=10, description="n components:",
                              style={"description_width": "130px"},
                              layout=widgets.Layout(width="380px"))
w13b_btn = widgets.Button(description="Run PCA", button_style="info", icon="braille")
w13b_save = widgets.Button(description="Save figures + CSV", icon="save")
w13b_out = widgets.Output()


def _on_scope13b(change):
    """Colour by region for Scope=full, otherwise by session."""
    w13b_colorby.value = "region" if change["new"] == "full" else "session"


w13b_scope.observe(_on_scope13b, names="value")


def _refresh13b(_=None):
    opts = [(p.name, str(p)) for p in list_spectra_exports()]
    w13b_folder.options = opts
    if opts:
        w13b_folder.value = opts[0][1]


def _fig_to_click_html13b(fig):
    """Inline-HTML Plotly render with a click-to-identify info panel."""
    div_id = f"plotly_{uuid.uuid4().hex}"
    info_id = f"info_{uuid.uuid4().hex}"
    plot_html = fig.to_html(full_html=False, include_plotlyjs="cdn", div_id=div_id,
                            config={"responsive": True, "displaylogo": False,
                                    "scrollZoom": True})
    return f"""
    <div id="{info_id}" style="margin:0 0 12px 0; padding:10px 12px;
        border:1px solid #d0d7de; border-radius:8px; background:#f6f8fa; font-family:sans-serif;">
        Click a point to see which object it is.
    </div>
    {plot_html}
    <script>
    (function() {{
        const gd = document.getElementById("{div_id}");
        const info = document.getElementById("{info_id}");
        if (!gd || !info) return;
        function esc(v) {{
            return String(v ?? "").replace(/&/g,"&amp;").replace(/</g,"&lt;")
                .replace(/>/g,"&gt;").replace(/"/g,"&quot;").replace(/'/g,"&#39;");
        }}
        gd.on('plotly_click', function(data) {{
            if (!data || !data.points || !data.points.length) return;
            const pt = data.points[0], cd = pt.customdata || [];
            info.innerHTML =
                "<b>Object:</b> " + esc(cd[0]) + "<br>" +
                "<b>Region:</b> " + esc(cd[1]) + " &nbsp; " +
                "<b>Session:</b> " + esc(cd[2]) + " &nbsp; " +
                "<b>Split:</b> " + esc(cd[3]) + "<br>" +
                "<b>PC1:</b> " + Number(pt.x).toFixed(3) + ", " +
                "<b>PC2:</b> " + Number(pt.y).toFixed(3) + ", " +
                "<b>PC3:</b> " + Number(pt.z).toFixed(3);
        }});
    }})();
    </script>
    """


def _pca_build13b(folder, scope, min_px):
    """(objects x bands) matrix with cutout_id + region kept, from an export folder."""
    long, meta = load_spectra_folder(folder)
    if long.empty:
        return None, None, None, meta
    long = _px_filter(long, meta, min_px)
    if scope != "full":
        long = long[long["region"] == scope]
    if long.empty:
        return None, None, None, meta

    frames, wl = [], None
    for region in REGION_ORDER:
        ids, M, wl_r = spectra_matrix(long, region)
        if M is None:
            continue
        if wl is None:
            wl = wl_r
        elif wl_r.shape != wl.shape or not np.allclose(wl, wl_r):
            continue      # different band grid - skip rather than misalign columns
        frames.append((pd.DataFrame({"cutout_id": ids, "region": region}), M))
    if not frames:
        return None, None, None, meta
    idx_all = pd.concat([f[0] for f in frames], ignore_index=True)
    M_all = np.vstack([f[1] for f in frames])
    return idx_all, M_all, wl, meta


def render13b(_=None):
    with w13b_out:
        clear_output(wait=True)
        if not w13b_folder.value:
            print("!! No export folder found - run Cell 13's export first.")
            return
        folder = Path(w13b_folder.value)
        idx, M, wl, meta = _pca_build13b(folder, w13b_scope.value, int(w13b_minpx.value))
        if idx is None:
            print("!! No spectra match these settings.")
            return
        if _SkPCA is None:
            print("!! scikit-learn not installed - PCA cannot run.")
            return

        keep = ~np.isnan(M).any(axis=1)
        n_dropped = int((~keep).sum())
        idx, M = idx[keep].reset_index(drop=True), M[keep]
        if M.shape[0] < 3:
            print(f"!! Only {M.shape[0]} objects with complete spectra after dropping "
                  f"{n_dropped} with NaN bands - too few for PCA.")
            return

        n_pcs = min(int(w13b_npcs.value), M.shape[0] - 1, M.shape[1])
        pca = _SkPCA(n_components=n_pcs, random_state=42)
        scores = pca.fit_transform(M - M.mean(axis=0, keepdims=True))
        evr = pca.explained_variance_ratio_

        if meta is not None and not meta.empty:
            cols = [c for c in ("cutout_id", "region", "session", "split") if c in meta.columns]
            idx = idx.merge(meta[cols].drop_duplicates(["cutout_id", "region"]),
                            on=["cutout_id", "region"], how="left")
        for c in ("session", "split"):
            if c not in idx.columns:
                idx[c] = ""
            idx[c] = idx[c].fillna("").astype(str)
        idx["mean_level"] = M.mean(axis=1)
        try:
            idx["mean_level_bin"] = pd.qcut(idx["mean_level"], 4,
                                            labels=["Q1(low)", "Q2", "Q3", "Q4(high)"],
                                            duplicates="drop").astype(str)
        except Exception:
            idx["mean_level_bin"] = "all"

        color_col = w13b_colorby.value if w13b_colorby.value in idx.columns else "region"
        color_vals = idx[color_col].astype(str)

        print(f"folder={folder.name}  scope={w13b_scope.value}")
        print(f"objects: {M.shape[0]:,}  ({n_dropped:,} dropped for NaN bands)  "
              f"bands: {M.shape[1]}  PCs: {n_pcs}")
        print("explained variance: "
              + "  ".join(f"PC{i + 1}={v * 100:.1f}%" for i, v in enumerate(evr))
              + f"   cumulative={evr.sum() * 100:.1f}%")

        fig_s, ax_s = plt.subplots(figsize=(_FW * 0.6, _FW * 0.32))
        ax_s.bar(range(1, n_pcs + 1), evr * 100, color=C_EDT_ALIGN)
        ax_s.plot(range(1, n_pcs + 1), np.cumsum(evr) * 100, color="#d1495b",
                  marker="o", lw=1.2)
        ax_s.set_xlabel("Principal Component")
        ax_s.set_ylabel("Variance Explained (%)")
        ax_s.set_title(f"Scree - {w13b_scope.value}")
        plt.tight_layout()
        plt.show()

        fig_l, ax_l = plt.subplots(figsize=(_FW, _FW * 0.32))
        cmap = plt.get_cmap("tab10")
        for i in range(min(n_pcs, 4)):
            ax_l.plot(wl, pca.components_[i], color=cmap(i), lw=1.4,
                      label=f"PC{i + 1} ({evr[i] * 100:.1f}%)")
        ax_l.axhline(0, color="black", lw=0.6)
        ax_l.set_xlabel("Wavelength (nm)")
        ax_l.set_ylabel("Loading")
        ax_l.set_title(f"PCA Loadings - {w13b_scope.value}")
        ax_l.legend(loc="best")
        plt.tight_layout()
        plt.show()

        fig3d = None
        if go is None:
            print("!! plotly not installed - skipping the 3D scatter.")
        else:
            n_show = min(3, n_pcs)
            zs = scores[:, 2] if n_show >= 3 else np.zeros(scores.shape[0])
            cats = sorted(color_vals.unique())
            pal = plt.get_cmap("tab20")
            fig3d = go.Figure()
            for ci, cat in enumerate(cats):
                m = (color_vals == cat).to_numpy()
                if color_col == "region":
                    col = REGION_COLORS.get(cat, "#7f7f7f")
                else:
                    r_, g_, b_, _ = pal(ci % 20)
                    col = f"rgb({int(r_ * 255)},{int(g_ * 255)},{int(b_ * 255)})"
                cd = np.column_stack([
                    idx.loc[m, "cutout_id"].astype(str).to_numpy(),
                    idx.loc[m, "region"].astype(str).to_numpy(),
                    idx.loc[m, "session"].astype(str).to_numpy(),
                    idx.loc[m, "split"].astype(str).to_numpy(),
                ])
                fig3d.add_trace(go.Scatter3d(
                    x=scores[m, 0], y=scores[m, 1], z=zs[m],
                    mode="markers", name=str(cat), customdata=cd,
                    marker=dict(size=4, opacity=0.85, color=col,
                                line=dict(color="black", width=0.4)),
                    hovertemplate=("<b>%{customdata[0]}</b><br>Region: %{customdata[1]}"
                                   "<br>Session: %{customdata[2]}<br>Split: %{customdata[3]}"
                                   "<br>PC1: %{x:.2f}  PC2: %{y:.2f}  PC3: %{z:.2f}"
                                   "<extra></extra>")))
            fig3d.update_layout(
                title=f"PCA - {w13b_scope.value} ({folder.name}, coloured by {color_col})",
                height=760, margin=dict(l=10, r=10, t=50, b=10),
                legend=dict(title=color_col, font=dict(size=12)),
                scene=dict(xaxis_title=f"PC1 ({evr[0] * 100:.1f}%)",
                           yaxis_title=f"PC2 ({evr[1] * 100:.1f}%)" if n_pcs > 1 else "PC2",
                           zaxis_title=(f"PC3 ({evr[2] * 100:.1f}%)" if n_show >= 3 else ""),
                           dragmode="turntable"))
            display(HTML(_fig_to_click_html13b(fig3d)))

        globals()["_LAST_PCA"] = {"idx": idx, "scores": scores, "evr": evr, "wl": wl,
                                  "components": pca.components_, "n_pcs": n_pcs,
                                  "fig_scree": fig_s, "fig_loadings": fig_l,
                                  "fig3d": fig3d, "folder": folder}


def save13b(_=None):
    with w13b_out:
        agg = globals().get("_LAST_PCA")
        if not agg:
            print("Run PCA first.")
            return
        tag = f"{agg['folder'].name}_{w13b_scope.value}"

        scores_df = agg["idx"].copy()
        for i in range(agg["n_pcs"]):
            scores_df[f"PC{i + 1}"] = agg["scores"][:, i]
        p_scores = DIR_TABLES / f"true_gt_pca_scores_{tag}.csv"
        scores_df.to_csv(p_scores, index=False)

        load_df = pd.DataFrame({"wavelength_nm": agg["wl"]})
        for i in range(agg["n_pcs"]):
            load_df[f"PC{i + 1}_loading"] = agg["components"][i]
        p_load = DIR_TABLES / f"true_gt_pca_loadings_{tag}.csv"
        load_df.to_csv(p_load, index=False)

        p_scree = DIR_FIGURES / f"true_gt_pca_scree_{tag}.png"
        p_loadfig = DIR_FIGURES / f"true_gt_pca_loadings_{tag}.png"
        agg["fig_scree"].savefig(p_scree, dpi=150, bbox_inches="tight")
        agg["fig_loadings"].savefig(p_loadfig, dpi=150, bbox_inches="tight")

        p_html = p_png = None
        if agg["fig3d"] is not None:
            p_html = DIR_FIGURES / f"true_gt_pca_3d_{tag}.html"
            p_html.write_text(agg["fig3d"].to_html(include_plotlyjs="cdn"))
            try:
                p_png = DIR_FIGURES / f"true_gt_pca_3d_{tag}.png"
                agg["fig3d"].write_image(str(p_png), width=1400, height=900, scale=2)
            except Exception as e:
                print(f"(3D PNG export skipped - needs `kaleido`: {type(e).__name__})")
                p_png = None

        log_params("truegt_pca_screening", figure_params={
            "export_folder": str(agg["folder"]), "scope": w13b_scope.value,
            "min_px": int(w13b_minpx.value), "n_pcs": agg["n_pcs"],
            "color_by": w13b_colorby.value},
            outputs={"scores_csv": str(p_scores), "loadings_csv": str(p_load),
                     "scree_png": str(p_scree), "loadings_png": str(p_loadfig),
                     "scatter_html": str(p_html) if p_html else "",
                     "scatter_png": str(p_png) if p_png else ""})
        print(f"saved -> {p_scores}\nsaved -> {p_load}\nsaved -> {p_scree}\nsaved -> {p_loadfig}")
        if p_html:
            print(f"saved -> {p_html}")
        if p_png:
            print(f"saved -> {p_png}")


w13b_refresh.on_click(_refresh13b)
w13b_btn.on_click(render13b)
w13b_save.on_click(save13b)
display(widgets.VBox([
    widgets.HBox([w13b_folder, w13b_refresh]),
    widgets.HBox([w13b_scope, w13b_minpx, w13b_colorby]),
    widgets.HBox([w13b_npcs]),
    widgets.HBox([w13b_btn, w13b_save]),
]), w13b_out)

In [ ]:
# === Cell 16: PCA and t-SNE Config Sweep ===
# Standalone (needs only KI_SORT_ROOT): reads the true-GT spectra exports of all four
# preprocessing configs, runs PCA(20) -> t-SNE per config, and reports region /
# session / mean_level_bin colouring plus per-PC eta^2. Outputs go to the run's
# results/ folder.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path
from sklearn.decomposition import PCA as _SkPCA
from sklearn.manifold import TSNE

# --- Plot style ---
_FW = 16 / 2.54
_FONT_SIZE = 10
_PREFERRED_FONTS = ["DejaVu Sans", "Helvetica", "Arial", "Nimbus Sans",
                    "TeX Gyre Heros", "Liberation Sans"]
_available13c = {f.name for f in fm.fontManager.ttflist}
_font13c = next((f for f in _PREFERRED_FONTS if f in _available13c), "DejaVu Sans")
plt.rcParams["font.family"] = _font13c
plt.rcParams["font.size"] = _FONT_SIZE
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["legend.frameon"] = False

REGION_COLORS = {"body": "#FF0000", "cap": "#00FF00", "label": "#0000FF"}

# --- Config ---
DATA_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/KI_Sort_Preprocessing/data")
RESULTS_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/KI_Sort_Preprocessing/"
                    "results/20260901_070019_hsi_preprocessing_v2")
DIR_FIGURES13C = RESULTS_ROOT / "figures"
DIR_TABLES13C = RESULTS_ROOT / "tables"
DIR_FIGURES13C.mkdir(parents=True, exist_ok=True)
DIR_TABLES13C.mkdir(parents=True, exist_ok=True)

CELL13C_CONFIGS = {
    "raw":      "preprocessed_spectra_raw_20260901_071026",
    "iarr":     "preprocessed_spectra_iarr_20260901_071448",
    "snv":      "preprocessed_spectra_snv_20260901_071701",
    "iarr_snv": "preprocessed_spectra_iarr_snv_20260901_071926",
}
CELL13C_REGIONS = ("body", "cap", "label")
CELL13C_NPCS = 20
CELL13C_PERPLEXITY = 30
CELL13C_RANDOM_STATE = 42


def _eta_sq(values, groups):
    df_ = pd.DataFrame({"v": values, "g": groups})
    grand = df_["v"].mean()
    ss_tot = ((df_["v"] - grand) ** 2).sum()
    ss_between = df_.groupby("g")["v"].apply(
        lambda s: len(s) * (s.mean() - grand) ** 2).sum()
    return ss_between / ss_tot if ss_tot > 0 else np.nan


def run_config13c(tag, folder_name):
    """Load one config's true-GT spectra export, run PCA -> t-SNE, compute confound eta^2,
    plot three colourings and save to DIR_TABLES13C / DIR_FIGURES13C.

    Returns (idx_df, eta_df) or (None, None).
    """
    folder = DATA_ROOT / folder_name
    meta_path = folder / "true_gt_spectra_meta.csv"
    if not meta_path.exists():
        print(f"!! [{tag}] meta CSV not found at {meta_path} - skipping.")
        return None, None

    meta = pd.read_csv(meta_path)
    meta = meta[meta["region"].isin(CELL13C_REGIONS)].reset_index(drop=True)

    rows, wl, missing, keep_meta_idx = [], None, [], []
    for i, r in meta.iterrows():
        fpath = folder / f"{r['cutout_id']}_{r['region']}"
        if not fpath.exists():
            missing.append(fpath.name)
            continue
        sp = pd.read_csv(fpath, sep=";", header=None, names=["wavelength_nm", "value"])
        if wl is None:
            wl = sp["wavelength_nm"].to_numpy()
        elif not np.allclose(wl, sp["wavelength_nm"].to_numpy()):
            missing.append(f"{fpath.name} (band grid mismatch - skipped)")
            continue
        rows.append(sp["value"].to_numpy())
        keep_meta_idx.append(i)

    if not rows:
        print(f"!! [{tag}] no spectrum files found under {folder} - skipping.")
        return None, None

    M = np.vstack(rows)
    idx = meta.loc[keep_meta_idx].reset_index(drop=True)
    if missing:
        print(f"!! [{tag}] {len(missing)} files missing/mismatched, skipped "
              f"(e.g. {missing[:3]})")

    keep = ~np.isnan(M).any(axis=1)
    n_dropped = int((~keep).sum())
    idx, M = idx[keep].reset_index(drop=True), M[keep]

    n_pcs = min(CELL13C_NPCS, M.shape[0] - 1, M.shape[1])
    pca = _SkPCA(n_components=n_pcs, random_state=CELL13C_RANDOM_STATE)
    scores = pca.fit_transform(M - M.mean(axis=0, keepdims=True))
    evr = pca.explained_variance_ratio_

    for c in ("session", "split"):
        if c not in idx.columns:
            idx[c] = ""
    idx["mean_level"] = M.mean(axis=1)
    try:
        idx["mean_level_bin"] = pd.qcut(
            idx["mean_level"], 4, labels=["Q1(low)", "Q2", "Q3", "Q4(high)"],
            duplicates="drop").astype(str)
    except Exception:
        idx["mean_level_bin"] = "all"

    print(f"[{tag}] objects: {M.shape[0]:,}  ({n_dropped:,} dropped for NaN bands)  "
          f"PCs: {n_pcs}  cumulative variance: {evr.sum() * 100:.1f}%")

    tsne = TSNE(n_components=2, perplexity=CELL13C_PERPLEXITY, init="pca",
                random_state=CELL13C_RANDOM_STATE)
    emb = tsne.fit_transform(scores)
    idx["tsne_x"] = emb[:, 0]
    idx["tsne_y"] = emb[:, 1]

    eta_rows = []
    for i in range(min(n_pcs, 5)):
        eta_rows.append({
            "config": tag, "PC": i + 1,
            "explained_var_pct": round(evr[i] * 100, 1),
            "eta2_session": round(_eta_sq(scores[:, i], idx["session"]), 3),
            "eta2_mean_level_bin": round(_eta_sq(scores[:, i], idx["mean_level_bin"]), 3),
        })
    eta_df = pd.DataFrame(eta_rows)
    print(eta_df.drop(columns="config").to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(_FW * 1.6, _FW * 0.55))

    ax = axes[0]
    for region, g in idx.groupby("region"):
        ax.scatter(g["tsne_x"], g["tsne_y"], s=10, alpha=0.7,
                   color=REGION_COLORS.get(region, "#7f7f7f"), label=region)
    ax.set_title("Coloured by Region")
    ax.legend(loc="best", fontsize=8)

    ax = axes[1]
    sessions = sorted(idx["session"].astype(str).unique())
    pal = plt.get_cmap("tab20")
    for i, sess in enumerate(sessions):
        g = idx[idx["session"].astype(str) == sess]
        ax.scatter(g["tsne_x"], g["tsne_y"], s=10, alpha=0.7, color=pal(i % 20))
    ax.set_title(f"Coloured by Session ({len(sessions)} sessions, no legend)")

    ax = axes[2]
    bin_order = ["Q1(low)", "Q2", "Q3", "Q4(high)"]
    pal_bin = plt.get_cmap("viridis")
    for i, b in enumerate(bin_order):
        g = idx[idx["mean_level_bin"] == b]
        if g.empty:
            continue
        ax.scatter(g["tsne_x"], g["tsne_y"], s=10, alpha=0.7,
                  color=pal_bin(i / max(1, len(bin_order) - 1)), label=b)
    ax.set_title("Coloured by Mean-Level Bin")
    ax.legend(loc="best", fontsize=8)

    for ax in axes:
        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")
    fig.suptitle(f"PCA(n={n_pcs}) -> t-SNE  -  variant={tag}", y=1.03)
    plt.tight_layout()
    plt.show()

    p_scores = DIR_TABLES13C / f"tsne_scores_{tag}_npcs{n_pcs}.csv"
    idx.to_csv(p_scores, index=False)
    p_fig = DIR_FIGURES13C / f"tsne_confound_{tag}_npcs{n_pcs}.png"
    fig.savefig(p_fig, dpi=150, bbox_inches="tight")
    print(f"saved -> {p_scores}\nsaved -> {p_fig}\n")

    return idx, eta_df


# --- Run all configs ---
all_idx13c, all_eta13c = {}, []
for tag, folder_name in CELL13C_CONFIGS.items():
    idx_c, eta_c = run_config13c(tag, folder_name)
    if idx_c is not None:
        all_idx13c[tag] = idx_c
        all_eta13c.append(eta_c)

if all_eta13c:
    eta_summary13c = pd.concat(all_eta13c, ignore_index=True)
    p_eta_summary = DIR_TABLES13C / f"tsne_confound_eta2_ALL_CONFIGS_npcs{CELL13C_NPCS}.csv"
    eta_summary13c.to_csv(p_eta_summary, index=False)
    print(f"saved -> {p_eta_summary}")
    print("\n=== PC1 eta^2 across configs (brightness vs. session confound) ===")
    print(eta_summary13c[eta_summary13c["PC"] == 1]
          .drop(columns="PC").to_string(index=False))

# --- Combined overview grid ---
if len(all_idx13c) > 1:
    cfg_order = [c for c in CELL13C_CONFIGS if c in all_idx13c]
    fig_grid, axes_grid = plt.subplots(len(cfg_order), 3,
                                       figsize=(_FW * 1.7, _FW * 0.55 * len(cfg_order)))
    bin_order = ["Q1(low)", "Q2", "Q3", "Q4(high)"]
    pal_bin = plt.get_cmap("viridis")
    for r, tag in enumerate(cfg_order):
        idx = all_idx13c[tag]
        ax = axes_grid[r, 0]
        for region, g in idx.groupby("region"):
            ax.scatter(g["tsne_x"], g["tsne_y"], s=6, alpha=0.6,
                      color=REGION_COLORS.get(region, "#7f7f7f"))
        ax.set_ylabel(tag, fontsize=10)
        if r == 0:
            ax.set_title("Region")
        ax = axes_grid[r, 1]
        sessions = sorted(idx["session"].astype(str).unique())
        pal = plt.get_cmap("tab20")
        for i, sess in enumerate(sessions):
            g = idx[idx["session"].astype(str) == sess]
            ax.scatter(g["tsne_x"], g["tsne_y"], s=6, alpha=0.6, color=pal(i % 20))
        if r == 0:
            ax.set_title("Session")
        ax = axes_grid[r, 2]
        for i, b in enumerate(bin_order):
            g = idx[idx["mean_level_bin"] == b]
            if g.empty:
                continue
            ax.scatter(g["tsne_x"], g["tsne_y"], s=6, alpha=0.6,
                      color=pal_bin(i / max(1, len(bin_order) - 1)))
        if r == 0:
            ax.set_title("Mean-Level Bin")
        for ax in axes_grid[r]:
            ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()
    p_grid = DIR_FIGURES13C / f"tsne_confound_ALL_CONFIGS_grid_npcs{CELL13C_NPCS}.png"
    fig_grid.savefig(p_grid, dpi=150, bbox_inches="tight")
    print(f"saved -> {p_grid}")

In [ ]:
# === Cell 17: t-SNE Silhouette and Hopkins Check ===
# Standalone: reads the tsne_scores_{tag}_npcs20.csv tables from the t-SNE sweep and
# runs KMeans (k=2..6) silhouette and Hopkins on tsne_x/tsne_y per region and config.
# t-SNE distances are not metric, so this checks cluster tightness, not significance.

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

RESULTS_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/KI_Sort_Preprocessing/"
                    "results/20260901_070019_hsi_preprocessing_v2")
DIR_TABLES13D = RESULTS_ROOT / "tables"

CELL13D_CONFIGS = ("raw", "iarr", "snv", "iarr_snv")
CELL13D_NPCS_TAG = 20            # must match the npcs{N} suffix written by the t-SNE sweep
CELL13D_REGIONS = ("body", "cap", "label")
CELL13D_RANDOM_STATE = 42


def hopkins(X, n_samples=None, seed=42):
    rng = np.random.RandomState(seed)
    n = X.shape[0]
    if n_samples is None:
        n_samples = min(n - 1, max(5, n // 5))
    d = X.shape[1]
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    idx = rng.choice(n, n_samples, replace=False)
    dist_real, _ = nn.kneighbors(X[idx], n_neighbors=2)
    u_real = dist_real[:, 1]
    mins, maxs = X.min(axis=0), X.max(axis=0)
    rand_pts = rng.uniform(mins, maxs, size=(n_samples, d))
    dist_rand, _ = nn.kneighbors(rand_pts, n_neighbors=1)
    u_rand = dist_rand[:, 0]
    return u_rand.sum() / (u_rand.sum() + u_real.sum())


rows13d = []
for tag in CELL13D_CONFIGS:
    p = DIR_TABLES13D / f"tsne_scores_{tag}_npcs{CELL13D_NPCS_TAG}.csv"
    if not p.exists():
        print(f"!! [{tag}] {p.name} not found - run Cell 16 first for this config.")
        continue
    df = pd.read_csv(p)

    for region in CELL13D_REGIONS:
        sub = df[df["region"] == region].dropna(subset=["tsne_x", "tsne_y"])
        X = sub[["tsne_x", "tsne_y"]].to_numpy()
        n = X.shape[0]
        if n < 10:
            continue
        Xs = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)

        H = np.mean([hopkins(Xs, seed=s) for s in range(5)])

        best_k, best_sil, sil_by_k = None, -1, {}
        for k in range(2, 7):
            if n <= k:
                continue
            km = KMeans(n_clusters=k, n_init=10, random_state=CELL13D_RANDOM_STATE).fit(Xs)
            sil = silhouette_score(Xs, km.labels_)
            sil_by_k[k] = sil
            if sil > best_sil:
                best_sil, best_k = sil, k

        rows13d.append({
            "config": tag, "region": region, "n": n,
            "hopkins_tsne": round(H, 3),
            "best_k_tsne": best_k, "best_silhouette_tsne": round(best_sil, 3),
            "sil_k2_tsne": round(sil_by_k.get(2, np.nan), 3),
            "sil_k3_tsne": round(sil_by_k.get(3, np.nan), 3),
        })

res13d = pd.DataFrame(rows13d)
pd.set_option("display.width", 160)
print(res13d.to_string(index=False))

p_out13d = DIR_TABLES13D / "tsne_cluster_validation_summary.csv"
res13d.to_csv(p_out13d, index=False)
print(f"\nsaved -> {p_out13d}")

# --- Compare against PCA(3) baseline ---
# Best silhouette per config/region from an earlier KMeans sweep in PC1-3 space
# (cluster_summary.csv), hardcoded; update if that sweep is rerun.
pca3_baseline = pd.DataFrame([
    {"config": "raw", "region": "body", "best_silhouette_pca3": 0.540},
    {"config": "raw", "region": "cap", "best_silhouette_pca3": 0.389},
    {"config": "raw", "region": "label", "best_silhouette_pca3": 0.360},
    {"config": "iarr", "region": "body", "best_silhouette_pca3": 0.540},
    {"config": "iarr", "region": "cap", "best_silhouette_pca3": 0.388},
    {"config": "iarr", "region": "label", "best_silhouette_pca3": 0.358},
    {"config": "snv", "region": "body", "best_silhouette_pca3": 0.487},
    {"config": "snv", "region": "cap", "best_silhouette_pca3": 0.298},
    {"config": "snv", "region": "label", "best_silhouette_pca3": 0.379},
    {"config": "iarr_snv", "region": "body", "best_silhouette_pca3": 0.427},
    {"config": "iarr_snv", "region": "cap", "best_silhouette_pca3": 0.421},
    {"config": "iarr_snv", "region": "label", "best_silhouette_pca3": 0.492},
])
compare13d = res13d.merge(pca3_baseline, on=["config", "region"], how="left")
compare13d["sil_delta_tsne_minus_pca3"] = (
    compare13d["best_silhouette_tsne"] - compare13d["best_silhouette_pca3"]).round(3)
print("\n=== t-SNE(20 PCs) vs. PCA(3-PC) silhouette comparison ===")
print(compare13d[["config", "region", "best_silhouette_pca3", "best_silhouette_tsne",
                  "sil_delta_tsne_minus_pca3"]].to_string(index=False))

p_compare13d = DIR_TABLES13D / "tsne_vs_pca3_silhouette_comparison.csv"
compare13d.to_csv(p_compare13d, index=False)
print(f"\nsaved -> {p_compare13d}")

In [ ]:
# === Cell 18: PCA and t-SNE Cluster Report ===
# Standalone: reads the true-GT spectra exports of each preprocessing config, then per
# config:
#   1. PCA to CELL14_N_PCS on the objects x bands matrix
#   2. t-SNE on the first CELL14_N_PCS_TSNE principal components
#   3. Hopkins and a full silhouette k-sweep (k=2..CELL14_K_MAX), per region and pooled,
#      in PCA (PC1-3) and t-SNE space; descriptive scores, not significance tests
#   4. per-PC eta^2 against session and brightness quartile (PCA scores only, since
#      t-SNE coordinates are not metric)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D            # noqa: F401  (registers "3d")
from pathlib import Path
from sklearn.decomposition import PCA as _SkPCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

# --- Plot style ---
_FW = 16 / 2.54
_FONT_SIZE = 10
_PREFERRED_FONTS = ["DejaVu Sans", "Helvetica", "Arial", "Nimbus Sans",
                    "TeX Gyre Heros", "Liberation Sans"]
_available14 = {f.name for f in fm.fontManager.ttflist}
_font14 = next((f for f in _PREFERRED_FONTS if f in _available14), "DejaVu Sans")
plt.rcParams["font.family"] = _font14
plt.rcParams["font.size"] = _FONT_SIZE
plt.rcParams["axes.titlesize"] = _FONT_SIZE + 1
plt.rcParams["axes.labelsize"] = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["xtick.labelsize"] = _FONT_SIZE - 1
plt.rcParams["ytick.labelsize"] = _FONT_SIZE - 1
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["legend.frameon"] = False

REGION_COLORS = {"body": "#FF0000", "cap": "#00FF00", "label": "#0000FF"}

# --- Config ---
DATA_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/KI_Sort_Preprocessing/data")
RESULTS_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/KI_Sort_Preprocessing/"
                    "results/20260901_070019_hsi_preprocessing_v2")
DIR_FIGURES14 = RESULTS_ROOT / "figures"
DIR_TABLES14 = RESULTS_ROOT / "tables"

CELL14_CONFIGS = {
    "raw":      "preprocessed_spectra_raw_20260901_071026",
    "iarr":     "preprocessed_spectra_iarr_20260901_071448",
    "snv":      "preprocessed_spectra_snv_20260901_071701",
    "iarr_snv": "preprocessed_spectra_iarr_snv_20260901_071926",
}
CELL14_REGIONS = ("body", "cap", "label")

CELL14_N_PCS = 20              # components retained by PCA
CELL14_N_PCS_TSNE = 20         # leading components handed to t-SNE (<= CELL14_N_PCS)
CELL14_N_PCS_REPORT = 5        # PCs listed per-PC in the eta^2 table
CELL14_K_MAX = 6               # silhouette sweep runs k=2..CELL14_K_MAX
CELL14_PERPLEXITY = 30
CELL14_RANDOM_STATE = 42
CELL14_HOPKINS_REPEATS = 5     # Hopkins is subsample-based; averaged over this many seeds

DIR_FIGURES14.mkdir(parents=True, exist_ok=True)
DIR_TABLES14.mkdir(parents=True, exist_ok=True)


# --- Helpers ---

def load_spectra_matrix14(folder, regions):
    """(objects x bands) matrix + aligned metadata frame, read from the export folder.

    Files are '<cutout_id>_<region>', two columns 'wavelength_nm;value', no header.
    The band grid of the first file defines the columns; files with a different grid
    are skipped.
    """
    meta_path = folder / "true_gt_spectra_meta.csv"
    if not meta_path.exists():
        return None, None, None, [f"meta CSV absent: {meta_path}"]

    meta = pd.read_csv(meta_path)
    meta = meta[meta["region"].isin(regions)].reset_index(drop=True)

    rows, wl, skipped, keep_rows = [], None, [], []
    for i, r in meta.iterrows():
        fpath = folder / f"{r['cutout_id']}_{r['region']}"
        if not fpath.exists():
            skipped.append(f"{fpath.name} (absent)")
            continue
        sp = pd.read_csv(fpath, sep=";", header=None, names=["wavelength_nm", "value"])
        wl_this = sp["wavelength_nm"].to_numpy()
        if wl is None:
            wl = wl_this
        elif wl_this.shape != wl.shape or not np.allclose(wl, wl_this):
            skipped.append(f"{fpath.name} (band grid mismatch)")
            continue
        rows.append(sp["value"].to_numpy())
        keep_rows.append(i)

    if not rows:
        return None, None, None, skipped or ["no spectrum files found"]

    M = np.vstack(rows)
    idx = meta.loc[keep_rows].reset_index(drop=True)

    finite = ~np.isnan(M).any(axis=1)
    if (~finite).any():
        skipped.append(f"{int((~finite).sum())} spectra dropped for NaN bands")
    return M[finite], idx[finite].reset_index(drop=True), wl, skipped


def hopkins14(X, seed):
    """Hopkins statistic: ~0.5 for uniformly distributed data, towards 1 for clustering
    tendency."""
    rng = np.random.RandomState(seed)
    n, d = X.shape
    n_samples = min(n - 1, max(5, n // 5))
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    take = rng.choice(n, n_samples, replace=False)
    d_real, _ = nn.kneighbors(X[take], n_neighbors=2)
    u_real = d_real[:, 1]                       # nearest neighbour excluding self
    rand_pts = rng.uniform(X.min(axis=0), X.max(axis=0), size=(n_samples, d))
    d_rand, _ = nn.kneighbors(rand_pts, n_neighbors=1)
    u_rand = d_rand[:, 0]
    denom = u_rand.sum() + u_real.sum()
    return u_rand.sum() / denom if denom > 0 else np.nan


def cluster_profile14(X, k_max, random_state, hopkins_repeats):
    """Hopkins plus a KMeans silhouette sweep over k.

    Returns (hopkins, sweep, k_argmax, silhouette_at_argmax, labels_at_argmax).
    """
    n = X.shape[0]
    Xs = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)
    H = float(np.mean([hopkins14(Xs, seed=s) for s in range(hopkins_repeats)]))

    sweep, labels_by_k = {}, {}
    for k in range(2, k_max + 1):
        if n <= k:
            continue
        km = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(Xs)
        sweep[k] = float(silhouette_score(Xs, km.labels_))
        labels_by_k[k] = km.labels_
    if not sweep:
        return H, {}, None, np.nan, None
    k_arg = max(sweep, key=sweep.get)
    return H, sweep, k_arg, sweep[k_arg], labels_by_k[k_arg]


def eta_sq14(values, groups):
    """One-way ANOVA eta^2: share of the variance of `values` explained by `groups`."""
    df_ = pd.DataFrame({"v": np.asarray(values, dtype=float), "g": np.asarray(groups)})
    grand = df_["v"].mean()
    ss_tot = ((df_["v"] - grand) ** 2).sum()
    if ss_tot <= 0:
        return np.nan
    ss_between = df_.groupby("g")["v"].apply(
        lambda s: len(s) * (s.mean() - grand) ** 2).sum()
    return float(ss_between / ss_tot)


# --- Per-config pipeline ---

def process_config14(tag, folder_name):
    folder = DATA_ROOT / folder_name
    M, idx, wl, skipped = load_spectra_matrix14(folder, CELL14_REGIONS)
    if M is None:
        print(f"!! [{tag}] {skipped[0]} - skipping this config.")
        return None
    if skipped:
        print(f"!! [{tag}] {len(skipped)} note(s): {skipped[:3]}")

    n_pcs = int(min(CELL14_N_PCS, M.shape[0] - 1, M.shape[1]))
    pca = _SkPCA(n_components=n_pcs, random_state=CELL14_RANDOM_STATE)
    scores = pca.fit_transform(M - M.mean(axis=0, keepdims=True))
    evr = pca.explained_variance_ratio_

    for c in ("session", "split"):
        if c not in idx.columns:
            idx[c] = ""
    idx["session"] = idx["session"].astype(str)
    idx["mean_level"] = M.mean(axis=1)          # continuous brightness, for colouring
    try:                                        # discrete brightness, for eta^2
        idx["mean_level_bin"] = pd.qcut(
            idx["mean_level"], 4, labels=["Q1(low)", "Q2", "Q3", "Q4(high)"],
            duplicates="drop").astype(str)
    except ValueError:
        idx["mean_level_bin"] = "all"

    n_tsne_in = int(min(CELL14_N_PCS_TSNE, n_pcs))
    perplexity = float(min(CELL14_PERPLEXITY, max(5.0, (M.shape[0] - 1) / 3.0)))
    tsne = TSNE(n_components=2, perplexity=perplexity, init="pca",
                random_state=CELL14_RANDOM_STATE)
    emb = tsne.fit_transform(scores[:, :n_tsne_in])
    idx["tsne_x"], idx["tsne_y"] = emb[:, 0], emb[:, 1]

    for i in range(n_pcs):
        idx[f"PC{i + 1}"] = scores[:, i]

    cum3 = float(evr[:min(3, n_pcs)].sum() * 100)
    print(f"[{tag}] objects={M.shape[0]:,} bands={M.shape[1]} PCs={n_pcs} "
          f"tsne_in={n_tsne_in} perplexity={perplexity:g}")
    print(f"[{tag}] explained variance: "
          + "  ".join(f"PC{i + 1}={evr[i] * 100:.1f}%"
                      for i in range(min(CELL14_N_PCS_REPORT, n_pcs)))
          + f"   cumulative(PC1-{min(3, n_pcs)})={cum3:.1f}%"
          + f"   cumulative(all {n_pcs})={evr.sum() * 100:.1f}%")

    # --- Confound: PC variance explained by session / brightness ---
    eta_rows = []
    for i in range(min(CELL14_N_PCS_REPORT, n_pcs)):
        eta_rows.append({
            "config": tag, "PC": i + 1,
            "explained_var_pct": round(float(evr[i] * 100), 2),
            "eta2_session": round(eta_sq14(scores[:, i], idx["session"]), 3),
            "eta2_mean_level_bin": round(eta_sq14(scores[:, i], idx["mean_level_bin"]), 3),
        })
    eta_df = pd.DataFrame(eta_rows)
    print(eta_df.drop(columns="config").to_string(index=False))

    # --- Cluster tendency, per region and pooled, in both spaces ---
    metric_rows, pooled_tsne_labels = [], None
    scopes = [(r, idx["region"] == r) for r in CELL14_REGIONS] + \
             [("full", pd.Series(True, index=idx.index))]
    for scope_name, mask in scopes:
        m = mask.to_numpy()
        if m.sum() < 10:
            continue
        for space, X in (("pca", scores[m, :min(3, n_pcs)]),
                         ("tsne", emb[m])):
            H, sweep, k_arg, sil_arg, labels = cluster_profile14(
                X, CELL14_K_MAX, CELL14_RANDOM_STATE, CELL14_HOPKINS_REPEATS)
            row = {"config": tag, "scope": scope_name, "space": space,
                   "n": int(m.sum()), "hopkins": round(H, 3),
                   "k_argmax_silhouette": k_arg,
                   "silhouette_at_argmax": round(sil_arg, 3) if sweep else np.nan}
            for k in range(2, CELL14_K_MAX + 1):
                row[f"silhouette_k{k}"] = round(sweep[k], 3) if k in sweep else np.nan
            if labels is not None:
                sizes = np.bincount(labels).tolist()
                row["cluster_sizes_at_argmax"] = "/".join(str(s) for s in sizes)
            metric_rows.append(row)
            if scope_name == "full" and space == "tsne" and labels is not None:
                pooled_tsne_labels = labels
    metrics_df = pd.DataFrame(metric_rows)

    if pooled_tsne_labels is not None:
        idx["tsne_cluster"] = pooled_tsne_labels
    else:
        idx["tsne_cluster"] = -1

    return {"tag": tag, "idx": idx, "scores": scores, "evr": evr, "n_pcs": n_pcs,
            "n_tsne_in": n_tsne_in, "perplexity": perplexity, "n_bands": M.shape[1],
            "eta_df": eta_df, "metrics_df": metrics_df, "cum3": cum3}


# --- Panel painters ---

def _paint_region(ax, idx, three_d=False, size=6):
    for region, g in idx.groupby("region"):
        col = REGION_COLORS.get(region, "#7f7f7f")
        if three_d:
            ax.scatter(g["PC1"], g["PC2"], g["PC3"], s=size, alpha=0.6, color=col,
                       depthshade=False, label=region)
        else:
            ax.scatter(g["tsne_x"], g["tsne_y"], s=size, alpha=0.6, color=col,
                       label=region)


def _paint_session(ax, idx, three_d=False, size=6):
    pal = plt.get_cmap("tab20")
    for i, sess in enumerate(sorted(idx["session"].unique())):
        g = idx[idx["session"] == sess]
        col = pal(i % 20)
        if three_d:
            ax.scatter(g["PC1"], g["PC2"], g["PC3"], s=size, alpha=0.6, color=col,
                       depthshade=False)
        else:
            ax.scatter(g["tsne_x"], g["tsne_y"], s=size, alpha=0.6, color=col)


def _paint_level(ax, idx, three_d=False, size=6):
    """Colour by continuous mean level (eta^2 uses the quartile bin instead)."""
    if three_d:
        sc = ax.scatter(idx["PC1"], idx["PC2"], idx["PC3"], s=size, alpha=0.6,
                        c=idx["mean_level"], cmap="viridis", depthshade=False)
    else:
        sc = ax.scatter(idx["tsne_x"], idx["tsne_y"], s=size, alpha=0.6,
                        c=idx["mean_level"], cmap="viridis")
    return sc


def _paint_clusters(ax, idx, size=6):
    """t-SNE only: KMeans labels at the silhouette argmax of the pooled t-SNE embedding."""
    labels = np.sort(idx["tsne_cluster"].unique())
    pal = plt.get_cmap("tab10")
    handles = []
    for i, lab in enumerate(labels):
        g = idx[idx["tsne_cluster"] == lab]
        col = pal(i % 10)
        ax.scatter(g["tsne_x"], g["tsne_y"], s=size, alpha=0.6, color=col)
        handles.append(Line2D([0], [0], marker="o", linestyle="", markersize=4,
                              color=col, label=f"c{lab} (n={len(g)})"))
    return handles


# --- Run all configs ---

results14, eta_all14, metrics_all14 = {}, [], []
for tag, folder_name in CELL14_CONFIGS.items():
    out = process_config14(tag, folder_name)
    if out is not None:
        results14[tag] = out
        eta_all14.append(out["eta_df"])
        metrics_all14.append(out["metrics_df"])
    print()

if not results14:
    print("!! No config produced results - check DATA_ROOT and the folder names.")
else:
    cfg_order14 = [c for c in CELL14_CONFIGS if c in results14]
    n_rows14 = len(cfg_order14)

    # --- Figure 1: t-SNE grid ---
    fig_t, axes_t = plt.subplots(n_rows14, 4,
                                 figsize=(_FW * 2.2, _FW * 0.58 * n_rows14),
                                 squeeze=False)
    for r, tag in enumerate(cfg_order14):
        res = results14[tag]
        idx = res["idx"]

        ax = axes_t[r][0]
        _paint_region(ax, idx)
        ax.set_ylabel(f"{tag}\nt-SNE 2", fontsize=_FONT_SIZE)
        if r == 0:
            ax.set_title("Region")
        ax.legend(loc="best", markerscale=1.6, fontsize=_FONT_SIZE - 2)

        ax = axes_t[r][1]
        _paint_session(ax, idx)
        if r == 0:
            ax.set_title("Session")
        ax.text(0.02, 0.98, f"{idx['session'].nunique()} sessions", va="top",
                ha="left", transform=ax.transAxes, fontsize=_FONT_SIZE - 2)

        ax = axes_t[r][2]
        sc = _paint_level(ax, idx)
        if r == 0:
            ax.set_title("Mean Level")
        cb = fig_t.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
        cb.set_label("Mean Level", fontsize=_FONT_SIZE - 2)
        cb.ax.tick_params(labelsize=_FONT_SIZE - 3)

        ax = axes_t[r][3]
        handles = _paint_clusters(ax, idx)
        if r == 0:
            ax.set_title("k-Means Clusters")
        ax.legend(handles=handles, loc="best", fontsize=_FONT_SIZE - 3)
        sizes = idx["tsne_cluster"].value_counts().sort_index().to_list()
        ax.text(0.02, 0.02, "n: " + "/".join(str(s) for s in sizes), va="bottom",
                ha="left", transform=ax.transAxes, fontsize=_FONT_SIZE - 2)

        for c in range(4):
            axes_t[r][c].set_xlabel("t-SNE 1")
            if c > 0:
                axes_t[r][c].set_ylabel("t-SNE 2")
            axes_t[r][c].set_xticks([])
            axes_t[r][c].set_yticks([])
        axes_t[r][0].set_ylabel(f"{tag}\nt-SNE 2", fontsize=_FONT_SIZE)

    fig_t.suptitle(
        f"t-SNE on {results14[cfg_order14[0]]['n_tsne_in']} Principal Components "
        f"(perplexity {results14[cfg_order14[0]]['perplexity']:g})", y=1.005)
    fig_t.tight_layout()
    plt.show()
    p_fig_t = (DIR_FIGURES14 /
               f"tsne_grid_ALL_CONFIGS_npcs{CELL14_N_PCS}_tsnein{CELL14_N_PCS_TSNE}.png")
    fig_t.savefig(p_fig_t, dpi=150, bbox_inches="tight")

    # --- Figure 2: 3D PCA grid ---
    fig_p = plt.figure(figsize=(_FW * 1.9, _FW * 0.62 * n_rows14))
    for r, tag in enumerate(cfg_order14):
        res = results14[tag]
        idx, evr = res["idx"], res["evr"]
        titles = ("Region", "Session", "Mean Level")
        for c in range(3):
            ax = fig_p.add_subplot(n_rows14, 3, r * 3 + c + 1, projection="3d")
            if c == 0:
                _paint_region(ax, idx, three_d=True)
                ax.legend(loc="upper left", markerscale=1.6, fontsize=_FONT_SIZE - 3)
            elif c == 1:
                _paint_session(ax, idx, three_d=True)
                ax.text2D(0.02, 0.96, f"{idx['session'].nunique()} sessions",
                          transform=ax.transAxes, fontsize=_FONT_SIZE - 2)
            else:
                sc = _paint_level(ax, idx, three_d=True)
                cb = fig_p.colorbar(sc, ax=ax, fraction=0.03, pad=0.10)
                cb.set_label("Mean Level", fontsize=_FONT_SIZE - 2)
                cb.ax.tick_params(labelsize=_FONT_SIZE - 3)

            ax.set_xlabel(f"PC1 ({evr[0] * 100:.1f}%)", fontsize=_FONT_SIZE - 2)
            ax.set_ylabel(f"PC2 ({evr[1] * 100:.1f}%)" if res["n_pcs"] > 1 else "PC2",
                          fontsize=_FONT_SIZE - 2)
            ax.set_zlabel(f"PC3 ({evr[2] * 100:.1f}%)" if res["n_pcs"] > 2 else "PC3",
                          fontsize=_FONT_SIZE - 2)
            ax.tick_params(labelsize=_FONT_SIZE - 4)
            if r == 0:
                ax.set_title(titles[c], pad=12)
            if c == 0:
                ax.text2D(-0.18, 0.5, f"{tag}\ncum. PC1-3 = {res['cum3']:.1f}%",
                          transform=ax.transAxes, rotation=90, va="center",
                          ha="center", fontsize=_FONT_SIZE)

    fig_p.suptitle("Principal Component Scores, First Three Components", y=1.003)
    fig_p.tight_layout()
    plt.show()
    p_fig_p = DIR_FIGURES14 / f"pca3d_grid_ALL_CONFIGS_npcs{CELL14_N_PCS}.png"
    fig_p.savefig(p_fig_p, dpi=150, bbox_inches="tight")

    # --- Tables ---
    eta_out14 = pd.concat(eta_all14, ignore_index=True)
    metrics_out14 = pd.concat(metrics_all14, ignore_index=True)

    run_meta14 = pd.DataFrame([{
        "config": r["tag"], "n_objects": len(r["idx"]), "n_bands": r["n_bands"],
        "n_pcs": r["n_pcs"], "n_pcs_into_tsne": r["n_tsne_in"],
        "perplexity": r["perplexity"], "random_state": CELL14_RANDOM_STATE,
        "k_sweep_max": CELL14_K_MAX,
        "cum_var_pc1_3_pct": round(r["cum3"], 2),
        "cum_var_all_pcs_pct": round(float(r["evr"].sum() * 100), 2),
        "source_folder": str(DATA_ROOT / CELL14_CONFIGS[r["tag"]]),
    } for r in results14.values()])

    p_eta14 = DIR_TABLES14 / f"pca_tsne_confound_eta2_npcs{CELL14_N_PCS}.csv"
    p_metrics14 = DIR_TABLES14 / f"pca_tsne_cluster_metrics_npcs{CELL14_N_PCS}.csv"
    p_runmeta14 = DIR_TABLES14 / f"pca_tsne_run_metadata_npcs{CELL14_N_PCS}.csv"
    eta_out14.to_csv(p_eta14, index=False)
    metrics_out14.to_csv(p_metrics14, index=False)
    run_meta14.to_csv(p_runmeta14, index=False)

    for r in results14.values():
        keep = ["cutout_id", "region", "session", "split", "mean_level",
                "mean_level_bin", "tsne_x", "tsne_y", "tsne_cluster"] + \
               [f"PC{i + 1}" for i in range(r["n_pcs"])]
        keep = [c for c in keep if c in r["idx"].columns]
        p_sc = DIR_TABLES14 / f"pca_tsne_scores_{r['tag']}_npcs{CELL14_N_PCS}.csv"
        r["idx"][keep].to_csv(p_sc, index=False)
        print(f"saved -> {p_sc}")

    print(f"saved -> {p_fig_t}")
    print(f"saved -> {p_fig_p}")
    print(f"saved -> {p_eta14}")
    print(f"saved -> {p_metrics14}")
    print(f"saved -> {p_runmeta14}")

    print("\n=== PC1 confound summary across configs ===")
    print(eta_out14[eta_out14["PC"] == 1].drop(columns="PC").to_string(index=False))
    print("\n=== Cluster tendency, pooled scope ===")
    print(metrics_out14[metrics_out14["scope"] == "full"][
        ["config", "space", "n", "hopkins", "k_argmax_silhouette",
         "silhouette_at_argmax", "cluster_sizes_at_argmax"]].to_string(index=False))

In [ ]:
# === Cell 19: UMAP Stability Grid ===
# Standalone: UMAP embeddings of one preprocessing config (CELL14B_CONFIG) over an
# n_neighbors x min_dist grid, to check how the embedding shape depends on these
# hyperparameters. Points are coloured by region.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path
from sklearn.decomposition import PCA as _SkPCA

try:
    import umap
except ImportError:
    umap = None

# --- Plot style ---
_FW = 16 / 2.54
_FONT_SIZE = 10
_PREFERRED_FONTS = ["DejaVu Sans", "Helvetica", "Arial", "Nimbus Sans",
                    "TeX Gyre Heros", "Liberation Sans"]
_available14b = {f.name for f in fm.fontManager.ttflist}
_font14b = next((f for f in _PREFERRED_FONTS if f in _available14b), "DejaVu Sans")
plt.rcParams["font.family"] = _font14b
plt.rcParams["font.size"] = _FONT_SIZE
plt.rcParams["axes.titlesize"] = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 2
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["legend.frameon"] = False

REGION_COLORS = {"body": "#FF0000", "cap": "#00FF00", "label": "#0000FF"}

# --- Config ---
DATA_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/KI_Sort_Preprocessing/data")
RESULTS_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/KI_Sort_Preprocessing/"
                    "results/20260901_070019_hsi_preprocessing_v2")
DIR_FIGURES14B = RESULTS_ROOT / "figures"
DIR_TABLES14B = RESULTS_ROOT / "tables"
DIR_FIGURES14B.mkdir(parents=True, exist_ok=True)
DIR_TABLES14B.mkdir(parents=True, exist_ok=True)

CELL14B_CONFIGS = {                # same folders as the PCA and t-SNE cluster report
    "raw":      "preprocessed_spectra_raw_20260901_071026",
    "iarr":     "preprocessed_spectra_iarr_20260901_071448",
    "snv":      "preprocessed_spectra_snv_20260901_071701",
    "iarr_snv": "preprocessed_spectra_iarr_snv_20260901_071926",
}
CELL14B_CONFIG = "snv"              # which single config this run investigates
CELL14B_REGIONS = ("body", "cap", "label")
CELL14B_N_PCS = 20                  # PCA components computed upstream of UMAP
CELL14B_N_PCS_UMAP = 20             # leading PCs actually fed into UMAP

CELL14B_N_NEIGHBORS_GRID = (15, 30, 50, 100)
CELL14B_MIN_DIST_GRID = (0.0, 0.1, 0.3, 0.5)
CELL14B_METRIC = "euclidean"        # matches PCA/t-SNE/Hopkins/silhouette elsewhere
CELL14B_RANDOM_STATE = 42


# --- Load spectra and PCA (same logic as the PCA and t-SNE cluster report) ---

def load_spectra_matrix14b(folder, regions):
    meta_path = folder / "true_gt_spectra_meta.csv"
    if not meta_path.exists():
        return None, None, None, [f"meta CSV absent: {meta_path}"]
    meta = pd.read_csv(meta_path)
    meta = meta[meta["region"].isin(regions)].reset_index(drop=True)

    rows, wl, skipped, keep_rows = [], None, [], []
    for i, r in meta.iterrows():
        fpath = folder / f"{r['cutout_id']}_{r['region']}"
        if not fpath.exists():
            skipped.append(f"{fpath.name} (absent)")
            continue
        sp = pd.read_csv(fpath, sep=";", header=None, names=["wavelength_nm", "value"])
        wl_this = sp["wavelength_nm"].to_numpy()
        if wl is None:
            wl = wl_this
        elif wl_this.shape != wl.shape or not np.allclose(wl, wl_this):
            skipped.append(f"{fpath.name} (band grid mismatch)")
            continue
        rows.append(sp["value"].to_numpy())
        keep_rows.append(i)

    if not rows:
        return None, None, None, skipped or ["no spectrum files found"]

    M = np.vstack(rows)
    idx = meta.loc[keep_rows].reset_index(drop=True)
    finite = ~np.isnan(M).any(axis=1)
    if (~finite).any():
        skipped.append(f"{int((~finite).sum())} spectra dropped for NaN bands")
    return M[finite], idx[finite].reset_index(drop=True), wl, skipped


if umap is None:
    print("!! umap-learn not installed in this environment. "
          "`pip install umap-learn --break-system-packages` and rerun this cell.")
else:
    folder14b = DATA_ROOT / CELL14B_CONFIGS[CELL14B_CONFIG]
    M14b, idx14b, wl14b, skipped14b = load_spectra_matrix14b(folder14b, CELL14B_REGIONS)

    if M14b is None:
        print(f"!! [{CELL14B_CONFIG}] {skipped14b[0]} - aborting.")
    else:
        if skipped14b:
            print(f"!! [{CELL14B_CONFIG}] {len(skipped14b)} note(s): {skipped14b[:3]}")

        n_pcs14b = int(min(CELL14B_N_PCS, M14b.shape[0] - 1, M14b.shape[1]))
        pca14b = _SkPCA(n_components=n_pcs14b, random_state=CELL14B_RANDOM_STATE)
        scores14b = pca14b.fit_transform(M14b - M14b.mean(axis=0, keepdims=True))
        evr14b = pca14b.explained_variance_ratio_
        n_umap_in14b = int(min(CELL14B_N_PCS_UMAP, n_pcs14b))

        print(f"[{CELL14B_CONFIG}] objects={M14b.shape[0]:,}  bands={M14b.shape[1]}  "
              f"PCs computed={n_pcs14b}  PCs into UMAP={n_umap_in14b}  "
              f"cumulative variance (PCs into UMAP)="
              f"{evr14b[:n_umap_in14b].sum() * 100:.1f}%")

        n_rows14b = len(CELL14B_N_NEIGHBORS_GRID)
        n_cols14b = len(CELL14B_MIN_DIST_GRID)
        fig14b, axes14b = plt.subplots(
            n_rows14b, n_cols14b,
            figsize=(_FW * 0.62 * n_cols14b, _FW * 0.62 * n_rows14b), squeeze=False)

        grid_rows14b = []
        embeddings14b = {}
        for ri, n_neighbors in enumerate(CELL14B_N_NEIGHBORS_GRID):
            for ci, min_dist in enumerate(CELL14B_MIN_DIST_GRID):
                reducer = umap.UMAP(
                    n_neighbors=min(n_neighbors, M14b.shape[0] - 1),
                    min_dist=min_dist, n_components=2, metric=CELL14B_METRIC,
                    random_state=CELL14B_RANDOM_STATE)
                emb = reducer.fit_transform(scores14b[:, :n_umap_in14b])
                embeddings14b[(n_neighbors, min_dist)] = emb

                ax = axes14b[ri][ci]
                for region, g_idx in idx14b.groupby("region"):
                    m = g_idx.index.to_numpy()
                    ax.scatter(emb[m, 0], emb[m, 1], s=4, alpha=0.55,
                              color=REGION_COLORS.get(region, "#7f7f7f"))
                ax.set_xticks([]); ax.set_yticks([])
                if ri == 0:
                    ax.set_title(f"min_dist={min_dist:g}", fontsize=_FONT_SIZE - 1)
                if ci == 0:
                    ax.set_ylabel(f"n_neighbors={n_neighbors}", fontsize=_FONT_SIZE - 1)

                grid_rows14b.append({
                    "config": CELL14B_CONFIG, "n_neighbors": n_neighbors,
                    "min_dist": min_dist, "n_objects": M14b.shape[0],
                })

        handles = [plt.Line2D([0], [0], marker="o", linestyle="", markersize=6,
                              color=REGION_COLORS[r], label=r)
                  for r in CELL14B_REGIONS]
        fig14b.legend(handles=handles, loc="upper center", ncol=3,
                      bbox_to_anchor=(0.5, 1.02), frameon=False)
        fig14b.suptitle(
            f"UMAP Stability Grid - config={CELL14B_CONFIG}  "
            f"(PCA->UMAP on {n_umap_in14b} PCs, metric={CELL14B_METRIC})", y=1.05)
        fig14b.tight_layout()
        plt.show()

        p_fig14b = (DIR_FIGURES14B /
                   f"umap_stability_grid_{CELL14B_CONFIG}_npcs{n_umap_in14b}.png")
        fig14b.savefig(p_fig14b, dpi=150, bbox_inches="tight")

        # --- Long-format coordinates for every grid cell ---
        coord_rows14b = []
        base_cols = ["cutout_id", "region", "session", "split"]
        base_cols = [c for c in base_cols if c in idx14b.columns]
        for (n_neighbors, min_dist), emb in embeddings14b.items():
            cell_df = idx14b[base_cols].copy()
            cell_df["n_neighbors"] = n_neighbors
            cell_df["min_dist"] = min_dist
            cell_df["umap_x"] = emb[:, 0]
            cell_df["umap_y"] = emb[:, 1]
            coord_rows14b.append(cell_df)
        coords_out14b = pd.concat(coord_rows14b, ignore_index=True)

        p_coords14b = (DIR_TABLES14B /
                       f"umap_stability_grid_coords_{CELL14B_CONFIG}_npcs{n_umap_in14b}.csv")
        p_gridmeta14b = (DIR_TABLES14B /
                         f"umap_stability_grid_metadata_{CELL14B_CONFIG}_npcs{n_umap_in14b}.csv")
        coords_out14b.to_csv(p_coords14b, index=False)
        pd.DataFrame(grid_rows14b).to_csv(p_gridmeta14b, index=False)

        print(f"saved -> {p_fig14b}")
        print(f"saved -> {p_coords14b}")
        print(f"saved -> {p_gridmeta14b}")

In [ ]:
# === Cell 20: UMAP Metadata-Label Silhouette ===
# Standalone: one UMAP embedding per preprocessing config at a fixed (n_neighbors,
# min_dist) from the stability grid, with Hopkins and silhouette against metadata labels
# (region, session, mean_level_bin) rather than KMeans assignments.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path
from sklearn.decomposition import PCA as _SkPCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

try:
    import umap
except ImportError:
    umap = None

# --- Plot style ---
_FW = 16 / 2.54
_FONT_SIZE = 10
_PREFERRED_FONTS = ["DejaVu Sans", "Helvetica", "Arial", "Nimbus Sans",
                    "TeX Gyre Heros", "Liberation Sans"]
_available14c = {f.name for f in fm.fontManager.ttflist}
_font14c = next((f for f in _PREFERRED_FONTS if f in _available14c), "DejaVu Sans")
plt.rcParams["font.family"] = _font14c
plt.rcParams["font.size"] = _FONT_SIZE
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["legend.frameon"] = False

REGION_COLORS = {"body": "#FF0000", "cap": "#00FF00", "label": "#0000FF"}

# --- Config ---
DATA_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/KI_Sort_Preprocessing/data")
RESULTS_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Spectral_Waste/KI_Sort_Preprocessing/"
                    "results/20260901_070019_hsi_preprocessing_v2")
DIR_FIGURES14C = RESULTS_ROOT / "figures"
DIR_TABLES14C = RESULTS_ROOT / "tables"
DIR_FIGURES14C.mkdir(parents=True, exist_ok=True)
DIR_TABLES14C.mkdir(parents=True, exist_ok=True)

CELL14C_CONFIGS = {
    "raw":      "preprocessed_spectra_raw_20260901_071026",
    "iarr":     "preprocessed_spectra_iarr_20260901_071448",
    "snv":      "preprocessed_spectra_snv_20260901_071701",
    "iarr_snv": "preprocessed_spectra_iarr_snv_20260901_071926",
}
CELL14C_REGIONS = ("body", "cap", "label")
CELL14C_N_PCS = 20
CELL14C_N_PCS_UMAP = 20
CELL14C_N_NEIGHBORS = 30           # fixed, chosen from the UMAP stability grid
CELL14C_MIN_DIST = 0.3             # fixed, chosen from the UMAP stability grid
CELL14C_METRIC = "euclidean"
CELL14C_RANDOM_STATE = 42
CELL14C_LABEL_VARS = ("region", "session", "mean_level_bin")


def load_spectra_matrix14c(folder, regions):
    meta_path = folder / "true_gt_spectra_meta.csv"
    if not meta_path.exists():
        return None, None, None, [f"meta CSV absent: {meta_path}"]
    meta = pd.read_csv(meta_path)
    meta = meta[meta["region"].isin(regions)].reset_index(drop=True)

    rows, wl, skipped, keep_rows = [], None, [], []
    for i, r in meta.iterrows():
        fpath = folder / f"{r['cutout_id']}_{r['region']}"
        if not fpath.exists():
            skipped.append(f"{fpath.name} (absent)")
            continue
        sp = pd.read_csv(fpath, sep=";", header=None, names=["wavelength_nm", "value"])
        wl_this = sp["wavelength_nm"].to_numpy()
        if wl is None:
            wl = wl_this
        elif wl_this.shape != wl.shape or not np.allclose(wl, wl_this):
            skipped.append(f"{fpath.name} (band grid mismatch)")
            continue
        rows.append(sp["value"].to_numpy())
        keep_rows.append(i)

    if not rows:
        return None, None, None, skipped or ["no spectrum files found"]

    M = np.vstack(rows)
    idx = meta.loc[keep_rows].reset_index(drop=True)
    finite = ~np.isnan(M).any(axis=1)
    if (~finite).any():
        skipped.append(f"{int((~finite).sum())} spectra dropped for NaN bands")
    return M[finite], idx[finite].reset_index(drop=True), wl, skipped


def hopkins14c(X, seed):
    rng = np.random.RandomState(seed)
    n, d = X.shape
    n_samples = min(n - 1, max(5, n // 5))
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    take = rng.choice(n, n_samples, replace=False)
    d_real, _ = nn.kneighbors(X[take], n_neighbors=2)
    u_real = d_real[:, 1]
    rand_pts = rng.uniform(X.min(axis=0), X.max(axis=0), size=(n_samples, d))
    d_rand, _ = nn.kneighbors(rand_pts, n_neighbors=1)
    u_rand = d_rand[:, 0]
    denom = u_rand.sum() + u_real.sum()
    return u_rand.sum() / denom if denom > 0 else np.nan


if umap is None:
    print("!! umap-learn not installed in this environment. "
          "`pip install umap-learn --break-system-packages` and rerun this cell.")
else:
    label_rows14c, embeddings_by_config14c = [], {}
    for tag, folder_name in CELL14C_CONFIGS.items():
        folder = DATA_ROOT / folder_name
        M, idx, wl, skipped = load_spectra_matrix14c(folder, CELL14C_REGIONS)
        if M is None:
            print(f"!! [{tag}] {skipped[0]} - skipping this config.")
            continue
        if skipped:
            print(f"!! [{tag}] {len(skipped)} note(s): {skipped[:3]}")

        n_pcs = int(min(CELL14C_N_PCS, M.shape[0] - 1, M.shape[1]))
        pca = _SkPCA(n_components=n_pcs, random_state=CELL14C_RANDOM_STATE)
        scores = pca.fit_transform(M - M.mean(axis=0, keepdims=True))
        n_umap_in = int(min(CELL14C_N_PCS_UMAP, n_pcs))

        for c in ("session", "split"):
            if c not in idx.columns:
                idx[c] = ""
        idx["session"] = idx["session"].astype(str)
        idx["mean_level"] = M.mean(axis=1)
        try:
            idx["mean_level_bin"] = pd.qcut(
                idx["mean_level"], 4, labels=["Q1(low)", "Q2", "Q3", "Q4(high)"],
                duplicates="drop").astype(str)
        except ValueError:
            idx["mean_level_bin"] = "all"

        n_neighbors_eff = min(CELL14C_N_NEIGHBORS, M.shape[0] - 1)
        reducer = umap.UMAP(n_neighbors=n_neighbors_eff, min_dist=CELL14C_MIN_DIST,
                            n_components=2, metric=CELL14C_METRIC,
                            random_state=CELL14C_RANDOM_STATE)
        emb = reducer.fit_transform(scores[:, :n_umap_in])
        idx["umap_x"], idx["umap_y"] = emb[:, 0], emb[:, 1]
        embeddings_by_config14c[tag] = idx.copy()

        Xs = (emb - emb.mean(axis=0)) / (emb.std(axis=0) + 1e-12)
        H = float(np.mean([hopkins14c(Xs, seed=s) for s in range(5)]))

        print(f"[{tag}] objects={M.shape[0]:,}  n_neighbors={n_neighbors_eff}  "
              f"min_dist={CELL14C_MIN_DIST}  PCs into UMAP={n_umap_in}  hopkins={H:.3f}")

        row = {"config": tag, "n": M.shape[0], "hopkins": round(H, 3)}
        for var in CELL14C_LABEL_VARS:
            groups = idx[var].astype(str)
            if groups.nunique() < 2 or groups.nunique() >= len(groups):
                sil = np.nan
            else:
                sil = silhouette_score(Xs, groups)
            row[f"silhouette_{var}"] = round(float(sil), 3) if not np.isnan(sil) else np.nan
        label_rows14c.append(row)

    label_summary14c = pd.DataFrame(label_rows14c)
    print("\n=== Silhouette against TRUE metadata labels (no KMeans, non-circular) ===")
    print(label_summary14c.to_string(index=False))

    p_summary14c = (DIR_TABLES14C /
                    f"umap_true_label_silhouette_ALL_CONFIGS_"
                    f"nn{CELL14C_N_NEIGHBORS}_md{CELL14C_MIN_DIST}.csv")
    label_summary14c.to_csv(p_summary14c, index=False)
    print(f"\nsaved -> {p_summary14c}")

    # --- Combined coordinates of all configs in one CSV ---
    if embeddings_by_config14c:
        coord_rows14c = []
        for tag, idx in embeddings_by_config14c.items():
            keep_cols = ["cutout_id", "region", "session", "split", "mean_level",
                        "mean_level_bin", "umap_x", "umap_y"]
            keep_cols = [c for c in keep_cols if c in idx.columns]
            cdf = idx[keep_cols].copy()
            cdf["config"] = tag
            coord_rows14c.append(cdf)
        coords_all14c = pd.concat(coord_rows14c, ignore_index=True)
        p_coords14c = (DIR_TABLES14C /
                       f"umap_coords_ALL_CONFIGS_nn{CELL14C_N_NEIGHBORS}_"
                       f"md{CELL14C_MIN_DIST}.csv")
        coords_all14c.to_csv(p_coords14c, index=False)
        print(f"saved -> {p_coords14c}")

# --- Overview grid: region / session / mean_level colouring, one column per config ---
if embeddings_by_config14c:
    cfg_order14c = list(embeddings_by_config14c.keys())
    n_cols14c = len(cfg_order14c)
    fig14c, axes14c = plt.subplots(3, n_cols14c,
                                   figsize=(_FW * 0.62 * n_cols14c, _FW * 1.75),
                                   squeeze=False)

    for c, tag in enumerate(cfg_order14c):
        idx = embeddings_by_config14c[tag]

        # Row 0: region
        ax = axes14c[0][c]
        for region, g in idx.groupby("region"):
            ax.scatter(g["umap_x"], g["umap_y"], s=5, alpha=0.6,
                      color=REGION_COLORS.get(region, "#7f7f7f"))
        ax.set_title(tag, fontsize=_FONT_SIZE)
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel("Region", fontsize=_FONT_SIZE)

        # Row 1: session
        ax = axes14c[1][c]
        sessions = sorted(idx["session"].astype(str).unique())
        pal = plt.get_cmap("tab20")
        for i, sess in enumerate(sessions):
            g = idx[idx["session"].astype(str) == sess]
            ax.scatter(g["umap_x"], g["umap_y"], s=5, alpha=0.6, color=pal(i % 20))
        ax.set_xticks([]); ax.set_yticks([])
        ax.text(0.02, 0.98, f"{len(sessions)} sessions", va="top", ha="left",
                transform=ax.transAxes, fontsize=_FONT_SIZE - 3)
        if c == 0:
            ax.set_ylabel("Session", fontsize=_FONT_SIZE)

        # Row 2: continuous mean_level
        ax = axes14c[2][c]
        sc = ax.scatter(idx["umap_x"], idx["umap_y"], s=5, alpha=0.6,
                        c=idx["mean_level"], cmap="viridis")
        ax.set_xticks([]); ax.set_yticks([])
        cb = fig14c.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
        cb.ax.tick_params(labelsize=_FONT_SIZE - 4)
        if c == 0:
            ax.set_ylabel("Mean Level", fontsize=_FONT_SIZE)

    handles = [plt.Line2D([0], [0], marker="o", linestyle="", markersize=6,
                          color=REGION_COLORS[r], label=r) for r in CELL14C_REGIONS]
    fig14c.legend(handles=handles, loc="upper center", ncol=3,
                 bbox_to_anchor=(0.5, 1.015), frameon=False)
    fig14c.suptitle(f"UMAP (n_neighbors={CELL14C_N_NEIGHBORS}, "
                    f"min_dist={CELL14C_MIN_DIST})", y=1.035)
    fig14c.tight_layout()
    plt.show()
    p_fig14c = (DIR_FIGURES14C /
               f"umap_ALL_CONFIGS_confound_grid_nn{CELL14C_N_NEIGHBORS}_"
               f"md{CELL14C_MIN_DIST}.png")
    fig14c.savefig(p_fig14c, dpi=150, bbox_inches="tight")
    print(f"saved -> {p_fig14c}")

In [ ]:
# === Cell 21: Run Summary and Export ===
# Funnel from indexed frames to exported objects, locked parameters per stage and an
# artifact list; writes summary.md and funnel.csv.

def _safe_len(path, kind="csv"):
    p = Path(path)
    if not p.exists():
        return None
    if kind == "csv":
        try:
            return len(pd.read_csv(p))
        except Exception:
            return None
    return None


def dir_size(path):
    p = Path(path)
    if not p.is_dir():
        return 0, 0
    n, tot = 0, 0
    for f in p.rglob("*"):
        if f.is_file():
            n += 1
            tot += f.stat().st_size
    return n, tot


def build_funnel():
    man = read_manifest()
    st = man.get("stages", {})
    fi = globals().get("FRAME_INDEX")
    rows = []

    def add(name, n, note=""):
        rows.append({"step": name, "n": n, "note": note})

    if fi is not None and len(fi):
        add("frames indexed", len(fi))
        add("frames scoped (RGB+HSI)", int(fi["in_scope"].sum()))
    ra = st.get("alignment", {}).get("results", {})
    if ra:
        add("sessions with measured offset", ra.get("n_sessions"),
            f"of {ra.get('n_sessions_total')}; DX {ra.get('dx')} px, DY per session")
    r1 = st.get("stage1", {}).get("results", {})
    if r1:
        add("frames with >=1 detection", r1.get("n_frames_with_det"),
            f"{r1.get('n_frames')} processed")
        add("stage 1 instances", r1.get("n_instances"))
    r2 = st.get("stage2", {}).get("results", {})
    if r2:
        add("cutouts generated", r2.get("n_cutouts"))
    r3 = st.get("stage3", {}).get("results", {})
    if r3:
        add("pseudo-GT processed", r3.get("n_processed"))
        add("label detected", r3.get("n_label_detected"))
        add("cap detected", r3.get("n_cap_detected"))
    # Single row for the Filter Tuning stage
    rf = st.get("filter", {}).get("results", {})
    if rf:
        add("filter-locked objects", rf.get("n_kept"), f"from {rf.get('n_pool')}")
    re_ = st.get("yolo_export", {}).get("results", {})
    if re_:
        add("exported objects", re_.get("n_exported"),
            f"single flat split, {re_.get('n_polygons')} polygon rows")
    r6 = st.get("hsi_extract", {}).get("results", {})
    if r6:
        add("objects with spectra", r6.get("n_objects"))
    return pd.DataFrame(rows)


def summarise_run(write_files=True):
    man = read_manifest()
    st = man.get("stages", {})

    print("=" * 78)
    print(f"RUN SUMMARY  {RUN_ID}")
    print("=" * 78)
    print(f"config hash : {CONFIG_HASH}")
    print(f"input       : {INPUT_ROOT}")
    print(f"data out    : {DATA_OUT_ROOT}")
    print(f"results out : {RESULTS_OUT_ROOT}")
    print()

    print("STAGE STATUS")
    print("-" * 78)
    expected = ["stage0_scope", "alignment", "stage1", "stage2", "stage3",
                "filter", "yolo_export", "hsi_extract"]
    for s in expected:
        rec = st.get(s, {})
        locked = "locked" if rec.get("param_hash") else "-"
        ran = rec.get("runtime", {})
        rt = fmt_dur(ran.get("elapsed_s")) if ran else "-"
        n = ran.get("n_done", "-") if ran else "-"
        flag = "" if rec else "   << never run"
        print(f"  {s:14s} {locked:8s} items={str(n):>8s}  time={rt:>10s}{flag}")
    print()

    funnel = build_funnel()
    if len(funnel):
        print("FUNNEL")
        print("-" * 78)
        prev = None
        for _, r in funnel.iterrows():
            n = r["n"]
            if n is None:
                continue
            delta = ""
            if prev is not None and isinstance(n, (int, np.integer)) and prev:
                delta = f"  ({100 * n / prev:5.1f}% of prev)"
            print(f"  {r['step']:30s} {n:>9,}{delta}   {r['note']}")
            if isinstance(n, (int, np.integer)):
                prev = n
        print()

    print("ARTIFACTS")
    print("-" * 78)
    arts = []
    for label, path in [
        ("frame index", DIR_TABLES / "frame_index.csv"),
        ("stage1 instances", DIR_STAGE1 / "instances.csv"),
        ("stage1 masks", DIR_STAGE1 / "masks"),
        ("cutouts csv", DIR_CUTOUTS / "cutouts.csv"),
        ("cutout images", DIR_CUTOUTS / "cutouts"),
        ("pseudo-GT csv", DIR_PSEUDOGT / "pseudo_gt.csv"),
        ("pseudo-GT masks", DIR_PSEUDOGT / "masks"),
        ("YOLO dataset", DIR_YOLO_DS),
        ("filtered objects", DIR_TABLES / "filtered_objects.csv"),
        ("export index", DIR_YOLO_DS / "export_index.csv"),
        ("dataset card", DIR_YOLO_DS / "DATASET_CARD.md"),
        ("region spectra", DIR_SPECTRA / "region_spectra.csv"),
        ("IARR reference", DIR_SPECTRA / "iarr_reference_spectrum.csv"),
        ("session alignment offsets", DIR_TABLES / "session_alignment_offsets.csv"),
        ("alignment measurements", DIR_TABLES / "frame_alignment_measurements.csv"),
        ("alignment methods", RESULTS_OUT_ROOT / "alignment_methods.md"),
        ("alignment parameters", RESULTS_OUT_ROOT / "alignment_parameters.json"),
        ("run manifest", MANIFEST_PATH),
    ]:
        p = Path(path)
        if not p.exists():
            continue
        if p.is_dir():
            n, tot = dir_size(p)
            arts.append({"artifact": label, "path": str(p), "items": n, "size": fmt_bytes(tot)})
        else:
            arts.append({"artifact": label, "path": str(p),
                         "items": _safe_len(p) if p.suffix == ".csv" else "",
                         "size": fmt_bytes(p.stat().st_size)})
    if arts:
        display(HTML(pd.DataFrame(arts).to_html(index=False)))

    if write_files:
        lines = [f"# Run Summary - {RUN_ID}", "",
                 f"- config hash: `{CONFIG_HASH}`",
                 f"- input: `{INPUT_ROOT}`",
                 f"- data out: `{DATA_OUT_ROOT}`",
                 f"- results out: `{RESULTS_OUT_ROOT}`",
                 f"- compute: {COMPUTE_PLAN['mode']} ({COMPUTE_PLAN['n_workers']} worker(s))", ""]
        lines += ["## Funnel", "", "| step | n | note |", "|---|---:|---|"]
        for _, r in funnel.iterrows():
            lines.append(f"| {r['step']} | {r['n']:,} | {r['note']} |"
                         if isinstance(r["n"], (int, np.integer)) else
                         f"| {r['step']} | {r['n']} | {r['note']} |")
        lines += ["", "## Locked parameters", ""]
        for s in expected:
            rec = st.get(s, {})
            if not rec.get("params"):
                continue
            lines.append(f"### {s}  (`{rec.get('param_hash')}`)")
            lines.append("")
            lines.append("```json")
            lines.append(json.dumps(rec["params"], indent=2, default=str))
            lines.append("```")
            lines.append("")
        # Alignment methods section of summary.md, built from alignment_parameters.json.
        lines += ["## Methods - RGB to HSI Alignment", ""]
        _apath = RESULTS_OUT_ROOT / "alignment_parameters.json"
        if _apath.exists():
            _ap = json.loads(_apath.read_text())
            _t, _s = _ap.get("transform", {}), _ap.get("stats", {})
            lines += [
                "Region masks are mapped to the hyperspectral grid with the geometric",
                "transform published by the dataset authors, corrected by two measured",
                "residual terms. The authors' `multimodal_manual_align` crops each modality",
                f"to a square (`rgb[:{_t.get('rgb_crop_rows')}, :]`, "
                f"`hyper[:, :{_t.get('hsi_crop_cols')}]`, both anchored at the origin) and",
                "resizes both to a common grid; the common size cancels in the composition.",
                "",
                "```",
                f"x_hsi = (x_rgb + 0.5) * {_t.get('scale_x', float('nan')):.7f} - 0.5 + "
                f"({_t.get('dx_px')})",
                f"y_hsi = (y_rgb + 0.5) * {_t.get('scale_y', float('nan')):.7f} - 0.5 + "
                f"DY(session)",
                "```",
                "",
                f"- Axis scales differ by "
                f"{(_t.get('scale_x', 1) / _t.get('scale_y', 1) - 1) * 100:.3f} % "
                "(the crops square up each modality independently)",
                f"- DX = {_t.get('dx_px')} px, a dataset-wide constant, band-independent",
                f"- DY measured per capture session over "
                f"{_s.get('frames_per_session', '?')} frames each "
                f"({_s.get('n_sessions', '?')} sessions, {_s.get('n_frames', '?')} frames)",
                f"- DY range {_s.get('dy_min', '?')} to {_s.get('dy_max', '?')} px; "
                f"between-session std {_s.get('between_session_std', float('nan')):.2f} px, "
                f"within-session std {_s.get('within_session_std', float('nan')):.2f} px",
                f"- Residual ~{_s.get('within_session_std', float('nan')):.1f} px is "
                "scene-depth parallax, removable by no global transform",
                f"- Region masks eroded by {_t.get('erode_px_default')} px before sampling",
                f"- Objects touching RGB row >= {_t.get('rgb_crop_rows')} dropped, not clipped",
                "",
                "Full derivation, rejected approaches and provenance:",
                "`20260819_SpectralWaste_Alignment_Design.md` and `alignment_methods.md`.",
                "",
            ]
        else:
            lines += ["_Alignment not yet measured - run Cell 6._", ""]

        lines += ["## Artifacts", "", "| artifact | path | items | size |", "|---|---|---:|---:|"]
        for a in arts:
            lines.append(f"| {a['artifact']} | `{a['path']}` | {a['items']} | {a['size']} |")
        sp = RESULTS_OUT_ROOT / "summary.md"
        sp.write_text("\n".join(lines))
        funnel.to_csv(DIR_TABLES / "funnel.csv", index=False)
        print(f"\nsummary -> {sp}")
        print(f"funnel  -> {DIR_TABLES / 'funnel.csv'}")

    never = [s for s in expected if s not in st]
    if never:
        print(f"\n!! never run: {', '.join(never)}")
    print("=" * 78)
    return funnel


w14_btn = widgets.Button(description="Summarise run", button_style="success", icon="flag-checkered")
w14_out = widgets.Output()


def _run14(_):
    with w14_out:
        clear_output(wait=True)
        summarise_run()


w14_btn.on_click(_run14)
display(w14_btn, w14_out)